# Wheat sell-or-hold paper: foundation and deep-learning benchmark

This notebook makes probabilistic price forecasts for 17 Pakistani wheat price series (WFP retail wheat, FAO retail flour, PBS wholesale wheat) with models that need a GPU and internet access:

| Section | Models | Typical time on a T4 GPU |
|---|---|---|
| 2 | Chronos-Bolt (base), Chronos-2, Chronos-2 with world wheat price, TimesFM 2.5 | 5–15 min |
| 3 | DeepAR, Temporal Fusion Transformer (trained here, refitted every year) | 30–60 min |

Forecasts are made at every month from 2010 onward, 1–12 months ahead, as 19 quantiles. Each forecast uses only data up to its origin month.

**How to run**
1. *Runtime → Change runtime type → T4 GPU → Save.*
2. *Runtime → Run all.* Allow Google Drive access when asked.
3. **Every result is saved to your Google Drive** (folder `wheat_benchmark_outputs`) as soon as it is made. If the runtime disconnects, just reconnect and *Run all* again: finished models and finished years are skipped, and work continues where it stopped.
4. If a cell asks you to **restart the session** after installing packages, click restart, then *Run all* again.
5. Nothing needs to be downloaded. When the last cell prints `ALL DONE`, the forecast files are in the Drive folder `wheat_benchmark_outputs`; copy them to `results/colab_raw/`.

Tips against disconnects: keep this tab open and visible, keep the laptop plugged in and stop it from sleeping, and don't run other Colab notebooks at the same time.

## 0. Connect Google Drive (results are saved there)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil
os.makedirs('/content/drive/MyDrive/wheat_benchmark_outputs', exist_ok=True)
# keep anything an earlier run left on the Colab disk
for p in glob.glob('/content/outputs/*.csv.gz'):
    dst = '/content/drive/MyDrive/wheat_benchmark_outputs/' + os.path.basename(p)
    if not os.path.exists(dst):
        shutil.copy(p, dst); print('copied earlier result to Drive:', os.path.basename(p))
print('Drive ready')

## 1. Data (written to the Colab disk)

In [ ]:
%%writefile /content/panel.csv
unique_id,ds,y
wfp_wheat_Lahore,2006-01-01,500.0
wfp_wheat_Lahore,2006-02-01,500.0
wfp_wheat_Lahore,2006-03-01,500.0
wfp_wheat_Lahore,2006-04-01,488.0
wfp_wheat_Lahore,2006-05-01,464.0
wfp_wheat_Lahore,2006-06-01,452.0
wfp_wheat_Lahore,2006-07-01,456.0
wfp_wheat_Lahore,2006-08-01,476.0
wfp_wheat_Lahore,2006-09-01,484.0
wfp_wheat_Lahore,2006-10-01,484.0
wfp_wheat_Lahore,2006-11-01,488.0
wfp_wheat_Lahore,2006-12-01,512.0
wfp_wheat_Lahore,2007-01-01,515.2
wfp_wheat_Lahore,2007-02-01,515.2
wfp_wheat_Lahore,2007-03-01,515.2
wfp_wheat_Lahore,2007-04-01,515.2
wfp_wheat_Lahore,2007-05-01,467.6
wfp_wheat_Lahore,2007-06-01,480.0
wfp_wheat_Lahore,2007-07-01,510.0
wfp_wheat_Lahore,2007-08-01,520.4
wfp_wheat_Lahore,2007-09-01,584.8
wfp_wheat_Lahore,2007-10-01,605.2
wfp_wheat_Lahore,2007-11-01,640.0
wfp_wheat_Lahore,2007-12-01,706.4
wfp_wheat_Lahore,2008-01-01,732.4
wfp_wheat_Lahore,2008-02-01,634.8
wfp_wheat_Lahore,2008-03-01,642.4
wfp_wheat_Lahore,2008-04-01,672.4
wfp_wheat_Lahore,2008-05-01,772.4
wfp_wheat_Lahore,2008-06-01,766.4
wfp_wheat_Lahore,2008-07-01,788.4
wfp_wheat_Lahore,2008-08-01,792.4
wfp_wheat_Lahore,2008-09-01,792.0
wfp_wheat_Lahore,2008-10-01,872.0
wfp_wheat_Lahore,2008-11-01,872.4
wfp_wheat_Lahore,2008-12-01,872.4
wfp_wheat_Lahore,2009-01-01,884.0
wfp_wheat_Lahore,2009-02-01,929.2
wfp_wheat_Lahore,2009-03-01,927.6
wfp_wheat_Lahore,2009-04-01,946.4
wfp_wheat_Lahore,2009-05-01,948.8
wfp_wheat_Lahore,2009-06-01,932.4
wfp_wheat_Lahore,2009-07-01,947.6
wfp_wheat_Lahore,2009-08-01,996.4
wfp_wheat_Lahore,2009-09-01,1004.8
wfp_wheat_Lahore,2009-10-01,1004.8
wfp_wheat_Lahore,2009-11-01,1018.0
wfp_wheat_Lahore,2009-12-01,1044.8
wfp_wheat_Lahore,2010-01-01,1095.2
wfp_wheat_Lahore,2010-02-01,1097.6
wfp_wheat_Lahore,2010-03-01,1092.8
wfp_wheat_Lahore,2010-04-01,1018.8
wfp_wheat_Lahore,2010-05-01,925.2
wfp_wheat_Lahore,2010-06-01,927.6
wfp_wheat_Lahore,2010-07-01,923.6
wfp_wheat_Lahore,2010-08-01,937.6
wfp_wheat_Lahore,2010-09-01,988.4
wfp_wheat_Lahore,2010-10-01,1052.4
wfp_wheat_Lahore,2010-11-01,1066.0
wfp_wheat_Lahore,2010-12-01,1077.6
wfp_wheat_Lahore,2011-01-01,1082.4
wfp_wheat_Lahore,2011-02-01,1082.4
wfp_wheat_Lahore,2011-03-01,1090.4
wfp_wheat_Lahore,2011-04-01,1061.6
wfp_wheat_Lahore,2011-05-01,950.0
wfp_wheat_Lahore,2011-06-01,966.4
wfp_wheat_Lahore,2011-07-01,967.6
wfp_wheat_Lahore,2011-08-01,981.6
wfp_wheat_Lahore,2011-09-01,1038.8
wfp_wheat_Lahore,2011-10-01,1050.0
wfp_wheat_Lahore,2011-11-01,1080.0
wfp_wheat_Lahore,2011-12-01,1121.6
wfp_wheat_Lahore,2012-01-01,1106.4
wfp_wheat_Lahore,2012-02-01,1117.6
wfp_wheat_Lahore,2012-03-01,1117.6
wfp_wheat_Lahore,2012-04-01,1095.6
wfp_wheat_Lahore,2012-05-01,1030.0
wfp_wheat_Lahore,2012-06-01,1055.2
wfp_wheat_Lahore,2012-07-01,1057.6
wfp_wheat_Lahore,2012-08-01,1083.2
wfp_wheat_Lahore,2012-09-01,1116.8
wfp_wheat_Lahore,2012-10-01,1129.6
wfp_wheat_Lahore,2012-11-01,1152.4
wfp_wheat_Lahore,2012-12-01,1233.2
wfp_wheat_Lahore,2013-01-01,1239.6
wfp_wheat_Lahore,2013-02-01,1255.2
wfp_wheat_Lahore,2013-03-01,1267.6
wfp_wheat_Lahore,2013-04-01,1264.4
wfp_wheat_Lahore,2013-05-01,1216.8
wfp_wheat_Lahore,2013-06-01,1286.8
wfp_wheat_Lahore,2013-07-01,1325.2
wfp_wheat_Lahore,2013-08-01,1383.2
wfp_wheat_Lahore,2013-09-01,1462.4
wfp_wheat_Lahore,2013-10-01,1460.8
wfp_wheat_Lahore,2013-11-01,1524.8
wfp_wheat_Lahore,2013-12-01,1592.4
wfp_wheat_Lahore,2014-01-01,1626.8
wfp_wheat_Lahore,2014-02-01,1644.8
wfp_wheat_Lahore,2014-03-01,1650.8
wfp_wheat_Lahore,2014-04-01,1502.0
wfp_wheat_Lahore,2014-05-01,1308.8
wfp_wheat_Lahore,2014-06-01,1320.0
wfp_wheat_Lahore,2014-07-01,1324.8
wfp_wheat_Lahore,2014-08-01,1377.6
wfp_wheat_Lahore,2014-09-01,1395.2
wfp_wheat_Lahore,2014-10-01,1400.8
wfp_wheat_Lahore,2014-11-01,1410.0
wfp_wheat_Lahore,2014-12-01,1410.0
wfp_wheat_Lahore,2015-01-01,1410.0
wfp_wheat_Lahore,2015-02-01,1410.0
wfp_wheat_Lahore,2015-03-01,1410.0
wfp_wheat_Lahore,2015-04-01,1373.2
wfp_wheat_Lahore,2015-05-01,1306.4
wfp_wheat_Lahore,2015-06-01,1300.0
wfp_wheat_Lahore,2015-07-01,1346.8
wfp_wheat_Lahore,2015-08-01,1362.4
wfp_wheat_Lahore,2015-09-01,1364.8
wfp_wheat_Lahore,2015-10-01,1380.8
wfp_wheat_Lahore,2015-11-01,1410.0
wfp_wheat_Lahore,2015-12-01,1424.8
wfp_wheat_Lahore,2016-01-01,1424.8
wfp_wheat_Lahore,2016-02-01,1444.8
wfp_wheat_Lahore,2016-03-01,1444.8
wfp_wheat_Lahore,2016-04-01,1384.8
wfp_wheat_Lahore,2016-05-01,1324.8
wfp_wheat_Lahore,2016-06-01,1315.2
wfp_wheat_Lahore,2016-07-01,1326.8
wfp_wheat_Lahore,2016-08-01,1344.8
wfp_wheat_Lahore,2016-09-01,1349.2
wfp_wheat_Lahore,2016-10-01,1375.2
wfp_wheat_Lahore,2016-11-01,1384.8
wfp_wheat_Lahore,2016-12-01,1384.8
wfp_wheat_Lahore,2017-01-01,1384.8
wfp_wheat_Lahore,2017-02-01,1424.8
wfp_wheat_Lahore,2017-03-01,1424.8
wfp_wheat_Lahore,2017-04-01,1425.2
wfp_wheat_Lahore,2017-05-01,1317.6
wfp_wheat_Lahore,2017-06-01,1305.2
wfp_wheat_Lahore,2017-07-01,1294.8
wfp_wheat_Lahore,2017-08-01,1264.0
wfp_wheat_Lahore,2017-09-01,1264.0
wfp_wheat_Lahore,2017-10-01,1264.0
wfp_wheat_Lahore,2017-11-01,1322.8
wfp_wheat_Lahore,2017-12-01,1390.0
wfp_wheat_Lahore,2018-01-01,1390.0
wfp_wheat_Lahore,2018-02-01,1390.0
wfp_wheat_Lahore,2018-03-01,1392.0
wfp_wheat_Lahore,2018-04-01,1390.0
wfp_wheat_Lahore,2018-05-01,1310.0
wfp_wheat_Lahore,2018-06-01,1310.0
wfp_wheat_Lahore,2018-07-01,1310.0
wfp_wheat_Lahore,2018-08-01,1340.0
wfp_wheat_Lahore,2018-09-01,1360.0
wfp_wheat_Lahore,2018-10-01,1390.0
wfp_wheat_Lahore,2018-11-01,1390.0
wfp_wheat_Lahore,2018-12-01,1390.0
wfp_wheat_Lahore,2019-01-01,1390.0
wfp_wheat_Lahore,2019-02-01,1390.0
wfp_wheat_Lahore,2019-03-01,1390.0
wfp_wheat_Lahore,2019-04-01,1390.0
wfp_wheat_Lahore,2019-05-01,1390.0
wfp_wheat_Lahore,2019-06-01,1390.0
wfp_wheat_Lahore,2019-07-01,1420.0
wfp_wheat_Lahore,2019-08-01,1420.0
wfp_wheat_Lahore,2019-09-01,1420.0
wfp_wheat_Lahore,2019-10-01,1420.0
wfp_wheat_Lahore,2019-11-01,1470.0
wfp_wheat_Lahore,2019-12-01,1724.0
wfp_wheat_Lahore,2020-01-01,1923.2
wfp_wheat_Lahore,2020-02-01,1806.8
wfp_wheat_Lahore,2020-03-01,1737.2
wfp_wheat_Lahore,2020-04-01,1792.0
wfp_wheat_Lahore,2020-05-01,1655.2
wfp_wheat_Lahore,2020-06-01,1897.6
wfp_wheat_Lahore,2020-07-01,2052.8
wfp_wheat_Lahore,2020-08-01,2180.0
wfp_wheat_Lahore,2020-09-01,2204.0
wfp_wheat_Lahore,2020-10-01,2416.0
wfp_wheat_Lahore,2020-11-01,2312.4
wfp_wheat_Lahore,2020-12-01,2190.0
wfp_wheat_Lahore,2021-01-01,2443.2
wfp_wheat_Lahore,2021-02-01,2410.0
wfp_wheat_Lahore,2021-03-01,2400.0
wfp_wheat_Lahore,2021-04-01,2165.6
wfp_wheat_Lahore,2021-05-01,2063.2
wfp_wheat_Lahore,2021-06-01,2115.2
wfp_wheat_Lahore,2021-07-01,2113.2
wfp_wheat_Lahore,2021-08-01,2113.6
wfp_wheat_Lahore,2021-09-01,2230.0
wfp_wheat_Lahore,2021-10-01,2395.6
wfp_wheat_Lahore,2021-11-01,2458.4
wfp_wheat_Lahore,2021-12-01,2460.0
wfp_wheat_Lahore,2022-01-01,2620.8
wfp_wheat_Lahore,2022-02-01,2558.4
wfp_wheat_Lahore,2022-03-01,2524.8
wfp_wheat_Lahore,2022-04-01,2475.2
wfp_wheat_Lahore,2022-05-01,2557.6
wfp_wheat_Lahore,2022-06-01,2791.2
wfp_wheat_Lahore,2022-07-01,3024.8
wfp_wheat_Lahore,2022-08-01,3000.0
wfp_wheat_Lahore,2022-09-01,3392.0
wfp_wheat_Lahore,2022-10-01,3538.4
wfp_wheat_Lahore,2022-11-01,3513.2
wfp_wheat_Lahore,2022-12-01,4006.8
wfp_wheat_Lahore,2023-01-01,4880.8
wfp_wheat_Lahore,2023-02-01,4382.4
wfp_wheat_Lahore,2023-03-01,4842.0
wfp_wheat_Lahore,2023-04-01,4950.0
wfp_wheat_Lahore,2023-05-01,4490.0
wfp_wheat_Lahore,2023-06-01,4460.0
wfp_wheat_Lahore,2023-07-01,4880.0
wfp_wheat_Lahore,2023-08-01,4900.0
wfp_wheat_Lahore,2023-09-01,4900.0
wfp_wheat_Lahore,2023-10-01,4850.0
wfp_wheat_Lahore,2023-11-01,5005.2
wfp_wheat_Lahore,2023-12-01,5200.0
wfp_wheat_Lahore,2024-01-01,5200.0
wfp_wheat_Lahore,2024-02-01,5175.2
wfp_wheat_Lahore,2024-03-01,5075.2
wfp_wheat_Lahore,2024-04-01,4425.2
wfp_wheat_Lahore,2024-05-01,3466.4
wfp_wheat_Lahore,2024-06-01,3150.0
wfp_wheat_Lahore,2024-07-01,3324.0
wfp_wheat_Lahore,2024-08-01,3099.2
wfp_wheat_Lahore,2024-09-01,2975.2
wfp_wheat_Lahore,2024-10-01,3100.0
wfp_wheat_Lahore,2024-11-01,3250.0
wfp_wheat_Lahore,2024-12-01,3274.0
wfp_wheat_Lahore,2025-01-01,3200.0
wfp_wheat_Lahore,2025-02-01,3200.0
wfp_wheat_Lahore,2025-03-01,3224.0
wfp_wheat_Lahore,2025-04-01,2924.0
wfp_wheat_Lahore,2025-05-01,2624.0
wfp_wheat_Lahore,2025-06-01,2648.0
wfp_wheat_Lahore,2025-07-01,2598.0
wfp_wheat_Lahore,2025-08-01,2885.6
wfp_wheat_Lahore,2025-09-01,3650.0
wfp_wheat_Lahore,2025-10-01,3796.4
wfp_wheat_Lahore,2025-11-01,4000.0
wfp_wheat_Lahore,2025-12-01,4200.0
wfp_wheat_Lahore,2026-01-01,4600.0
wfp_wheat_Lahore,2026-02-01,4680.0
wfp_wheat_Lahore,2026-03-01,4320.0
wfp_wheat_Lahore,2026-04-01,4200.0
wfp_wheat_Lahore,2026-05-01,4120.0
wfp_wheat_Lahore,2026-06-01,4200.0
wfp_wheat_Lahore,2026-07-01,4640.0
wfp_wheat_Lahore,2026-08-01,5000.0
wfp_wheat_Multan,2006-01-01,452.0
wfp_wheat_Multan,2006-02-01,452.0
wfp_wheat_Multan,2006-03-01,448.0
wfp_wheat_Multan,2006-04-01,416.0
wfp_wheat_Multan,2006-05-01,420.0
wfp_wheat_Multan,2006-06-01,424.0
wfp_wheat_Multan,2006-07-01,428.0
wfp_wheat_Multan,2006-08-01,448.0
wfp_wheat_Multan,2006-09-01,448.0
wfp_wheat_Multan,2006-10-01,448.0
wfp_wheat_Multan,2006-11-01,456.0
wfp_wheat_Multan,2006-12-01,464.0
wfp_wheat_Multan,2007-01-01,464.0
wfp_wheat_Multan,2007-02-01,470.0
wfp_wheat_Multan,2007-03-01,468.0
wfp_wheat_Multan,2007-04-01,468.0
wfp_wheat_Multan,2007-05-01,434.0
wfp_wheat_Multan,2007-06-01,459.6
wfp_wheat_Multan,2007-07-01,486.0
wfp_wheat_Multan,2007-08-01,476.0
wfp_wheat_Multan,2007-09-01,532.0
wfp_wheat_Multan,2007-10-01,560.0
wfp_wheat_Multan,2007-11-01,596.0
wfp_wheat_Multan,2007-12-01,652.0
wfp_wheat_Multan,2008-01-01,716.0
wfp_wheat_Multan,2008-02-01,620.0
wfp_wheat_Multan,2008-03-01,620.0
wfp_wheat_Multan,2008-04-01,670.0
wfp_wheat_Multan,2008-05-01,725.2
wfp_wheat_Multan,2008-06-01,712.0
wfp_wheat_Multan,2008-07-01,744.8
wfp_wheat_Multan,2008-08-01,760.0
wfp_wheat_Multan,2008-09-01,736.0
wfp_wheat_Multan,2008-10-01,876.0
wfp_wheat_Multan,2008-11-01,857.6
wfp_wheat_Multan,2008-12-01,825.2
wfp_wheat_Multan,2009-01-01,872.0
wfp_wheat_Multan,2009-02-01,882.4
wfp_wheat_Multan,2009-03-01,890.0
wfp_wheat_Multan,2009-04-01,936.0
wfp_wheat_Multan,2009-05-01,927.6
wfp_wheat_Multan,2009-06-01,915.2
wfp_wheat_Multan,2009-07-01,946.0
wfp_wheat_Multan,2009-08-01,972.4
wfp_wheat_Multan,2009-09-01,982.4
wfp_wheat_Multan,2009-10-01,996.0
wfp_wheat_Multan,2009-11-01,1010.0
wfp_wheat_Multan,2009-12-01,1022.0
wfp_wheat_Multan,2010-01-01,1045.2
wfp_wheat_Multan,2010-02-01,1050.0
wfp_wheat_Multan,2010-03-01,1050.0
wfp_wheat_Multan,2010-04-01,944.0
wfp_wheat_Multan,2010-05-01,892.4
wfp_wheat_Multan,2010-06-01,900.0
wfp_wheat_Multan,2010-07-01,896.0
wfp_wheat_Multan,2010-08-01,907.6
wfp_wheat_Multan,2010-09-01,982.0
wfp_wheat_Multan,2010-10-01,1010.0
wfp_wheat_Multan,2010-11-01,1040.0
wfp_wheat_Multan,2010-12-01,1040.0
wfp_wheat_Multan,2011-01-01,1037.6
wfp_wheat_Multan,2011-02-01,1042.4
wfp_wheat_Multan,2011-03-01,1044.0
wfp_wheat_Multan,2011-04-01,955.2
wfp_wheat_Multan,2011-05-01,880.0
wfp_wheat_Multan,2011-06-01,904.0
wfp_wheat_Multan,2011-07-01,940.0
wfp_wheat_Multan,2011-08-01,940.0
wfp_wheat_Multan,2011-09-01,957.6
wfp_wheat_Multan,2011-10-01,975.2
wfp_wheat_Multan,2011-11-01,1007.6
wfp_wheat_Multan,2011-12-01,1050.0
wfp_wheat_Multan,2012-01-01,1052.4
wfp_wheat_Multan,2012-02-01,1039.6
wfp_wheat_Multan,2012-03-01,1050.0
wfp_wheat_Multan,2012-04-01,1037.6
wfp_wheat_Multan,2012-05-01,996.0
wfp_wheat_Multan,2012-06-01,1027.6
wfp_wheat_Multan,2012-07-01,1012.4
wfp_wheat_Multan,2012-08-01,1080.0
wfp_wheat_Multan,2012-09-01,1107.6
wfp_wheat_Multan,2012-10-01,1115.2
wfp_wheat_Multan,2012-11-01,1148.0
wfp_wheat_Multan,2012-12-01,1170.0
wfp_wheat_Multan,2013-01-01,1296.0
wfp_wheat_Multan,2013-02-01,1255.2
wfp_wheat_Multan,2013-03-01,1245.2
wfp_wheat_Multan,2013-04-01,1155.2
wfp_wheat_Multan,2013-05-01,1194.0
wfp_wheat_Multan,2013-06-01,1267.6
wfp_wheat_Multan,2013-07-01,1327.6
wfp_wheat_Multan,2013-08-01,1406.0
wfp_wheat_Multan,2013-09-01,1404.8
wfp_wheat_Multan,2013-10-01,1416.0
wfp_wheat_Multan,2013-11-01,1470.0
wfp_wheat_Multan,2013-12-01,1544.8
wfp_wheat_Multan,2014-01-01,1510.0
wfp_wheat_Multan,2014-02-01,1537.6
wfp_wheat_Multan,2014-03-01,1600.0
wfp_wheat_Multan,2014-04-01,1386.0
wfp_wheat_Multan,2014-05-01,1235.2
wfp_wheat_Multan,2014-06-01,1244.8
wfp_wheat_Multan,2014-07-01,1308.0
wfp_wheat_Multan,2014-08-01,1400.0
wfp_wheat_Multan,2014-09-01,1347.6
wfp_wheat_Multan,2014-10-01,1340.0
wfp_wheat_Multan,2014-11-01,1352.4
wfp_wheat_Multan,2014-12-01,1324.8
wfp_wheat_Multan,2015-01-01,1320.0
wfp_wheat_Multan,2015-02-01,1316.0
wfp_wheat_Multan,2015-03-01,1320.0
wfp_wheat_Multan,2015-04-01,1248.0
wfp_wheat_Multan,2015-05-01,1240.0
wfp_wheat_Multan,2015-06-01,1190.0
wfp_wheat_Multan,2015-07-01,1232.0
wfp_wheat_Multan,2015-08-01,1230.0
wfp_wheat_Multan,2015-09-01,1220.0
wfp_wheat_Multan,2015-10-01,1300.0
wfp_wheat_Multan,2015-11-01,1335.2
wfp_wheat_Multan,2015-12-01,1334.0
wfp_wheat_Multan,2016-01-01,1340.0
wfp_wheat_Multan,2016-02-01,1352.4
wfp_wheat_Multan,2016-03-01,1360.0
wfp_wheat_Multan,2016-04-01,1260.0
wfp_wheat_Multan,2016-05-01,1200.0
wfp_wheat_Multan,2016-06-01,1242.0
wfp_wheat_Multan,2016-07-01,1280.0
wfp_wheat_Multan,2016-08-01,1280.0
wfp_wheat_Multan,2016-09-01,1284.0
wfp_wheat_Multan,2016-10-01,1295.2
wfp_wheat_Multan,2016-11-01,1307.6
wfp_wheat_Multan,2016-12-01,1342.0
wfp_wheat_Multan,2017-01-01,1340.0
wfp_wheat_Multan,2017-02-01,1340.0
wfp_wheat_Multan,2017-03-01,1336.0
wfp_wheat_Multan,2017-04-01,1242.4
wfp_wheat_Multan,2017-05-01,1200.0
wfp_wheat_Multan,2017-06-01,1200.0
wfp_wheat_Multan,2017-07-01,1190.0
wfp_wheat_Multan,2017-08-01,1180.0
wfp_wheat_Multan,2017-09-01,1202.4
wfp_wheat_Multan,2017-10-01,1235.2
wfp_wheat_Multan,2017-11-01,1304.0
wfp_wheat_Multan,2017-12-01,1320.0
wfp_wheat_Multan,2018-01-01,1320.0
wfp_wheat_Multan,2018-02-01,1305.2
wfp_wheat_Multan,2018-03-01,1296.0
wfp_wheat_Multan,2018-04-01,1230.0
wfp_wheat_Multan,2018-05-01,1180.0
wfp_wheat_Multan,2018-06-01,1280.0
wfp_wheat_Multan,2018-07-01,1284.8
wfp_wheat_Multan,2018-08-01,1300.0
wfp_wheat_Multan,2018-09-01,1302.4
wfp_wheat_Multan,2018-10-01,1310.0
wfp_wheat_Multan,2018-11-01,1318.0
wfp_wheat_Multan,2018-12-01,1320.0
wfp_wheat_Multan,2019-01-01,1330.0
wfp_wheat_Multan,2019-02-01,1350.0
wfp_wheat_Multan,2019-03-01,1360.0
wfp_wheat_Multan,2019-04-01,1300.8
wfp_wheat_Multan,2019-05-01,1294.0
wfp_wheat_Multan,2019-06-01,1382.4
wfp_wheat_Multan,2019-07-01,1400.0
wfp_wheat_Multan,2019-08-01,1404.0
wfp_wheat_Multan,2019-09-01,1441.2
wfp_wheat_Multan,2019-10-01,1451.2
wfp_wheat_Multan,2019-11-01,1571.2
wfp_wheat_Multan,2019-12-01,1630.0
wfp_wheat_Multan,2020-01-01,1830.0
wfp_wheat_Multan,2020-02-01,1688.8
wfp_wheat_Multan,2020-03-01,1678.8
wfp_wheat_Multan,2020-04-01,1532.0
wfp_wheat_Multan,2020-05-01,1410.0
wfp_wheat_Multan,2020-06-01,1625.2
wfp_wheat_Multan,2020-07-01,1870.8
wfp_wheat_Multan,2020-08-01,2070.0
wfp_wheat_Multan,2020-09-01,2103.2
wfp_wheat_Multan,2020-10-01,2230.0
wfp_wheat_Multan,2020-11-01,2270.0
wfp_wheat_Multan,2020-12-01,2270.0
wfp_wheat_Multan,2021-01-01,2270.0
wfp_wheat_Multan,2021-02-01,2284.8
wfp_wheat_Multan,2021-03-01,2290.0
wfp_wheat_Multan,2021-04-01,1924.8
wfp_wheat_Multan,2021-05-01,1867.6
wfp_wheat_Multan,2021-06-01,1950.0
wfp_wheat_Multan,2021-07-01,1976.8
wfp_wheat_Multan,2021-08-01,1983.2
wfp_wheat_Multan,2021-09-01,2040.0
wfp_wheat_Multan,2021-10-01,2231.6
wfp_wheat_Multan,2021-11-01,2245.6
wfp_wheat_Multan,2021-12-01,2290.0
wfp_wheat_Multan,2022-01-01,2290.0
wfp_wheat_Multan,2022-02-01,2357.6
wfp_wheat_Multan,2022-03-01,2380.0
wfp_wheat_Multan,2022-04-01,2288.8
wfp_wheat_Multan,2022-05-01,2220.0
wfp_wheat_Multan,2022-06-01,2294.4
wfp_wheat_Multan,2022-07-01,2374.8
wfp_wheat_Multan,2022-08-01,2615.6
wfp_wheat_Multan,2022-09-01,2836.8
wfp_wheat_Multan,2022-10-01,3072.8
wfp_wheat_Multan,2022-11-01,3180.0
wfp_wheat_Multan,2022-12-01,2764.0
wfp_wheat_Multan,2023-01-01,4525.6
wfp_wheat_Multan,2023-02-01,4487.6
wfp_wheat_Multan,2023-03-01,4255.6
wfp_wheat_Multan,2023-04-01,4062.0
wfp_wheat_Multan,2023-05-01,4024.8
wfp_wheat_Multan,2023-06-01,4237.2
wfp_wheat_Multan,2023-07-01,4874.8
wfp_wheat_Multan,2023-08-01,4812.4
wfp_wheat_Multan,2023-09-01,4674.8
wfp_wheat_Multan,2023-10-01,4574.8
wfp_wheat_Multan,2023-11-01,4637.6
wfp_wheat_Multan,2023-12-01,4800.0
wfp_wheat_Multan,2024-01-01,4800.0
wfp_wheat_Multan,2024-02-01,4800.0
wfp_wheat_Multan,2024-03-01,4600.0
wfp_wheat_Multan,2024-04-01,3749.6
wfp_wheat_Multan,2024-05-01,3199.6
wfp_wheat_Multan,2024-06-01,3149.6
wfp_wheat_Multan,2024-07-01,3149.6
wfp_wheat_Multan,2024-08-01,2899.6
wfp_wheat_Multan,2024-09-01,2504.4
wfp_wheat_Multan,2024-10-01,2644.8
wfp_wheat_Multan,2024-11-01,2899.6
wfp_wheat_Multan,2024-12-01,2949.6
wfp_wheat_Multan,2025-01-01,2949.6
wfp_wheat_Multan,2025-02-01,2949.6
wfp_wheat_Multan,2025-03-01,2949.6
wfp_wheat_Multan,2025-04-01,2634.8
wfp_wheat_Multan,2025-05-01,2384.8
wfp_wheat_Multan,2025-06-01,2449.6
wfp_wheat_Multan,2025-07-01,2449.6
wfp_wheat_Multan,2025-08-01,2862.4
wfp_wheat_Multan,2025-09-01,3737.2
wfp_wheat_Multan,2025-10-01,3624.8
wfp_wheat_Multan,2025-11-01,3640.0
wfp_wheat_Multan,2025-12-01,3624.0
wfp_wheat_Multan,2026-01-01,4400.0
wfp_wheat_Multan,2026-02-01,4680.0
wfp_wheat_Multan,2026-03-01,4320.0
wfp_wheat_Multan,2026-04-01,3720.0
wfp_wheat_Multan,2026-05-01,3840.0
wfp_wheat_Multan,2026-06-01,4240.0
wfp_wheat_Multan,2026-07-01,4640.0
wfp_wheat_Multan,2026-08-01,4840.0
wfp_wheat_Karachi,2006-01-01,480.0
wfp_wheat_Karachi,2006-02-01,480.0
wfp_wheat_Karachi,2006-03-01,480.0
wfp_wheat_Karachi,2006-04-01,480.0
wfp_wheat_Karachi,2006-05-01,480.0
wfp_wheat_Karachi,2006-06-01,480.0
wfp_wheat_Karachi,2006-07-01,484.0
wfp_wheat_Karachi,2006-08-01,496.0
wfp_wheat_Karachi,2006-09-01,496.0
wfp_wheat_Karachi,2006-10-01,496.0
wfp_wheat_Karachi,2006-11-01,508.0
wfp_wheat_Karachi,2006-12-01,512.0
wfp_wheat_Karachi,2007-01-01,520.0
wfp_wheat_Karachi,2007-02-01,520.0
wfp_wheat_Karachi,2007-03-01,520.0
wfp_wheat_Karachi,2007-04-01,520.0
wfp_wheat_Karachi,2007-05-01,514.0
wfp_wheat_Karachi,2007-06-01,524.0
wfp_wheat_Karachi,2007-07-01,550.0
wfp_wheat_Karachi,2007-08-01,560.0
wfp_wheat_Karachi,2007-09-01,584.0
wfp_wheat_Karachi,2007-10-01,620.0
wfp_wheat_Karachi,2007-11-01,628.0
wfp_wheat_Karachi,2007-12-01,740.0
wfp_wheat_Karachi,2008-01-01,824.0
wfp_wheat_Karachi,2008-02-01,840.0
wfp_wheat_Karachi,2008-03-01,760.0
wfp_wheat_Karachi,2008-04-01,800.0
wfp_wheat_Karachi,2008-05-01,920.0
wfp_wheat_Karachi,2008-06-01,920.0
wfp_wheat_Karachi,2008-07-01,964.0
wfp_wheat_Karachi,2008-08-01,960.0
wfp_wheat_Karachi,2008-09-01,992.0
wfp_wheat_Karachi,2008-10-01,1150.0
wfp_wheat_Karachi,2008-11-01,1160.0
wfp_wheat_Karachi,2008-12-01,1125.2
wfp_wheat_Karachi,2009-01-01,1020.0
wfp_wheat_Karachi,2009-02-01,960.0
wfp_wheat_Karachi,2009-03-01,1005.2
wfp_wheat_Karachi,2009-04-01,1052.0
wfp_wheat_Karachi,2009-05-01,1015.2
wfp_wheat_Karachi,2009-06-01,1000.0
wfp_wheat_Karachi,2009-07-01,1009.6
wfp_wheat_Karachi,2009-08-01,1030.0
wfp_wheat_Karachi,2009-09-01,1040.0
wfp_wheat_Karachi,2009-10-01,1040.0
wfp_wheat_Karachi,2009-11-01,1040.0
wfp_wheat_Karachi,2009-12-01,1040.0
wfp_wheat_Karachi,2010-01-01,1040.0
wfp_wheat_Karachi,2010-02-01,1055.2
wfp_wheat_Karachi,2010-03-01,1060.0
wfp_wheat_Karachi,2010-04-01,1032.0
wfp_wheat_Karachi,2010-05-01,1000.0
wfp_wheat_Karachi,2010-06-01,1000.0
wfp_wheat_Karachi,2010-07-01,988.0
wfp_wheat_Karachi,2010-08-01,985.2
wfp_wheat_Karachi,2010-09-01,1024.0
wfp_wheat_Karachi,2010-10-01,1060.0
wfp_wheat_Karachi,2010-11-01,1060.0
wfp_wheat_Karachi,2010-12-01,1060.0
wfp_wheat_Karachi,2011-01-01,1075.2
wfp_wheat_Karachi,2011-02-01,1100.0
wfp_wheat_Karachi,2011-03-01,1116.0
wfp_wheat_Karachi,2011-04-01,1060.0
wfp_wheat_Karachi,2011-05-01,1055.2
wfp_wheat_Karachi,2011-06-01,1088.0
wfp_wheat_Karachi,2011-07-01,1160.0
wfp_wheat_Karachi,2011-08-01,1120.0
wfp_wheat_Karachi,2011-09-01,1120.0
wfp_wheat_Karachi,2011-10-01,1120.0
wfp_wheat_Karachi,2011-11-01,1140.0
wfp_wheat_Karachi,2011-12-01,1140.0
wfp_wheat_Karachi,2012-01-01,1140.0
wfp_wheat_Karachi,2012-02-01,1150.0
wfp_wheat_Karachi,2012-03-01,1160.0
wfp_wheat_Karachi,2012-04-01,1160.0
wfp_wheat_Karachi,2012-05-01,1148.0
wfp_wheat_Karachi,2012-06-01,1140.0
wfp_wheat_Karachi,2012-07-01,1164.8
wfp_wheat_Karachi,2012-08-01,1244.0
wfp_wheat_Karachi,2012-09-01,1215.2
wfp_wheat_Karachi,2012-10-01,1240.0
wfp_wheat_Karachi,2012-11-01,1280.0
wfp_wheat_Karachi,2012-12-01,1284.8
wfp_wheat_Karachi,2013-01-01,1380.0
wfp_wheat_Karachi,2013-02-01,1400.0
wfp_wheat_Karachi,2013-03-01,1440.0
wfp_wheat_Karachi,2013-04-01,1380.0
wfp_wheat_Karachi,2013-05-01,1384.0
wfp_wheat_Karachi,2013-06-01,1400.0
wfp_wheat_Karachi,2013-07-01,1415.2
wfp_wheat_Karachi,2013-08-01,1456.0
wfp_wheat_Karachi,2013-09-01,1480.0
wfp_wheat_Karachi,2013-10-01,1480.0
wfp_wheat_Karachi,2013-11-01,1480.0
wfp_wheat_Karachi,2013-12-01,1480.0
wfp_wheat_Karachi,2014-01-01,1480.0
wfp_wheat_Karachi,2014-02-01,1540.0
wfp_wheat_Karachi,2014-03-01,1560.0
wfp_wheat_Karachi,2014-04-01,1496.0
wfp_wheat_Karachi,2014-05-01,1430.0
wfp_wheat_Karachi,2014-06-01,1360.0
wfp_wheat_Karachi,2014-07-01,1360.0
wfp_wheat_Karachi,2014-08-01,1360.0
wfp_wheat_Karachi,2014-09-01,1360.0
wfp_wheat_Karachi,2014-10-01,1360.0
wfp_wheat_Karachi,2014-11-01,1375.2
wfp_wheat_Karachi,2014-12-01,1380.0
wfp_wheat_Karachi,2015-01-01,1400.0
wfp_wheat_Karachi,2015-02-01,1388.0
wfp_wheat_Karachi,2015-03-01,1400.0
wfp_wheat_Karachi,2015-04-01,1400.0
wfp_wheat_Karachi,2015-05-01,1400.0
wfp_wheat_Karachi,2015-06-01,1400.0
wfp_wheat_Karachi,2015-07-01,1400.0
wfp_wheat_Karachi,2015-08-01,1400.0
wfp_wheat_Karachi,2015-09-01,1400.0
wfp_wheat_Karachi,2015-10-01,1472.0
wfp_wheat_Karachi,2015-11-01,1480.0
wfp_wheat_Karachi,2015-12-01,1480.0
wfp_wheat_Karachi,2016-01-01,1480.0
wfp_wheat_Karachi,2016-02-01,1450.0
wfp_wheat_Karachi,2016-03-01,1424.0
wfp_wheat_Karachi,2016-04-01,1400.0
wfp_wheat_Karachi,2016-05-01,1400.0
wfp_wheat_Karachi,2016-06-01,1400.0
wfp_wheat_Karachi,2016-07-01,1424.0
wfp_wheat_Karachi,2016-08-01,1410.0
wfp_wheat_Karachi,2016-09-01,1384.0
wfp_wheat_Karachi,2016-10-01,1440.0
wfp_wheat_Karachi,2016-11-01,1450.0
wfp_wheat_Karachi,2016-12-01,1460.0
wfp_wheat_Karachi,2017-01-01,1460.0
wfp_wheat_Karachi,2017-02-01,1470.0
wfp_wheat_Karachi,2017-03-01,1464.0
wfp_wheat_Karachi,2017-04-01,1440.0
wfp_wheat_Karachi,2017-05-01,1420.0
wfp_wheat_Karachi,2017-06-01,1400.0
wfp_wheat_Karachi,2017-07-01,1400.0
wfp_wheat_Karachi,2017-08-01,1400.0
wfp_wheat_Karachi,2017-09-01,1400.0
wfp_wheat_Karachi,2017-10-01,1402.0
wfp_wheat_Karachi,2017-11-01,1450.0
wfp_wheat_Karachi,2017-12-01,1482.0
wfp_wheat_Karachi,2018-01-01,1500.8
wfp_wheat_Karachi,2018-02-01,1520.0
wfp_wheat_Karachi,2018-03-01,1512.0
wfp_wheat_Karachi,2018-04-01,1460.0
wfp_wheat_Karachi,2018-05-01,1440.0
wfp_wheat_Karachi,2018-06-01,1450.0
wfp_wheat_Karachi,2018-07-01,1475.2
wfp_wheat_Karachi,2018-08-01,1490.0
wfp_wheat_Karachi,2018-09-01,1500.0
wfp_wheat_Karachi,2018-10-01,1515.2
wfp_wheat_Karachi,2018-11-01,1532.0
wfp_wheat_Karachi,2018-12-01,1540.0
wfp_wheat_Karachi,2019-01-01,1552.0
wfp_wheat_Karachi,2019-02-01,1560.0
wfp_wheat_Karachi,2019-03-01,1560.0
wfp_wheat_Karachi,2019-04-01,1530.0
wfp_wheat_Karachi,2019-05-01,1532.0
wfp_wheat_Karachi,2019-06-01,1550.0
wfp_wheat_Karachi,2019-07-01,1560.0
wfp_wheat_Karachi,2019-08-01,1584.0
wfp_wheat_Karachi,2019-09-01,1620.0
wfp_wheat_Karachi,2019-10-01,1656.0
wfp_wheat_Karachi,2019-11-01,1760.0
wfp_wheat_Karachi,2019-12-01,1840.0
wfp_wheat_Karachi,2020-01-01,2040.0
wfp_wheat_Karachi,2020-02-01,2030.0
wfp_wheat_Karachi,2020-03-01,1906.8
wfp_wheat_Karachi,2020-04-01,1834.8
wfp_wheat_Karachi,2020-05-01,1760.0
wfp_wheat_Karachi,2020-06-01,1800.0
wfp_wheat_Karachi,2020-07-01,1904.0
wfp_wheat_Karachi,2020-08-01,2060.0
wfp_wheat_Karachi,2020-09-01,2146.8
wfp_wheat_Karachi,2020-10-01,2280.0
wfp_wheat_Karachi,2020-11-01,2140.0
wfp_wheat_Karachi,2020-12-01,2184.0
wfp_wheat_Karachi,2021-01-01,2280.0
wfp_wheat_Karachi,2021-02-01,2300.0
wfp_wheat_Karachi,2021-03-01,2300.0
wfp_wheat_Karachi,2021-04-01,2232.0
wfp_wheat_Karachi,2021-05-01,2320.0
wfp_wheat_Karachi,2021-06-01,2280.0
wfp_wheat_Karachi,2021-07-01,2240.0
wfp_wheat_Karachi,2021-08-01,2240.0
wfp_wheat_Karachi,2021-09-01,2410.0
wfp_wheat_Karachi,2021-10-01,2566.8
wfp_wheat_Karachi,2021-11-01,2640.0
wfp_wheat_Karachi,2021-12-01,2640.0
wfp_wheat_Karachi,2022-01-01,2680.0
wfp_wheat_Karachi,2022-02-01,2580.0
wfp_wheat_Karachi,2022-03-01,2404.8
wfp_wheat_Karachi,2022-04-01,2440.0
wfp_wheat_Karachi,2022-05-01,2706.8
wfp_wheat_Karachi,2022-06-01,3007.2
wfp_wheat_Karachi,2022-07-01,3280.0
wfp_wheat_Karachi,2022-08-01,3280.0
wfp_wheat_Karachi,2022-09-01,3707.2
wfp_wheat_Karachi,2022-10-01,3804.8
wfp_wheat_Karachi,2022-11-01,3744.8
wfp_wheat_Karachi,2022-12-01,3952.0
wfp_wheat_Karachi,2023-01-01,4354.8
wfp_wheat_Karachi,2023-02-01,3954.8
wfp_wheat_Karachi,2023-03-01,4960.0
wfp_wheat_Karachi,2023-04-01,5000.0
wfp_wheat_Karachi,2023-05-01,5020.0
wfp_wheat_Karachi,2023-06-01,5000.0
wfp_wheat_Karachi,2023-07-01,5500.0
wfp_wheat_Karachi,2023-08-01,5500.0
wfp_wheat_Karachi,2023-09-01,5300.0
wfp_wheat_Karachi,2023-10-01,5080.0
wfp_wheat_Karachi,2023-11-01,4700.0
wfp_wheat_Karachi,2023-12-01,4800.0
wfp_wheat_Karachi,2024-01-01,4920.0
wfp_wheat_Karachi,2024-02-01,4820.0
wfp_wheat_Karachi,2024-03-01,4560.0
wfp_wheat_Karachi,2024-04-01,4080.0
wfp_wheat_Karachi,2024-05-01,3440.0
wfp_wheat_Karachi,2024-06-01,3280.0
wfp_wheat_Karachi,2024-07-01,3320.0
wfp_wheat_Karachi,2024-08-01,3280.0
wfp_wheat_Karachi,2024-09-01,3160.0
wfp_wheat_Karachi,2024-10-01,3240.0
wfp_wheat_Karachi,2024-11-01,3380.0
wfp_wheat_Karachi,2024-12-01,3300.0
wfp_wheat_Karachi,2025-01-01,3260.0
wfp_wheat_Karachi,2025-02-01,3160.0
wfp_wheat_Karachi,2025-03-01,3120.0
wfp_wheat_Karachi,2025-04-01,2810.0
wfp_wheat_Karachi,2025-05-01,2580.0
wfp_wheat_Karachi,2025-06-01,2600.0
wfp_wheat_Karachi,2025-07-01,2620.0
wfp_wheat_Karachi,2025-08-01,2900.0
wfp_wheat_Karachi,2025-09-01,3603.6
wfp_wheat_Karachi,2025-10-01,3960.0
wfp_wheat_Karachi,2025-11-01,3840.0
wfp_wheat_Karachi,2025-12-01,4080.0
wfp_wheat_Karachi,2026-01-01,4360.0
wfp_wheat_Karachi,2026-02-01,4200.0
wfp_wheat_Karachi,2026-03-01,3800.0
wfp_wheat_Karachi,2026-04-01,3800.0
wfp_wheat_Karachi,2026-05-01,4160.0
wfp_wheat_Karachi,2026-06-01,4280.0
wfp_wheat_Karachi,2026-07-01,4720.0
wfp_wheat_Karachi,2026-08-01,5000.0
wfp_wheat_Karachi,2026-09-01,5200.0
wfp_wheat_Peshawar,2006-01-01,500.0
wfp_wheat_Peshawar,2006-02-01,520.0
wfp_wheat_Peshawar,2006-03-01,520.0
wfp_wheat_Peshawar,2006-04-01,500.0
wfp_wheat_Peshawar,2006-05-01,472.0
wfp_wheat_Peshawar,2006-06-01,472.0
wfp_wheat_Peshawar,2006-07-01,476.0
wfp_wheat_Peshawar,2006-08-01,480.0
wfp_wheat_Peshawar,2006-09-01,480.0
wfp_wheat_Peshawar,2006-10-01,492.0
wfp_wheat_Peshawar,2006-11-01,504.0
wfp_wheat_Peshawar,2006-12-01,520.0
wfp_wheat_Peshawar,2007-01-01,530.0
wfp_wheat_Peshawar,2007-02-01,530.0
wfp_wheat_Peshawar,2007-03-01,530.0
wfp_wheat_Peshawar,2007-04-01,530.0
wfp_wheat_Peshawar,2007-05-01,480.0
wfp_wheat_Peshawar,2007-06-01,490.0
wfp_wheat_Peshawar,2007-07-01,520.0
wfp_wheat_Peshawar,2007-08-01,520.0
wfp_wheat_Peshawar,2007-09-01,556.0
wfp_wheat_Peshawar,2007-10-01,560.0
wfp_wheat_Peshawar,2007-11-01,640.0
wfp_wheat_Peshawar,2007-12-01,765.2
wfp_wheat_Peshawar,2008-01-01,936.0
wfp_wheat_Peshawar,2008-02-01,800.0
wfp_wheat_Peshawar,2008-03-01,800.0
wfp_wheat_Peshawar,2008-04-01,880.0
wfp_wheat_Peshawar,2008-05-01,1040.0
wfp_wheat_Peshawar,2008-06-01,940.0
wfp_wheat_Peshawar,2008-07-01,988.0
wfp_wheat_Peshawar,2008-08-01,1022.4
wfp_wheat_Peshawar,2008-09-01,1124.0
wfp_wheat_Peshawar,2008-10-01,1300.0
wfp_wheat_Peshawar,2008-11-01,1232.0
wfp_wheat_Peshawar,2008-12-01,1200.0
wfp_wheat_Peshawar,2009-01-01,1172.0
wfp_wheat_Peshawar,2009-02-01,1105.2
wfp_wheat_Peshawar,2009-03-01,1015.2
wfp_wheat_Peshawar,2009-04-01,1040.0
wfp_wheat_Peshawar,2009-05-01,997.6
wfp_wheat_Peshawar,2009-06-01,932.4
wfp_wheat_Peshawar,2009-07-01,976.0
wfp_wheat_Peshawar,2009-08-01,1035.2
wfp_wheat_Peshawar,2009-09-01,1040.0
wfp_wheat_Peshawar,2009-10-01,1045.2
wfp_wheat_Peshawar,2009-11-01,1040.0
wfp_wheat_Peshawar,2009-12-01,1060.0
wfp_wheat_Peshawar,2010-01-01,1060.0
wfp_wheat_Peshawar,2010-02-01,1060.0
wfp_wheat_Peshawar,2010-03-01,1055.2
wfp_wheat_Peshawar,2010-04-01,1000.0
wfp_wheat_Peshawar,2010-05-01,980.0
wfp_wheat_Peshawar,2010-06-01,980.0
wfp_wheat_Peshawar,2010-07-01,980.0
wfp_wheat_Peshawar,2010-08-01,1000.0
wfp_wheat_Peshawar,2010-09-01,1050.0
wfp_wheat_Peshawar,2010-10-01,1050.0
wfp_wheat_Peshawar,2010-11-01,1040.0
wfp_wheat_Peshawar,2010-12-01,1040.0
wfp_wheat_Peshawar,2011-01-01,1130.0
wfp_wheat_Peshawar,2011-02-01,1130.0
wfp_wheat_Peshawar,2011-03-01,1120.0
wfp_wheat_Peshawar,2011-04-01,1100.0
wfp_wheat_Peshawar,2011-05-01,1040.0
wfp_wheat_Peshawar,2011-06-01,1020.0
wfp_wheat_Peshawar,2011-07-01,1050.0
wfp_wheat_Peshawar,2011-08-01,1060.0
wfp_wheat_Peshawar,2011-09-01,1080.0
wfp_wheat_Peshawar,2011-10-01,1080.0
wfp_wheat_Peshawar,2011-11-01,1133.2
wfp_wheat_Peshawar,2011-12-01,1160.0
wfp_wheat_Peshawar,2012-01-01,1160.0
wfp_wheat_Peshawar,2012-02-01,1151.6
wfp_wheat_Peshawar,2012-03-01,1154.8
wfp_wheat_Peshawar,2012-04-01,1160.0
wfp_wheat_Peshawar,2012-05-01,1144.0
wfp_wheat_Peshawar,2012-06-01,1120.0
wfp_wheat_Peshawar,2012-07-01,1120.0
wfp_wheat_Peshawar,2012-08-01,1186.8
wfp_wheat_Peshawar,2012-09-01,1200.0
wfp_wheat_Peshawar,2012-10-01,1200.0
wfp_wheat_Peshawar,2012-11-01,1224.0
wfp_wheat_Peshawar,2012-12-01,1270.0
wfp_wheat_Peshawar,2013-01-01,1388.0
wfp_wheat_Peshawar,2013-02-01,1360.0
wfp_wheat_Peshawar,2013-03-01,1360.0
wfp_wheat_Peshawar,2013-04-01,1309.2
wfp_wheat_Peshawar,2013-05-01,1256.0
wfp_wheat_Peshawar,2013-06-01,1306.8
wfp_wheat_Peshawar,2013-07-01,1360.0
wfp_wheat_Peshawar,2013-08-01,1360.0
wfp_wheat_Peshawar,2013-09-01,1415.2
wfp_wheat_Peshawar,2013-10-01,1424.0
wfp_wheat_Peshawar,2013-11-01,1535.2
wfp_wheat_Peshawar,2013-12-01,1600.0
wfp_wheat_Peshawar,2014-01-01,1616.0
wfp_wheat_Peshawar,2014-02-01,1680.0
wfp_wheat_Peshawar,2014-03-01,1680.0
wfp_wheat_Peshawar,2014-04-01,1616.0
wfp_wheat_Peshawar,2014-05-01,1435.2
wfp_wheat_Peshawar,2014-06-01,1380.0
wfp_wheat_Peshawar,2014-07-01,1464.0
wfp_wheat_Peshawar,2014-08-01,1520.0
wfp_wheat_Peshawar,2014-09-01,1470.0
wfp_wheat_Peshawar,2014-10-01,1488.0
wfp_wheat_Peshawar,2014-11-01,1560.0
wfp_wheat_Peshawar,2014-12-01,1560.0
wfp_wheat_Peshawar,2015-01-01,1480.0
wfp_wheat_Peshawar,2015-02-01,1480.0
wfp_wheat_Peshawar,2015-03-01,1480.0
wfp_wheat_Peshawar,2015-04-01,1400.0
wfp_wheat_Peshawar,2015-05-01,1340.0
wfp_wheat_Peshawar,2015-06-01,1250.0
wfp_wheat_Peshawar,2015-07-01,1240.0
wfp_wheat_Peshawar,2015-08-01,1240.0
wfp_wheat_Peshawar,2015-09-01,1240.0
wfp_wheat_Peshawar,2015-10-01,1256.0
wfp_wheat_Peshawar,2015-11-01,1326.8
wfp_wheat_Peshawar,2015-12-01,1360.0
wfp_wheat_Peshawar,2016-01-01,1370.0
wfp_wheat_Peshawar,2016-02-01,1400.0
wfp_wheat_Peshawar,2016-03-01,1368.0
wfp_wheat_Peshawar,2016-04-01,1340.0
wfp_wheat_Peshawar,2016-05-01,1290.0
wfp_wheat_Peshawar,2016-06-01,1280.0
wfp_wheat_Peshawar,2016-07-01,1280.0
wfp_wheat_Peshawar,2016-08-01,1340.0
wfp_wheat_Peshawar,2016-09-01,1360.0
wfp_wheat_Peshawar,2016-10-01,1360.0
wfp_wheat_Peshawar,2016-11-01,1380.0
wfp_wheat_Peshawar,2016-12-01,1440.0
wfp_wheat_Peshawar,2017-01-01,1440.0
wfp_wheat_Peshawar,2017-02-01,1440.0
wfp_wheat_Peshawar,2017-03-01,1440.0
wfp_wheat_Peshawar,2017-04-01,1400.0
wfp_wheat_Peshawar,2017-05-01,1320.0
wfp_wheat_Peshawar,2017-06-01,1344.0
wfp_wheat_Peshawar,2017-07-01,1320.0
wfp_wheat_Peshawar,2017-08-01,1280.0
wfp_wheat_Peshawar,2017-09-01,1286.8
wfp_wheat_Peshawar,2017-10-01,1320.0
wfp_wheat_Peshawar,2017-11-01,1389.2
wfp_wheat_Peshawar,2017-12-01,1440.0
wfp_wheat_Peshawar,2018-01-01,1440.0
wfp_wheat_Peshawar,2018-02-01,1440.0
wfp_wheat_Peshawar,2018-03-01,1440.0
wfp_wheat_Peshawar,2018-04-01,1390.0
wfp_wheat_Peshawar,2018-05-01,1280.0
wfp_wheat_Peshawar,2018-06-01,1330.0
wfp_wheat_Peshawar,2018-07-01,1400.0
wfp_wheat_Peshawar,2018-08-01,1400.0
wfp_wheat_Peshawar,2018-09-01,1400.0
wfp_wheat_Peshawar,2018-10-01,1430.0
wfp_wheat_Peshawar,2018-11-01,1440.0
wfp_wheat_Peshawar,2018-12-01,1440.0
wfp_wheat_Peshawar,2019-01-01,1464.0
wfp_wheat_Peshawar,2019-02-01,1480.0
wfp_wheat_Peshawar,2019-03-01,1480.0
wfp_wheat_Peshawar,2019-04-01,1480.0
wfp_wheat_Peshawar,2019-05-01,1544.0
wfp_wheat_Peshawar,2019-06-01,1595.2
wfp_wheat_Peshawar,2019-07-01,1626.8
wfp_wheat_Peshawar,2019-08-01,1541.2
wfp_wheat_Peshawar,2019-09-01,1580.0
wfp_wheat_Peshawar,2019-10-01,1648.0
wfp_wheat_Peshawar,2019-11-01,1680.0
wfp_wheat_Peshawar,2019-12-01,1690.0
wfp_wheat_Peshawar,2020-01-01,1760.0
wfp_wheat_Peshawar,2020-02-01,1710.0
wfp_wheat_Peshawar,2020-03-01,1780.0
wfp_wheat_Peshawar,2020-04-01,1672.0
wfp_wheat_Peshawar,2020-05-01,1620.0
wfp_wheat_Peshawar,2020-06-01,1950.0
wfp_wheat_Peshawar,2020-07-01,2096.0
wfp_wheat_Peshawar,2020-08-01,2167.6
wfp_wheat_Peshawar,2020-09-01,2241.2
wfp_wheat_Peshawar,2020-10-01,2281.2
wfp_wheat_Peshawar,2020-11-01,2229.2
wfp_wheat_Peshawar,2020-12-01,1986.4
wfp_wheat_Peshawar,2021-01-01,2083.2
wfp_wheat_Peshawar,2021-02-01,2089.6
wfp_wheat_Peshawar,2021-03-01,2066.4
wfp_wheat_Peshawar,2021-04-01,1936.0
wfp_wheat_Peshawar,2021-05-01,1937.6
wfp_wheat_Peshawar,2021-06-01,1970.0
wfp_wheat_Peshawar,2021-07-01,2000.0
wfp_wheat_Peshawar,2021-08-01,2008.8
wfp_wheat_Peshawar,2021-09-01,2093.2
wfp_wheat_Peshawar,2021-10-01,2326.4
wfp_wheat_Peshawar,2021-11-01,2263.2
wfp_wheat_Peshawar,2021-12-01,2218.4
wfp_wheat_Peshawar,2022-01-01,2380.0
wfp_wheat_Peshawar,2022-02-01,2343.2
wfp_wheat_Peshawar,2022-03-01,2346.8
wfp_wheat_Peshawar,2022-04-01,2483.2
wfp_wheat_Peshawar,2022-05-01,2680.0
wfp_wheat_Peshawar,2022-06-01,3056.0
wfp_wheat_Peshawar,2022-07-01,3210.0
wfp_wheat_Peshawar,2022-08-01,3200.0
wfp_wheat_Peshawar,2022-09-01,3471.6
wfp_wheat_Peshawar,2022-10-01,3613.2
wfp_wheat_Peshawar,2022-11-01,3620.0
wfp_wheat_Peshawar,2022-12-01,4064.0
wfp_wheat_Peshawar,2023-01-01,4930.0
wfp_wheat_Peshawar,2023-02-01,4810.0
wfp_wheat_Peshawar,2023-03-01,5104.0
wfp_wheat_Peshawar,2023-04-01,4960.0
wfp_wheat_Peshawar,2023-05-01,5140.0
wfp_wheat_Peshawar,2023-06-01,4800.0
wfp_wheat_Peshawar,2023-07-01,5000.0
wfp_wheat_Peshawar,2023-08-01,4920.0
wfp_wheat_Peshawar,2023-09-01,4920.0
wfp_wheat_Peshawar,2023-10-01,4780.0
wfp_wheat_Peshawar,2023-11-01,4940.0
wfp_wheat_Peshawar,2023-12-01,5080.0
wfp_wheat_Peshawar,2024-01-01,5226.8
wfp_wheat_Peshawar,2024-02-01,5120.0
wfp_wheat_Peshawar,2024-03-01,4922.0
wfp_wheat_Peshawar,2024-04-01,4100.0
wfp_wheat_Peshawar,2024-05-01,3532.8
wfp_wheat_Peshawar,2024-06-01,3153.2
wfp_wheat_Peshawar,2024-07-01,3440.0
wfp_wheat_Peshawar,2024-08-01,3180.0
wfp_wheat_Peshawar,2024-09-01,2980.0
wfp_wheat_Peshawar,2024-10-01,3240.0
wfp_wheat_Peshawar,2024-11-01,3263.2
wfp_wheat_Peshawar,2024-12-01,3280.0
wfp_wheat_Peshawar,2025-01-01,3240.0
wfp_wheat_Peshawar,2025-02-01,3206.4
wfp_wheat_Peshawar,2025-03-01,3286.4
wfp_wheat_Peshawar,2025-04-01,3013.2
wfp_wheat_Peshawar,2025-05-01,2726.8
wfp_wheat_Peshawar,2025-06-01,2740.0
wfp_wheat_Peshawar,2025-07-01,2620.0
wfp_wheat_Peshawar,2025-08-01,2993.2
wfp_wheat_Peshawar,2025-09-01,4292.8
wfp_wheat_Peshawar,2025-10-01,4416.8
wfp_wheat_Peshawar,2025-11-01,4600.0
wfp_wheat_Peshawar,2025-12-01,4692.0
wfp_wheat_Peshawar,2026-01-01,5280.0
wfp_wheat_Peshawar,2026-02-01,5040.0
wfp_wheat_Peshawar,2026-03-01,4840.0
wfp_wheat_Peshawar,2026-04-01,4600.0
wfp_wheat_Peshawar,2026-05-01,4800.0
wfp_wheat_Peshawar,2026-06-01,5040.0
wfp_wheat_Peshawar,2026-07-01,5320.0
wfp_wheat_Peshawar,2026-08-01,5600.0
wfp_wheat_Peshawar,2026-09-01,5440.0
wfp_wheat_Quetta,2014-12-01,1540.0
wfp_wheat_Quetta,2015-01-01,1540.0
wfp_wheat_Quetta,2015-02-01,1540.0
wfp_wheat_Quetta,2015-03-01,1540.0
wfp_wheat_Quetta,2015-04-01,1540.0
wfp_wheat_Quetta,2015-05-01,1540.0
wfp_wheat_Quetta,2015-06-01,1440.0
wfp_wheat_Quetta,2015-07-01,1400.0
wfp_wheat_Quetta,2015-08-01,1400.0
wfp_wheat_Quetta,2015-09-01,1400.0
wfp_wheat_Quetta,2015-10-01,1408.0
wfp_wheat_Quetta,2015-11-01,1440.0
wfp_wheat_Quetta,2015-12-01,1472.0
wfp_wheat_Quetta,2016-01-01,1500.0
wfp_wheat_Quetta,2016-02-01,1480.0
wfp_wheat_Quetta,2016-03-01,1480.0
wfp_wheat_Quetta,2016-04-01,1440.0
wfp_wheat_Quetta,2016-05-01,1410.0
wfp_wheat_Quetta,2016-06-01,1400.0
wfp_wheat_Quetta,2016-07-01,1400.0
wfp_wheat_Quetta,2016-08-01,1400.0
wfp_wheat_Quetta,2016-09-01,1400.0
wfp_wheat_Quetta,2016-10-01,1400.0
wfp_wheat_Quetta,2016-11-01,1400.0
wfp_wheat_Quetta,2016-12-01,1400.0
wfp_wheat_Quetta,2017-01-01,1400.0
wfp_wheat_Quetta,2017-02-01,1400.0
wfp_wheat_Quetta,2017-03-01,1400.0
wfp_wheat_Quetta,2017-04-01,1400.0
wfp_wheat_Quetta,2017-05-01,1400.0
wfp_wheat_Quetta,2017-06-01,1400.0
wfp_wheat_Quetta,2017-07-01,1400.0
wfp_wheat_Quetta,2017-08-01,1400.0
wfp_wheat_Quetta,2017-09-01,1400.0
wfp_wheat_Quetta,2017-10-01,1400.0
wfp_wheat_Quetta,2017-11-01,1400.0
wfp_wheat_Quetta,2017-12-01,1400.0
wfp_wheat_Quetta,2018-01-01,1400.0
wfp_wheat_Quetta,2018-02-01,1400.0
wfp_wheat_Quetta,2018-03-01,1400.0
wfp_wheat_Quetta,2018-04-01,1400.0
wfp_wheat_Quetta,2018-05-01,1400.0
wfp_wheat_Quetta,2018-06-01,1400.0
wfp_wheat_Quetta,2018-07-01,1400.0
wfp_wheat_Quetta,2018-08-01,1400.0
wfp_wheat_Quetta,2018-09-01,1400.0
wfp_wheat_Quetta,2018-10-01,1400.0
wfp_wheat_Quetta,2018-11-01,1400.0
wfp_wheat_Quetta,2018-12-01,1420.0
wfp_wheat_Quetta,2019-01-01,1440.0
wfp_wheat_Quetta,2019-02-01,1440.0
wfp_wheat_Quetta,2019-03-01,1440.0
wfp_wheat_Quetta,2019-04-01,1440.0
wfp_wheat_Quetta,2019-05-01,1440.0
wfp_wheat_Quetta,2019-06-01,1440.0
wfp_wheat_Quetta,2019-07-01,1440.0
wfp_wheat_Quetta,2019-08-01,1472.0
wfp_wheat_Quetta,2019-09-01,1480.0
wfp_wheat_Quetta,2019-10-01,1480.0
wfp_wheat_Quetta,2019-11-01,1580.0
wfp_wheat_Quetta,2019-12-01,1760.0
wfp_wheat_Quetta,2020-01-01,1945.2
wfp_wheat_Quetta,2020-02-01,2175.2
wfp_wheat_Quetta,2020-03-01,2040.0
wfp_wheat_Quetta,2020-04-01,1960.0
wfp_wheat_Quetta,2020-05-01,1880.0
wfp_wheat_Quetta,2020-06-01,1850.0
wfp_wheat_Quetta,2020-07-01,1967.6
wfp_wheat_Quetta,2020-08-01,2069.6
wfp_wheat_Quetta,2020-09-01,2200.0
wfp_wheat_Quetta,2020-10-01,2276.0
wfp_wheat_Quetta,2020-11-01,2290.0
wfp_wheat_Quetta,2020-12-01,2292.0
wfp_wheat_Quetta,2021-01-01,2410.0
wfp_wheat_Quetta,2021-02-01,2440.0
wfp_wheat_Quetta,2021-03-01,2440.0
wfp_wheat_Quetta,2021-04-01,2440.0
wfp_wheat_Quetta,2021-05-01,2426.4
wfp_wheat_Quetta,2021-06-01,2380.0
wfp_wheat_Quetta,2021-07-01,2380.0
wfp_wheat_Quetta,2021-08-01,2380.0
wfp_wheat_Quetta,2021-09-01,2380.0
wfp_wheat_Quetta,2021-10-01,2428.4
wfp_wheat_Quetta,2021-11-01,2440.0
wfp_wheat_Quetta,2021-12-01,2420.0
wfp_wheat_Quetta,2022-01-01,2500.0
wfp_wheat_Quetta,2022-02-01,2530.0
wfp_wheat_Quetta,2022-03-01,2520.0
wfp_wheat_Quetta,2022-04-01,2520.0
wfp_wheat_Quetta,2022-05-01,2593.2
wfp_wheat_Quetta,2022-06-01,2856.0
wfp_wheat_Quetta,2022-07-01,3150.0
wfp_wheat_Quetta,2022-08-01,3150.0
wfp_wheat_Quetta,2022-09-01,3396.0
wfp_wheat_Quetta,2022-10-01,3410.0
wfp_wheat_Quetta,2022-11-01,3460.0
wfp_wheat_Quetta,2022-12-01,3800.0
wfp_wheat_Quetta,2023-01-01,4940.0
wfp_wheat_Quetta,2023-02-01,4970.0
wfp_wheat_Quetta,2023-03-01,5152.0
wfp_wheat_Quetta,2023-04-01,5020.0
wfp_wheat_Quetta,2023-05-01,5300.0
wfp_wheat_Quetta,2023-06-01,4880.0
wfp_wheat_Quetta,2023-07-01,4980.0
wfp_wheat_Quetta,2023-08-01,4960.0
wfp_wheat_Quetta,2023-09-01,4840.0
wfp_wheat_Quetta,2023-10-01,4740.0
wfp_wheat_Quetta,2023-11-01,4940.0
wfp_wheat_Quetta,2023-12-01,5160.0
wfp_wheat_Quetta,2024-01-01,5160.0
wfp_wheat_Quetta,2024-02-01,5160.0
wfp_wheat_Quetta,2024-03-01,5000.0
wfp_wheat_Quetta,2024-04-01,4660.0
wfp_wheat_Quetta,2024-05-01,3919.2
wfp_wheat_Quetta,2024-06-01,3640.0
wfp_wheat_Quetta,2024-07-01,3720.0
wfp_wheat_Quetta,2024-08-01,3640.0
wfp_wheat_Quetta,2024-09-01,3480.0
wfp_wheat_Quetta,2024-10-01,3440.0
wfp_wheat_Quetta,2024-11-01,3480.0
wfp_wheat_Quetta,2024-12-01,3420.0
wfp_wheat_Quetta,2025-01-01,3330.0
wfp_wheat_Quetta,2025-02-01,3240.0
wfp_wheat_Quetta,2025-03-01,3220.0
wfp_wheat_Quetta,2025-04-01,2920.0
wfp_wheat_Quetta,2025-05-01,2580.0
wfp_wheat_Quetta,2025-06-01,2560.0
wfp_wheat_Quetta,2025-07-01,2560.0
wfp_wheat_Quetta,2025-08-01,2800.0
wfp_wheat_Quetta,2025-09-01,3779.6
wfp_wheat_Quetta,2025-10-01,3880.0
wfp_wheat_Quetta,2025-11-01,4160.0
wfp_wheat_Quetta,2025-12-01,4600.0
wfp_wheat_Quetta,2026-01-01,4920.0
wfp_wheat_Quetta,2026-02-01,5120.0
wfp_wheat_Quetta,2026-03-01,5040.0
wfp_wheat_Quetta,2026-04-01,4720.0
wfp_wheat_Quetta,2026-05-01,4760.0
wfp_wheat_Quetta,2026-06-01,4800.0
wfp_wheat_Quetta,2026-07-01,4760.0
wfp_wheat_Quetta,2026-08-01,4800.0
wfp_wheat_Quetta,2026-09-01,5160.0
wfp_wheat_Punjab,2006-01-01,476.0
wfp_wheat_Punjab,2006-02-01,476.0
wfp_wheat_Punjab,2006-03-01,474.0
wfp_wheat_Punjab,2006-04-01,452.0
wfp_wheat_Punjab,2006-05-01,442.0
wfp_wheat_Punjab,2006-06-01,438.0
wfp_wheat_Punjab,2006-07-01,442.0
wfp_wheat_Punjab,2006-08-01,462.0
wfp_wheat_Punjab,2006-09-01,466.0
wfp_wheat_Punjab,2006-10-01,466.0
wfp_wheat_Punjab,2006-11-01,472.0
wfp_wheat_Punjab,2006-12-01,488.0
wfp_wheat_Punjab,2007-01-01,489.6
wfp_wheat_Punjab,2007-02-01,492.6
wfp_wheat_Punjab,2007-03-01,491.6
wfp_wheat_Punjab,2007-04-01,491.6
wfp_wheat_Punjab,2007-05-01,450.8
wfp_wheat_Punjab,2007-06-01,469.8
wfp_wheat_Punjab,2007-07-01,498.0
wfp_wheat_Punjab,2007-08-01,498.2
wfp_wheat_Punjab,2007-09-01,558.4
wfp_wheat_Punjab,2007-10-01,582.6
wfp_wheat_Punjab,2007-11-01,618.0
wfp_wheat_Punjab,2007-12-01,679.2
wfp_wheat_Punjab,2008-01-01,724.2
wfp_wheat_Punjab,2008-02-01,627.4
wfp_wheat_Punjab,2008-03-01,631.2
wfp_wheat_Punjab,2008-04-01,671.2
wfp_wheat_Punjab,2008-05-01,748.8
wfp_wheat_Punjab,2008-06-01,739.2
wfp_wheat_Punjab,2008-07-01,766.6
wfp_wheat_Punjab,2008-08-01,776.2
wfp_wheat_Punjab,2008-09-01,764.0
wfp_wheat_Punjab,2008-10-01,874.0
wfp_wheat_Punjab,2008-11-01,865.0
wfp_wheat_Punjab,2008-12-01,848.8
wfp_wheat_Punjab,2009-01-01,878.0
wfp_wheat_Punjab,2009-02-01,905.8
wfp_wheat_Punjab,2009-03-01,908.8
wfp_wheat_Punjab,2009-04-01,941.2
wfp_wheat_Punjab,2009-05-01,938.2
wfp_wheat_Punjab,2009-06-01,923.8
wfp_wheat_Punjab,2009-07-01,946.8
wfp_wheat_Punjab,2009-08-01,984.4
wfp_wheat_Punjab,2009-09-01,993.6
wfp_wheat_Punjab,2009-10-01,1000.4
wfp_wheat_Punjab,2009-11-01,1014.0
wfp_wheat_Punjab,2009-12-01,1033.4
wfp_wheat_Punjab,2010-01-01,1070.2
wfp_wheat_Punjab,2010-02-01,1073.8
wfp_wheat_Punjab,2010-03-01,1071.4
wfp_wheat_Punjab,2010-04-01,981.4
wfp_wheat_Punjab,2010-05-01,908.8
wfp_wheat_Punjab,2010-06-01,913.8
wfp_wheat_Punjab,2010-07-01,909.8
wfp_wheat_Punjab,2010-08-01,922.6
wfp_wheat_Punjab,2010-09-01,985.2
wfp_wheat_Punjab,2010-10-01,1031.2
wfp_wheat_Punjab,2010-11-01,1053.0
wfp_wheat_Punjab,2010-12-01,1058.8
wfp_wheat_Punjab,2011-01-01,1060.0
wfp_wheat_Punjab,2011-02-01,1062.4
wfp_wheat_Punjab,2011-03-01,1067.2
wfp_wheat_Punjab,2011-04-01,1008.4
wfp_wheat_Punjab,2011-05-01,915.0
wfp_wheat_Punjab,2011-06-01,935.2
wfp_wheat_Punjab,2011-07-01,953.8
wfp_wheat_Punjab,2011-08-01,960.8
wfp_wheat_Punjab,2011-09-01,998.2
wfp_wheat_Punjab,2011-10-01,1012.6
wfp_wheat_Punjab,2011-11-01,1043.8
wfp_wheat_Punjab,2011-12-01,1085.8
wfp_wheat_Punjab,2012-01-01,1079.4
wfp_wheat_Punjab,2012-02-01,1078.6
wfp_wheat_Punjab,2012-03-01,1083.8
wfp_wheat_Punjab,2012-04-01,1066.6
wfp_wheat_Punjab,2012-05-01,1013.0
wfp_wheat_Punjab,2012-06-01,1041.4
wfp_wheat_Punjab,2012-07-01,1035.0
wfp_wheat_Punjab,2012-08-01,1081.6
wfp_wheat_Punjab,2012-09-01,1112.2
wfp_wheat_Punjab,2012-10-01,1122.4
wfp_wheat_Punjab,2012-11-01,1150.2
wfp_wheat_Punjab,2012-12-01,1201.6
wfp_wheat_Punjab,2013-01-01,1267.8
wfp_wheat_Punjab,2013-02-01,1255.2
wfp_wheat_Punjab,2013-03-01,1256.4
wfp_wheat_Punjab,2013-04-01,1209.8
wfp_wheat_Punjab,2013-05-01,1205.4
wfp_wheat_Punjab,2013-06-01,1277.2
wfp_wheat_Punjab,2013-07-01,1326.4
wfp_wheat_Punjab,2013-08-01,1394.6
wfp_wheat_Punjab,2013-09-01,1433.6
wfp_wheat_Punjab,2013-10-01,1438.4
wfp_wheat_Punjab,2013-11-01,1497.4
wfp_wheat_Punjab,2013-12-01,1568.6
wfp_wheat_Punjab,2014-01-01,1568.4
wfp_wheat_Punjab,2014-02-01,1591.2
wfp_wheat_Punjab,2014-03-01,1625.4
wfp_wheat_Punjab,2014-04-01,1444.0
wfp_wheat_Punjab,2014-05-01,1272.0
wfp_wheat_Punjab,2014-06-01,1282.4
wfp_wheat_Punjab,2014-07-01,1316.4
wfp_wheat_Punjab,2014-08-01,1388.8
wfp_wheat_Punjab,2014-09-01,1371.4
wfp_wheat_Punjab,2014-10-01,1370.4
wfp_wheat_Punjab,2014-11-01,1381.2
wfp_wheat_Punjab,2014-12-01,1367.4
wfp_wheat_Punjab,2015-01-01,1365.0
wfp_wheat_Punjab,2015-02-01,1363.0
wfp_wheat_Punjab,2015-03-01,1365.0
wfp_wheat_Punjab,2015-04-01,1310.6
wfp_wheat_Punjab,2015-05-01,1273.2
wfp_wheat_Punjab,2015-06-01,1245.0
wfp_wheat_Punjab,2015-07-01,1289.4
wfp_wheat_Punjab,2015-08-01,1296.2
wfp_wheat_Punjab,2015-09-01,1292.4
wfp_wheat_Punjab,2015-10-01,1340.4
wfp_wheat_Punjab,2015-11-01,1372.6
wfp_wheat_Punjab,2015-12-01,1379.4
wfp_wheat_Punjab,2016-01-01,1382.4
wfp_wheat_Punjab,2016-02-01,1398.6
wfp_wheat_Punjab,2016-03-01,1402.4
wfp_wheat_Punjab,2016-04-01,1322.4
wfp_wheat_Punjab,2016-05-01,1262.4
wfp_wheat_Punjab,2016-06-01,1278.6
wfp_wheat_Punjab,2016-07-01,1303.4
wfp_wheat_Punjab,2016-08-01,1312.4
wfp_wheat_Punjab,2016-09-01,1316.6
wfp_wheat_Punjab,2016-10-01,1335.2
wfp_wheat_Punjab,2016-11-01,1346.2
wfp_wheat_Punjab,2016-12-01,1363.4
wfp_wheat_Punjab,2017-01-01,1362.4
wfp_wheat_Punjab,2017-02-01,1382.4
wfp_wheat_Punjab,2017-03-01,1380.4
wfp_wheat_Punjab,2017-04-01,1333.8
wfp_wheat_Punjab,2017-05-01,1258.8
wfp_wheat_Punjab,2017-06-01,1252.6
wfp_wheat_Punjab,2017-07-01,1242.4
wfp_wheat_Punjab,2017-08-01,1222.0
wfp_wheat_Punjab,2017-09-01,1233.2
wfp_wheat_Punjab,2017-10-01,1249.6
wfp_wheat_Punjab,2017-11-01,1313.4
wfp_wheat_Punjab,2017-12-01,1355.0
wfp_wheat_Punjab,2018-01-01,1355.0
wfp_wheat_Punjab,2018-02-01,1347.6
wfp_wheat_Punjab,2018-03-01,1344.0
wfp_wheat_Punjab,2018-04-01,1310.0
wfp_wheat_Punjab,2018-05-01,1245.0
wfp_wheat_Punjab,2018-06-01,1295.0
wfp_wheat_Punjab,2018-07-01,1297.4
wfp_wheat_Punjab,2018-08-01,1320.0
wfp_wheat_Punjab,2018-09-01,1331.2
wfp_wheat_Punjab,2018-10-01,1350.0
wfp_wheat_Punjab,2018-11-01,1354.0
wfp_wheat_Punjab,2018-12-01,1355.0
wfp_wheat_Punjab,2019-01-01,1360.0
wfp_wheat_Punjab,2019-02-01,1370.0
wfp_wheat_Punjab,2019-03-01,1375.0
wfp_wheat_Punjab,2019-04-01,1345.4
wfp_wheat_Punjab,2019-05-01,1342.0
wfp_wheat_Punjab,2019-06-01,1386.2
wfp_wheat_Punjab,2019-07-01,1410.0
wfp_wheat_Punjab,2019-08-01,1412.0
wfp_wheat_Punjab,2019-09-01,1430.6
wfp_wheat_Punjab,2019-10-01,1435.6
wfp_wheat_Punjab,2019-11-01,1520.6
wfp_wheat_Punjab,2019-12-01,1677.0
wfp_wheat_Punjab,2020-01-01,1876.6
wfp_wheat_Punjab,2020-02-01,1747.8
wfp_wheat_Punjab,2020-03-01,1708.0
wfp_wheat_Punjab,2020-04-01,1662.0
wfp_wheat_Punjab,2020-05-01,1532.6
wfp_wheat_Punjab,2020-06-01,1761.4
wfp_wheat_Punjab,2020-07-01,1961.8
wfp_wheat_Punjab,2020-08-01,2125.0
wfp_wheat_Punjab,2020-09-01,2153.6
wfp_wheat_Punjab,2020-10-01,2323.0
wfp_wheat_Punjab,2020-11-01,2291.2
wfp_wheat_Punjab,2020-12-01,2230.0
wfp_wheat_Punjab,2021-01-01,2356.6
wfp_wheat_Punjab,2021-02-01,2347.4
wfp_wheat_Punjab,2021-03-01,2345.0
wfp_wheat_Punjab,2021-04-01,2045.2
wfp_wheat_Punjab,2021-05-01,1965.4
wfp_wheat_Punjab,2021-06-01,2032.6
wfp_wheat_Punjab,2021-07-01,2045.0
wfp_wheat_Punjab,2021-08-01,2048.4
wfp_wheat_Punjab,2021-09-01,2135.0
wfp_wheat_Punjab,2021-10-01,2313.6
wfp_wheat_Punjab,2021-11-01,2352.0
wfp_wheat_Punjab,2021-12-01,2375.0
wfp_wheat_Punjab,2022-01-01,2455.4
wfp_wheat_Punjab,2022-02-01,2458.0
wfp_wheat_Punjab,2022-03-01,2452.4
wfp_wheat_Punjab,2022-04-01,2382.0
wfp_wheat_Punjab,2022-05-01,2388.8
wfp_wheat_Punjab,2022-06-01,2542.8
wfp_wheat_Punjab,2022-07-01,2699.8
wfp_wheat_Punjab,2022-08-01,2807.8
wfp_wheat_Punjab,2022-09-01,3114.4
wfp_wheat_Punjab,2022-10-01,3305.6
wfp_wheat_Punjab,2022-11-01,3346.6
wfp_wheat_Punjab,2022-12-01,3385.4
wfp_wheat_Punjab,2023-01-01,4703.2
wfp_wheat_Punjab,2023-02-01,4435.0
wfp_wheat_Punjab,2023-03-01,4548.8
wfp_wheat_Punjab,2023-04-01,4506.0
wfp_wheat_Punjab,2023-05-01,4257.4
wfp_wheat_Punjab,2023-06-01,4348.6
wfp_wheat_Punjab,2023-07-01,4877.4
wfp_wheat_Punjab,2023-08-01,4856.2
wfp_wheat_Punjab,2023-09-01,4787.4
wfp_wheat_Punjab,2023-10-01,4712.4
wfp_wheat_Punjab,2023-11-01,4821.4
wfp_wheat_Punjab,2023-12-01,5000.0
wfp_wheat_Punjab,2024-01-01,5000.0
wfp_wheat_Punjab,2024-02-01,4987.6
wfp_wheat_Punjab,2024-03-01,4837.6
wfp_wheat_Punjab,2024-04-01,4087.4
wfp_wheat_Punjab,2024-05-01,3333.0
wfp_wheat_Punjab,2024-06-01,3149.8
wfp_wheat_Punjab,2024-07-01,3236.8
wfp_wheat_Punjab,2024-08-01,2999.4
wfp_wheat_Punjab,2024-09-01,2739.8
wfp_wheat_Punjab,2024-10-01,2872.4
wfp_wheat_Punjab,2024-11-01,3074.8
wfp_wheat_Punjab,2024-12-01,3111.8
wfp_wheat_Punjab,2025-01-01,3074.8
wfp_wheat_Punjab,2025-02-01,3074.8
wfp_wheat_Punjab,2025-03-01,3086.8
wfp_wheat_Punjab,2025-04-01,2779.4
wfp_wheat_Punjab,2025-05-01,2504.4
wfp_wheat_Punjab,2025-06-01,2548.8
wfp_wheat_Punjab,2025-07-01,2523.8
wfp_wheat_Punjab,2025-08-01,2874.0
wfp_wheat_Punjab,2025-09-01,3693.6
wfp_wheat_Punjab,2025-10-01,3710.6
wfp_wheat_Punjab,2025-11-01,3820.0
wfp_wheat_Punjab,2025-12-01,3912.0
wfp_wheat_Punjab,2026-01-01,4500.0
wfp_wheat_Punjab,2026-02-01,4680.0
wfp_wheat_Punjab,2026-03-01,4320.0
wfp_wheat_Punjab,2026-04-01,3960.0
wfp_wheat_Punjab,2026-05-01,3980.0
wfp_wheat_Punjab,2026-06-01,4220.0
wfp_wheat_Punjab,2026-07-01,4640.0
wfp_wheat_Punjab,2026-08-01,4920.0
fao_flour_Quetta,2006-01-01,512.0
fao_flour_Quetta,2006-02-01,520.0
fao_flour_Quetta,2006-03-01,520.0
fao_flour_Quetta,2006-04-01,520.0
fao_flour_Quetta,2006-05-01,528.0
fao_flour_Quetta,2006-06-01,532.0
fao_flour_Quetta,2006-07-01,528.0
fao_flour_Quetta,2006-08-01,532.0
fao_flour_Quetta,2006-09-01,536.0
fao_flour_Quetta,2006-10-01,528.0
fao_flour_Quetta,2006-11-01,528.0
fao_flour_Quetta,2006-12-01,552.0
fao_flour_Quetta,2007-01-01,565.2
fao_flour_Quetta,2007-02-01,565.2
fao_flour_Quetta,2007-03-01,565.2
fao_flour_Quetta,2007-04-01,580.0
fao_flour_Quetta,2007-05-01,564.0
fao_flour_Quetta,2007-06-01,580.0
fao_flour_Quetta,2007-07-01,607.6
fao_flour_Quetta,2007-08-01,630.0
fao_flour_Quetta,2007-09-01,628.0
fao_flour_Quetta,2007-10-01,690.0
fao_flour_Quetta,2007-11-01,760.0
fao_flour_Quetta,2007-12-01,795.2
fao_flour_Quetta,2008-01-01,844.0
fao_flour_Quetta,2008-02-01,610.0
fao_flour_Quetta,2008-03-01,610.0
fao_flour_Quetta,2008-04-01,1100.0
fao_flour_Quetta,2008-05-01,1200.0
fao_flour_Quetta,2008-06-01,1080.0
fao_flour_Quetta,2008-07-01,1104.0
fao_flour_Quetta,2008-08-01,1100.0
fao_flour_Quetta,2008-09-01,1080.0
fao_flour_Quetta,2008-10-01,1320.0
fao_flour_Quetta,2008-11-01,1320.0
fao_flour_Quetta,2008-12-01,1300.0
fao_flour_Quetta,2009-01-01,1144.0
fao_flour_Quetta,2009-02-01,1080.0
fao_flour_Quetta,2009-03-01,1080.0
fao_flour_Quetta,2009-04-01,1143.2
fao_flour_Quetta,2009-05-01,1159.2
fao_flour_Quetta,2009-06-01,1159.2
fao_flour_Quetta,2009-07-01,1143.6
fao_flour_Quetta,2009-08-01,1081.6
fao_flour_Quetta,2009-09-01,1081.6
fao_flour_Quetta,2009-10-01,1081.6
fao_flour_Quetta,2009-11-01,1081.6
fao_flour_Quetta,2009-12-01,1081.6
fao_flour_Quetta,2010-01-01,1192.8
fao_flour_Quetta,2010-02-01,1230.0
fao_flour_Quetta,2010-03-01,1230.0
fao_flour_Quetta,2010-04-01,1212.0
fao_flour_Quetta,2010-05-01,1120.0
fao_flour_Quetta,2010-06-01,1060.0
fao_flour_Quetta,2010-07-01,1060.0
fao_flour_Quetta,2010-08-01,1075.2
fao_flour_Quetta,2010-09-01,1132.0
fao_flour_Quetta,2010-10-01,1195.2
fao_flour_Quetta,2010-11-01,1210.0
fao_flour_Quetta,2010-12-01,1220.0
fao_flour_Quetta,2011-01-01,1220.0
fao_flour_Quetta,2011-02-01,1235.2
fao_flour_Quetta,2011-03-01,1240.0
fao_flour_Quetta,2011-04-01,1240.0
fao_flour_Quetta,2011-05-01,1210.0
fao_flour_Quetta,2011-06-01,1200.0
fao_flour_Quetta,2011-07-01,1215.2
fao_flour_Quetta,2011-08-01,1200.0
fao_flour_Quetta,2011-09-01,1200.0
fao_flour_Quetta,2011-10-01,1200.0
fao_flour_Quetta,2011-11-01,1200.0
fao_flour_Quetta,2011-12-01,1260.0
fao_flour_Quetta,2012-01-01,1300.0
fao_flour_Quetta,2012-02-01,1300.0
fao_flour_Quetta,2012-03-01,1308.0
fao_flour_Quetta,2012-04-01,1320.0
fao_flour_Quetta,2012-05-01,1320.0
fao_flour_Quetta,2012-06-01,1320.0
fao_flour_Quetta,2012-07-01,1320.0
fao_flour_Quetta,2012-08-01,1320.0
fao_flour_Quetta,2012-09-01,1360.0
fao_flour_Quetta,2012-10-01,1360.0
fao_flour_Quetta,2012-11-01,1360.0
fao_flour_Quetta,2012-12-01,1360.0
fao_flour_Quetta,2013-01-01,1408.0
fao_flour_Quetta,2013-02-01,1420.0
fao_flour_Quetta,2013-03-01,1420.0
fao_flour_Quetta,2013-04-01,1420.0
fao_flour_Quetta,2013-05-01,1420.0
fao_flour_Quetta,2013-06-01,1510.0
fao_flour_Quetta,2013-07-01,1600.0
fao_flour_Quetta,2013-08-01,1600.0
fao_flour_Quetta,2013-09-01,1610.0
fao_flour_Quetta,2013-10-01,1620.0
fao_flour_Quetta,2013-11-01,1724.8
fao_flour_Quetta,2013-12-01,1880.0
fao_flour_Quetta,2014-01-01,1880.0
fao_flour_Quetta,2014-02-01,1880.0
fao_flour_Quetta,2014-03-01,1890.0
fao_flour_Quetta,2014-04-01,1900.0
fao_flour_Quetta,2014-05-01,1780.0
fao_flour_Quetta,2014-06-01,1640.0
fao_flour_Quetta,2014-07-01,1640.0
fao_flour_Quetta,2014-08-01,1640.0
fao_flour_Quetta,2014-09-01,1640.0
fao_flour_Quetta,2014-10-01,1640.0
fao_flour_Quetta,2014-11-01,1640.0
fao_flour_Quetta,2014-12-01,1640.0
fao_flour_Quetta,2015-01-01,1640.0
fao_flour_Quetta,2015-02-01,1640.0
fao_flour_Quetta,2015-03-01,1640.0
fao_flour_Quetta,2015-04-01,1640.0
fao_flour_Quetta,2015-05-01,1640.0
fao_flour_Quetta,2015-06-01,1560.0
fao_flour_Quetta,2015-07-01,1520.0
fao_flour_Quetta,2015-08-01,1520.0
fao_flour_Quetta,2015-09-01,1520.0
fao_flour_Quetta,2015-10-01,1528.0
fao_flour_Quetta,2015-11-01,1560.0
fao_flour_Quetta,2015-12-01,1592.0
fao_flour_Quetta,2016-01-01,1620.0
fao_flour_Quetta,2016-02-01,1600.0
fao_flour_Quetta,2016-03-01,1600.0
fao_flour_Quetta,2016-04-01,1600.0
fao_flour_Quetta,2016-05-01,1600.0
fao_flour_Quetta,2016-06-01,1600.0
fao_flour_Quetta,2016-07-01,1600.0
fao_flour_Quetta,2016-08-01,1600.0
fao_flour_Quetta,2016-09-01,1600.0
fao_flour_Quetta,2016-10-01,1600.0
fao_flour_Quetta,2016-11-01,1600.0
fao_flour_Quetta,2016-12-01,1600.0
fao_flour_Quetta,2017-01-01,1600.0
fao_flour_Quetta,2017-02-01,1600.0
fao_flour_Quetta,2017-03-01,1600.0
fao_flour_Quetta,2017-04-01,1600.0
fao_flour_Quetta,2017-05-01,1600.0
fao_flour_Quetta,2017-06-01,1600.0
fao_flour_Quetta,2017-07-01,1600.0
fao_flour_Quetta,2017-08-01,1600.0
fao_flour_Quetta,2017-09-01,1600.0
fao_flour_Quetta,2017-10-01,1600.0
fao_flour_Quetta,2017-11-01,1600.0
fao_flour_Quetta,2017-12-01,1600.0
fao_flour_Quetta,2018-01-01,1600.0
fao_flour_Quetta,2018-02-01,1600.0
fao_flour_Quetta,2018-03-01,1600.0
fao_flour_Quetta,2018-04-01,1600.0
fao_flour_Quetta,2018-05-01,1600.0
fao_flour_Quetta,2018-06-01,1600.0
fao_flour_Quetta,2018-07-01,1600.0
fao_flour_Quetta,2018-08-01,1600.0
fao_flour_Quetta,2018-09-01,1620.0
fao_flour_Quetta,2018-10-01,1640.0
fao_flour_Quetta,2018-11-01,1640.0
fao_flour_Quetta,2018-12-01,1640.0
fao_flour_Quetta,2019-01-01,1640.0
fao_flour_Quetta,2019-02-01,1660.0
fao_flour_Quetta,2019-03-01,1680.0
fao_flour_Quetta,2019-04-01,1680.0
fao_flour_Quetta,2019-05-01,1628.0
fao_flour_Quetta,2019-06-01,1720.0
fao_flour_Quetta,2019-07-01,1720.0
fao_flour_Quetta,2019-08-01,1720.0
fao_flour_Quetta,2019-09-01,1810.0
fao_flour_Quetta,2019-10-01,1872.0
fao_flour_Quetta,2019-11-01,2053.2
fao_flour_Quetta,2019-12-01,2124.8
fao_flour_Quetta,2020-01-01,2211.2
fao_flour_Quetta,2020-02-01,2093.6
fao_flour_Quetta,2020-03-01,2034.0
fao_flour_Quetta,2020-04-01,2022.4
fao_flour_Quetta,2020-05-01,1914.8
fao_flour_Quetta,2020-06-01,2144.4
fao_flour_Quetta,2020-07-01,2182.8
fao_flour_Quetta,2020-08-01,2369.6
fao_flour_Quetta,2020-09-01,2394.8
fao_flour_Quetta,2020-10-01,2453.6
fao_flour_Quetta,2020-11-01,2530.0
fao_flour_Quetta,2020-12-01,2510.0
fao_flour_Quetta,2021-01-01,1991.2
fao_flour_Quetta,2021-02-01,1920.0
fao_flour_Quetta,2021-03-01,1920.0
fao_flour_Quetta,2021-04-01,2154.0
fao_flour_Quetta,2021-05-01,2490.0
fao_flour_Quetta,2021-06-01,2490.0
fao_flour_Quetta,2021-07-01,2462.4
fao_flour_Quetta,2021-08-01,2460.0
fao_flour_Quetta,2021-09-01,2576.0
fao_flour_Quetta,2021-10-01,2760.0
fao_flour_Quetta,2021-11-01,2770.0
fao_flour_Quetta,2021-12-01,2800.0
fao_flour_Quetta,2022-01-01,2890.0
fao_flour_Quetta,2022-02-01,2920.0
fao_flour_Quetta,2022-03-01,2832.0
fao_flour_Quetta,2022-04-01,2790.0
fao_flour_Quetta,2022-05-01,2972.4
fao_flour_Quetta,2022-06-01,3370.0
fao_flour_Quetta,2022-07-01,3640.0
fao_flour_Quetta,2022-08-01,3629.2
fao_flour_Quetta,2022-09-01,4660.0
fao_flour_Quetta,2022-10-01,4220.0
fao_flour_Quetta,2022-11-01,4036.4
fao_flour_Quetta,2022-12-01,4570.0
fao_flour_Quetta,2023-01-01,5308.0
fao_flour_Quetta,2023-02-01,5295.2
fao_flour_Quetta,2023-03-01,5642.4
fao_flour_Quetta,2023-04-01,5670.0
fao_flour_Quetta,2023-05-01,5808.0
fao_flour_Quetta,2023-06-01,5300.0
fao_flour_Quetta,2023-07-01,5560.0
fao_flour_Quetta,2023-08-01,5758.0
fao_flour_Quetta,2023-09-01,5595.2
fao_flour_Quetta,2023-10-01,5374.0
fao_flour_Quetta,2023-11-01,5620.0
fao_flour_Quetta,2023-12-01,5702.4
fao_flour_Quetta,2024-01-01,5784.4
fao_flour_Quetta,2024-02-01,5670.0
fao_flour_Quetta,2024-03-01,5590.0
fao_flour_Quetta,2024-04-01,5021.6
fao_flour_Quetta,2024-05-01,4284.8
fao_flour_Quetta,2024-06-01,4130.0
fao_flour_Quetta,2024-07-01,4207.2
fao_flour_Quetta,2024-08-01,4042.4
fao_flour_Quetta,2024-09-01,3762.4
fao_flour_Quetta,2024-10-01,3820.0
fao_flour_Quetta,2024-11-01,3826.4
fao_flour_Quetta,2024-12-01,3914.8
fao_flour_Quetta,2025-01-01,3842.4
fao_flour_Quetta,2025-02-01,3760.0
fao_flour_Quetta,2025-03-01,3770.0
fao_flour_Quetta,2025-04-01,3526.0
fao_flour_Quetta,2025-05-01,3230.0
fao_flour_Quetta,2025-06-01,3180.0
fao_flour_Quetta,2025-07-01,3164.0
fao_flour_Quetta,2025-08-01,3324.8
fao_flour_Quetta,2025-09-01,4574.0
fao_flour_Quetta,2025-10-01,4710.0
fao_flour_Quetta,2025-11-01,4740.0
fao_flour_Quetta,2025-12-01,4912.0
fao_flour_Quetta,2026-01-01,5484.8
fao_flour_Quetta,2026-02-01,5546.8
fao_flour_Quetta,2026-03-01,5370.0
fao_flour_Quetta,2026-04-01,4867.6
fao_flour_Quetta,2026-05-01,5143.2
fao_flour_Quetta,2026-06-01,5450.0
fao_flour_Quetta,2026-07-01,5544.8
fao_flour_Quetta,2026-08-01,5510.0
fao_flour_Quetta,2026-09-01,5890.0
fao_flour_Quetta,2026-10-01,6284.8
fao_flour_Peshawar,2006-01-01,544.0
fao_flour_Peshawar,2006-02-01,544.0
fao_flour_Peshawar,2006-03-01,548.0
fao_flour_Peshawar,2006-04-01,548.0
fao_flour_Peshawar,2006-05-01,528.0
fao_flour_Peshawar,2006-06-01,524.0
fao_flour_Peshawar,2006-07-01,532.0
fao_flour_Peshawar,2006-08-01,544.0
fao_flour_Peshawar,2006-09-01,540.0
fao_flour_Peshawar,2006-10-01,532.0
fao_flour_Peshawar,2006-11-01,544.0
fao_flour_Peshawar,2006-12-01,552.0
fao_flour_Peshawar,2007-01-01,606.8
fao_flour_Peshawar,2007-02-01,606.8
fao_flour_Peshawar,2007-03-01,606.8
fao_flour_Peshawar,2007-04-01,606.8
fao_flour_Peshawar,2007-05-01,573.2
fao_flour_Peshawar,2007-06-01,606.8
fao_flour_Peshawar,2007-07-01,640.0
fao_flour_Peshawar,2007-08-01,640.0
fao_flour_Peshawar,2007-09-01,712.8
fao_flour_Peshawar,2007-10-01,786.8
fao_flour_Peshawar,2007-11-01,826.8
fao_flour_Peshawar,2007-12-01,860.0
fao_flour_Peshawar,2008-01-01,932.0
fao_flour_Peshawar,2008-02-01,693.2
fao_flour_Peshawar,2008-03-01,772.0
fao_flour_Peshawar,2008-04-01,1014.8
fao_flour_Peshawar,2008-05-01,1132.8
fao_flour_Peshawar,2008-06-01,963.2
fao_flour_Peshawar,2008-07-01,1068.0
fao_flour_Peshawar,2008-08-01,1057.6
fao_flour_Peshawar,2008-09-01,1244.0
fao_flour_Peshawar,2008-10-01,1432.0
fao_flour_Peshawar,2008-11-01,1256.0
fao_flour_Peshawar,2008-12-01,1226.4
fao_flour_Peshawar,2009-01-01,1164.0
fao_flour_Peshawar,2009-02-01,1123.2
fao_flour_Peshawar,2009-03-01,1098.8
fao_flour_Peshawar,2009-04-01,1112.0
fao_flour_Peshawar,2009-05-01,1053.2
fao_flour_Peshawar,2009-06-01,1018.8
fao_flour_Peshawar,2009-07-01,1061.6
fao_flour_Peshawar,2009-08-01,1122.4
fao_flour_Peshawar,2009-09-01,1109.6
fao_flour_Peshawar,2009-10-01,1158.8
fao_flour_Peshawar,2009-11-01,1144.8
fao_flour_Peshawar,2009-12-01,1150.4
fao_flour_Peshawar,2010-01-01,1186.8
fao_flour_Peshawar,2010-02-01,1179.2
fao_flour_Peshawar,2010-03-01,1164.4
fao_flour_Peshawar,2010-04-01,1134.8
fao_flour_Peshawar,2010-05-01,1073.6
fao_flour_Peshawar,2010-06-01,1078.8
fao_flour_Peshawar,2010-07-01,1074.4
fao_flour_Peshawar,2010-08-01,1135.6
fao_flour_Peshawar,2010-09-01,1194.4
fao_flour_Peshawar,2010-10-01,1218.4
fao_flour_Peshawar,2010-11-01,1203.2
fao_flour_Peshawar,2010-12-01,1218.0
fao_flour_Peshawar,2011-01-01,1231.2
fao_flour_Peshawar,2011-02-01,1208.0
fao_flour_Peshawar,2011-03-01,1215.2
fao_flour_Peshawar,2011-04-01,1196.8
fao_flour_Peshawar,2011-05-01,1137.2
fao_flour_Peshawar,2011-06-01,1120.8
fao_flour_Peshawar,2011-07-01,1161.6
fao_flour_Peshawar,2011-08-01,1136.8
fao_flour_Peshawar,2011-09-01,1176.8
fao_flour_Peshawar,2011-10-01,1177.6
fao_flour_Peshawar,2011-11-01,1200.0
fao_flour_Peshawar,2011-12-01,1211.2
fao_flour_Peshawar,2012-01-01,1210.0
fao_flour_Peshawar,2012-02-01,1220.0
fao_flour_Peshawar,2012-03-01,1208.4
fao_flour_Peshawar,2012-04-01,1232.4
fao_flour_Peshawar,2012-05-01,1221.6
fao_flour_Peshawar,2012-06-01,1210.0
fao_flour_Peshawar,2012-07-01,1210.0
fao_flour_Peshawar,2012-08-01,1228.0
fao_flour_Peshawar,2012-09-01,1297.6
fao_flour_Peshawar,2012-10-01,1300.0
fao_flour_Peshawar,2012-11-01,1324.0
fao_flour_Peshawar,2012-12-01,1363.2
fao_flour_Peshawar,2013-01-01,1505.2
fao_flour_Peshawar,2013-02-01,1456.8
fao_flour_Peshawar,2013-03-01,1449.2
fao_flour_Peshawar,2013-04-01,1404.0
fao_flour_Peshawar,2013-05-01,1402.4
fao_flour_Peshawar,2013-06-01,1486.0
fao_flour_Peshawar,2013-07-01,1530.8
fao_flour_Peshawar,2013-08-01,1595.2
fao_flour_Peshawar,2013-09-01,1653.2
fao_flour_Peshawar,2013-10-01,1651.2
fao_flour_Peshawar,2013-11-01,1757.2
fao_flour_Peshawar,2013-12-01,1792.8
fao_flour_Peshawar,2014-01-01,1760.0
fao_flour_Peshawar,2014-02-01,1756.8
fao_flour_Peshawar,2014-03-01,1778.4
fao_flour_Peshawar,2014-04-01,1709.2
fao_flour_Peshawar,2014-05-01,1542.8
fao_flour_Peshawar,2014-06-01,1518.4
fao_flour_Peshawar,2014-07-01,1539.2
fao_flour_Peshawar,2014-08-01,1642.0
fao_flour_Peshawar,2014-09-01,1634.8
fao_flour_Peshawar,2014-10-01,1611.2
fao_flour_Peshawar,2014-11-01,1608.4
fao_flour_Peshawar,2014-12-01,1568.4
fao_flour_Peshawar,2015-01-01,1579.2
fao_flour_Peshawar,2015-02-01,1569.2
fao_flour_Peshawar,2015-03-01,1546.0
fao_flour_Peshawar,2015-04-01,1500.0
fao_flour_Peshawar,2015-05-01,1478.4
fao_flour_Peshawar,2015-06-01,1479.2
fao_flour_Peshawar,2015-07-01,1483.6
fao_flour_Peshawar,2015-08-01,1503.2
fao_flour_Peshawar,2015-09-01,1503.2
fao_flour_Peshawar,2015-10-01,1526.0
fao_flour_Peshawar,2015-11-01,1553.2
fao_flour_Peshawar,2015-12-01,1558.4
fao_flour_Peshawar,2016-01-01,1565.2
fao_flour_Peshawar,2016-02-01,1564.0
fao_flour_Peshawar,2016-03-01,1552.0
fao_flour_Peshawar,2016-04-01,1535.2
fao_flour_Peshawar,2016-05-01,1492.8
fao_flour_Peshawar,2016-06-01,1475.2
fao_flour_Peshawar,2016-07-01,1480.4
fao_flour_Peshawar,2016-08-01,1480.4
fao_flour_Peshawar,2016-09-01,1486.8
fao_flour_Peshawar,2016-10-01,1499.2
fao_flour_Peshawar,2016-11-01,1530.0
fao_flour_Peshawar,2016-12-01,1532.8
fao_flour_Peshawar,2017-01-01,1536.8
fao_flour_Peshawar,2017-02-01,1536.8
fao_flour_Peshawar,2017-03-01,1536.8
fao_flour_Peshawar,2017-04-01,1519.2
fao_flour_Peshawar,2017-05-01,1488.4
fao_flour_Peshawar,2017-06-01,1476.8
fao_flour_Peshawar,2017-07-01,1456.8
fao_flour_Peshawar,2017-08-01,1436.8
fao_flour_Peshawar,2017-09-01,1446.8
fao_flour_Peshawar,2017-10-01,1456.8
fao_flour_Peshawar,2017-11-01,1517.6
fao_flour_Peshawar,2017-12-01,1551.6
fao_flour_Peshawar,2018-01-01,1551.6
fao_flour_Peshawar,2018-02-01,1553.2
fao_flour_Peshawar,2018-03-01,1530.0
fao_flour_Peshawar,2018-04-01,1491.6
fao_flour_Peshawar,2018-05-01,1452.4
fao_flour_Peshawar,2018-06-01,1476.0
fao_flour_Peshawar,2018-07-01,1546.8
fao_flour_Peshawar,2018-08-01,1566.8
fao_flour_Peshawar,2018-09-01,1566.8
fao_flour_Peshawar,2018-10-01,1590.0
fao_flour_Peshawar,2018-11-01,1596.8
fao_flour_Peshawar,2018-12-01,1580.0
fao_flour_Peshawar,2019-01-01,1564.4
fao_flour_Peshawar,2019-02-01,1563.2
fao_flour_Peshawar,2019-03-01,1563.2
fao_flour_Peshawar,2019-04-01,1563.2
fao_flour_Peshawar,2019-05-01,1626.0
fao_flour_Peshawar,2019-06-01,1703.2
fao_flour_Peshawar,2019-07-01,1740.0
fao_flour_Peshawar,2019-08-01,1711.6
fao_flour_Peshawar,2019-09-01,1735.6
fao_flour_Peshawar,2019-10-01,1828.4
fao_flour_Peshawar,2019-11-01,1918.8
fao_flour_Peshawar,2019-12-01,1869.6
fao_flour_Peshawar,2020-01-01,2050.0
fao_flour_Peshawar,2020-02-01,1803.6
fao_flour_Peshawar,2020-03-01,1824.0
fao_flour_Peshawar,2020-04-01,1845.2
fao_flour_Peshawar,2020-05-01,1806.4
fao_flour_Peshawar,2020-06-01,2065.6
fao_flour_Peshawar,2020-07-01,2248.8
fao_flour_Peshawar,2020-08-01,2195.6
fao_flour_Peshawar,2020-09-01,2221.2
fao_flour_Peshawar,2020-10-01,2444.8
fao_flour_Peshawar,2020-11-01,2297.6
fao_flour_Peshawar,2020-12-01,2076.8
fao_flour_Peshawar,2021-01-01,2041.2
fao_flour_Peshawar,2021-02-01,2036.4
fao_flour_Peshawar,2021-03-01,1954.8
fao_flour_Peshawar,2021-04-01,1919.2
fao_flour_Peshawar,2021-05-01,2130.0
fao_flour_Peshawar,2021-06-01,2202.8
fao_flour_Peshawar,2021-07-01,2224.4
fao_flour_Peshawar,2021-08-01,2253.2
fao_flour_Peshawar,2021-09-01,2432.0
fao_flour_Peshawar,2021-10-01,2324.4
fao_flour_Peshawar,2021-11-01,2216.8
fao_flour_Peshawar,2021-12-01,2200.0
fao_flour_Peshawar,2022-01-01,2200.0
fao_flour_Peshawar,2022-02-01,2200.0
fao_flour_Peshawar,2022-03-01,2200.0
fao_flour_Peshawar,2022-04-01,1644.8
fao_flour_Peshawar,2022-05-01,2897.6
fao_flour_Peshawar,2022-06-01,2547.6
fao_flour_Peshawar,2022-07-01,2292.4
fao_flour_Peshawar,2022-08-01,1960.0
fao_flour_Peshawar,2022-09-01,2769.6
fao_flour_Peshawar,2022-10-01,3500.0
fao_flour_Peshawar,2022-11-01,3479.6
fao_flour_Peshawar,2022-12-01,4392.0
fao_flour_Peshawar,2023-01-01,5181.2
fao_flour_Peshawar,2023-02-01,3740.4
fao_flour_Peshawar,2023-03-01,4854.8
fao_flour_Peshawar,2023-04-01,5833.6
fao_flour_Peshawar,2023-05-01,5866.0
fao_flour_Peshawar,2023-06-01,5027.6
fao_flour_Peshawar,2023-07-01,5523.2
fao_flour_Peshawar,2023-08-01,5479.6
fao_flour_Peshawar,2023-09-01,5530.0
fao_flour_Peshawar,2023-10-01,5439.6
fao_flour_Peshawar,2023-11-01,5642.0
fao_flour_Peshawar,2023-12-01,5625.6
fao_flour_Peshawar,2024-01-01,5649.6
fao_flour_Peshawar,2024-02-01,5596.8
fao_flour_Peshawar,2024-03-01,5501.6
fao_flour_Peshawar,2024-04-01,4512.4
fao_flour_Peshawar,2024-05-01,3575.2
fao_flour_Peshawar,2024-06-01,3373.6
fao_flour_Peshawar,2024-07-01,3861.6
fao_flour_Peshawar,2024-08-01,3532.8
fao_flour_Peshawar,2024-09-01,3310.0
fao_flour_Peshawar,2024-10-01,3644.8
fao_flour_Peshawar,2024-11-01,3737.6
fao_flour_Peshawar,2024-12-01,3671.2
fao_flour_Peshawar,2025-01-01,3531.2
fao_flour_Peshawar,2025-02-01,3435.6
fao_flour_Peshawar,2025-03-01,3484.0
fao_flour_Peshawar,2025-04-01,3232.8
fao_flour_Peshawar,2025-05-01,3013.6
fao_flour_Peshawar,2025-06-01,3002.8
fao_flour_Peshawar,2025-07-01,2948.0
fao_flour_Peshawar,2025-08-01,3192.0
fao_flour_Peshawar,2025-09-01,4581.6
fao_flour_Peshawar,2025-10-01,5570.4
fao_flour_Peshawar,2025-11-01,5074.4
fao_flour_Peshawar,2025-12-01,5228.0
fao_flour_Peshawar,2026-01-01,5721.2
fao_flour_Peshawar,2026-02-01,5232.0
fao_flour_Peshawar,2026-03-01,5293.6
fao_flour_Peshawar,2026-04-01,4778.8
fao_flour_Peshawar,2026-05-01,5151.6
fao_flour_Peshawar,2026-06-01,5308.0
fao_flour_Peshawar,2026-07-01,5735.6
fao_flour_Peshawar,2026-08-01,5838.0
fao_flour_Peshawar,2026-09-01,5803.2
fao_flour_Peshawar,2026-10-01,6157.2
fao_flour_Multan,2006-01-01,504.0
fao_flour_Multan,2006-02-01,504.0
fao_flour_Multan,2006-03-01,508.0
fao_flour_Multan,2006-04-01,500.0
fao_flour_Multan,2006-05-01,488.0
fao_flour_Multan,2006-06-01,484.0
fao_flour_Multan,2006-07-01,488.0
fao_flour_Multan,2006-08-01,508.0
fao_flour_Multan,2006-09-01,508.0
fao_flour_Multan,2006-10-01,500.0
fao_flour_Multan,2006-11-01,516.0
fao_flour_Multan,2006-12-01,520.0
fao_flour_Multan,2007-01-01,560.0
fao_flour_Multan,2007-02-01,560.0
fao_flour_Multan,2007-03-01,560.0
fao_flour_Multan,2007-04-01,560.0
fao_flour_Multan,2007-05-01,510.0
fao_flour_Multan,2007-06-01,560.0
fao_flour_Multan,2007-07-01,600.0
fao_flour_Multan,2007-08-01,600.0
fao_flour_Multan,2007-09-01,620.0
fao_flour_Multan,2007-10-01,680.0
fao_flour_Multan,2007-11-01,720.0
fao_flour_Multan,2007-12-01,720.0
fao_flour_Multan,2008-01-01,768.0
fao_flour_Multan,2008-02-01,640.0
fao_flour_Multan,2008-03-01,665.2
fao_flour_Multan,2008-04-01,795.2
fao_flour_Multan,2008-05-01,800.0
fao_flour_Multan,2008-06-01,800.0
fao_flour_Multan,2008-07-01,856.0
fao_flour_Multan,2008-08-01,892.4
fao_flour_Multan,2008-09-01,808.0
fao_flour_Multan,2008-10-01,996.0
fao_flour_Multan,2008-11-01,1000.0
fao_flour_Multan,2008-12-01,960.0
fao_flour_Multan,2009-01-01,952.0
fao_flour_Multan,2009-02-01,960.0
fao_flour_Multan,2009-03-01,880.0
fao_flour_Multan,2009-04-01,994.0
fao_flour_Multan,2009-05-01,1065.2
fao_flour_Multan,2009-06-01,1050.0
fao_flour_Multan,2009-07-01,1094.0
fao_flour_Multan,2009-08-01,1130.0
fao_flour_Multan,2009-09-01,1120.0
fao_flour_Multan,2009-10-01,1142.0
fao_flour_Multan,2009-11-01,1140.0
fao_flour_Multan,2009-12-01,1148.0
fao_flour_Multan,2010-01-01,1190.0
fao_flour_Multan,2010-02-01,1166.8
fao_flour_Multan,2010-03-01,1160.0
fao_flour_Multan,2010-04-01,1100.0
fao_flour_Multan,2010-05-01,1055.2
fao_flour_Multan,2010-06-01,1055.2
fao_flour_Multan,2010-07-01,1040.0
fao_flour_Multan,2010-08-01,1035.2
fao_flour_Multan,2010-09-01,1151.6
fao_flour_Multan,2010-10-01,1200.0
fao_flour_Multan,2010-11-01,1200.0
fao_flour_Multan,2010-12-01,1200.0
fao_flour_Multan,2011-01-01,1200.0
fao_flour_Multan,2011-02-01,1192.4
fao_flour_Multan,2011-03-01,1190.0
fao_flour_Multan,2011-04-01,1160.0
fao_flour_Multan,2011-05-01,1095.2
fao_flour_Multan,2011-06-01,1084.0
fao_flour_Multan,2011-07-01,1100.0
fao_flour_Multan,2011-08-01,1100.0
fao_flour_Multan,2011-09-01,1120.0
fao_flour_Multan,2011-10-01,1135.2
fao_flour_Multan,2011-11-01,1170.0
fao_flour_Multan,2011-12-01,1200.0
fao_flour_Multan,2012-01-01,1200.0
fao_flour_Multan,2012-02-01,1180.0
fao_flour_Multan,2012-03-01,1196.0
fao_flour_Multan,2012-04-01,1215.2
fao_flour_Multan,2012-05-01,1172.0
fao_flour_Multan,2012-06-01,1180.0
fao_flour_Multan,2012-07-01,1164.8
fao_flour_Multan,2012-08-01,1220.0
fao_flour_Multan,2012-09-01,1284.8
fao_flour_Multan,2012-10-01,1300.0
fao_flour_Multan,2012-11-01,1312.0
fao_flour_Multan,2012-12-01,1330.0
fao_flour_Multan,2013-01-01,1448.0
fao_flour_Multan,2013-02-01,1435.2
fao_flour_Multan,2013-03-01,1425.2
fao_flour_Multan,2013-04-01,1360.0
fao_flour_Multan,2013-05-01,1360.0
fao_flour_Multan,2013-06-01,1412.4
fao_flour_Multan,2013-07-01,1450.0
fao_flour_Multan,2013-08-01,1516.0
fao_flour_Multan,2013-09-01,1575.2
fao_flour_Multan,2013-10-01,1600.0
fao_flour_Multan,2013-11-01,1644.8
fao_flour_Multan,2013-12-01,1670.0
fao_flour_Multan,2014-01-01,1655.2
fao_flour_Multan,2014-02-01,1675.2
fao_flour_Multan,2014-03-01,1720.0
fao_flour_Multan,2014-04-01,1586.0
fao_flour_Multan,2014-05-01,1430.0
fao_flour_Multan,2014-06-01,1420.0
fao_flour_Multan,2014-07-01,1476.0
fao_flour_Multan,2014-08-01,1560.0
fao_flour_Multan,2014-09-01,1570.0
fao_flour_Multan,2014-10-01,1556.0
fao_flour_Multan,2014-11-01,1544.8
fao_flour_Multan,2014-12-01,1540.0
fao_flour_Multan,2015-01-01,1528.0
fao_flour_Multan,2015-02-01,1520.0
fao_flour_Multan,2015-03-01,1495.2
fao_flour_Multan,2015-04-01,1464.0
fao_flour_Multan,2015-05-01,1435.2
fao_flour_Multan,2015-06-01,1410.0
fao_flour_Multan,2015-07-01,1460.0
fao_flour_Multan,2015-08-01,1455.2
fao_flour_Multan,2015-09-01,1465.2
fao_flour_Multan,2015-10-01,1540.0
fao_flour_Multan,2015-11-01,1560.0
fao_flour_Multan,2015-12-01,1536.0
fao_flour_Multan,2016-01-01,1540.0
fao_flour_Multan,2016-02-01,1540.0
fao_flour_Multan,2016-03-01,1528.0
fao_flour_Multan,2016-04-01,1455.2
fao_flour_Multan,2016-05-01,1400.0
fao_flour_Multan,2016-06-01,1424.0
fao_flour_Multan,2016-07-01,1449.2
fao_flour_Multan,2016-08-01,1446.8
fao_flour_Multan,2016-09-01,1465.2
fao_flour_Multan,2016-10-01,1495.2
fao_flour_Multan,2016-11-01,1510.0
fao_flour_Multan,2016-12-01,1520.0
fao_flour_Multan,2017-01-01,1520.0
fao_flour_Multan,2017-02-01,1520.0
fao_flour_Multan,2017-03-01,1516.0
fao_flour_Multan,2017-04-01,1440.0
fao_flour_Multan,2017-05-01,1400.0
fao_flour_Multan,2017-06-01,1368.0
fao_flour_Multan,2017-07-01,1345.2
fao_flour_Multan,2017-08-01,1340.0
fao_flour_Multan,2017-09-01,1370.0
fao_flour_Multan,2017-10-01,1420.0
fao_flour_Multan,2017-11-01,1476.0
fao_flour_Multan,2017-12-01,1480.0
fao_flour_Multan,2018-01-01,1480.0
fao_flour_Multan,2018-02-01,1465.2
fao_flour_Multan,2018-03-01,1456.0
fao_flour_Multan,2018-04-01,1400.0
fao_flour_Multan,2018-05-01,1270.0
fao_flour_Multan,2018-06-01,1210.0
fao_flour_Multan,2018-07-01,1480.0
fao_flour_Multan,2018-08-01,1480.0
fao_flour_Multan,2018-09-01,1480.0
fao_flour_Multan,2018-10-01,1525.2
fao_flour_Multan,2018-11-01,1500.0
fao_flour_Multan,2018-12-01,1500.0
fao_flour_Multan,2019-01-01,1512.0
fao_flour_Multan,2019-02-01,1520.0
fao_flour_Multan,2019-03-01,1518.4
fao_flour_Multan,2019-04-01,1491.6
fao_flour_Multan,2019-05-01,1496.0
fao_flour_Multan,2019-06-01,1608.8
fao_flour_Multan,2019-07-01,1624.0
fao_flour_Multan,2019-08-01,1619.2
fao_flour_Multan,2019-09-01,1617.2
fao_flour_Multan,2019-10-01,1617.2
fao_flour_Multan,2019-11-01,1617.2
fao_flour_Multan,2019-12-01,1617.2
fao_flour_Multan,2020-01-01,1684.0
fao_flour_Multan,2020-02-01,1610.0
fao_flour_Multan,2020-03-01,1610.0
fao_flour_Multan,2020-04-01,1610.0
fao_flour_Multan,2020-05-01,1610.0
fao_flour_Multan,2020-06-01,1977.6
fao_flour_Multan,2020-07-01,1826.4
fao_flour_Multan,2020-08-01,1720.0
fao_flour_Multan,2020-09-01,1720.0
fao_flour_Multan,2020-10-01,1720.0
fao_flour_Multan,2020-11-01,1720.0
fao_flour_Multan,2020-12-01,1910.8
fao_flour_Multan,2021-01-01,1720.0
fao_flour_Multan,2021-02-01,1774.4
fao_flour_Multan,2021-03-01,1720.0
fao_flour_Multan,2021-04-01,1979.2
fao_flour_Multan,2021-05-01,2128.8
fao_flour_Multan,2021-06-01,2134.8
fao_flour_Multan,2021-07-01,2147.6
fao_flour_Multan,2021-08-01,2142.8
fao_flour_Multan,2021-09-01,2294.4
fao_flour_Multan,2021-10-01,2200.0
fao_flour_Multan,2021-11-01,2200.0
fao_flour_Multan,2021-12-01,2200.0
fao_flour_Multan,2022-01-01,2200.0
fao_flour_Multan,2022-02-01,2200.0
fao_flour_Multan,2022-03-01,2200.0
fao_flour_Multan,2022-04-01,2004.8
fao_flour_Multan,2022-05-01,2316.8
fao_flour_Multan,2022-06-01,1960.0
fao_flour_Multan,2022-07-01,1960.0
fao_flour_Multan,2022-08-01,1960.0
fao_flour_Multan,2022-09-01,2275.2
fao_flour_Multan,2022-10-01,2590.0
fao_flour_Multan,2022-11-01,2590.0
fao_flour_Multan,2022-12-01,2590.0
fao_flour_Multan,2023-01-01,2956.0
fao_flour_Multan,2023-02-01,3176.4
fao_flour_Multan,2023-03-01,4416.0
fao_flour_Multan,2023-04-01,5214.8
fao_flour_Multan,2023-05-01,4849.2
fao_flour_Multan,2023-06-01,4864.8
fao_flour_Multan,2023-07-01,5462.4
fao_flour_Multan,2023-08-01,5410.8
fao_flour_Multan,2023-09-01,5410.0
fao_flour_Multan,2023-10-01,5326.0
fao_flour_Multan,2023-11-01,5598.8
fao_flour_Multan,2023-12-01,5531.6
fao_flour_Multan,2024-01-01,5451.6
fao_flour_Multan,2024-02-01,5446.8
fao_flour_Multan,2024-03-01,5324.8
fao_flour_Multan,2024-04-01,4533.6
fao_flour_Multan,2024-05-01,3454.0
fao_flour_Multan,2024-06-01,3214.0
fao_flour_Multan,2024-07-01,3646.0
fao_flour_Multan,2024-08-01,3350.4
fao_flour_Multan,2024-09-01,3198.0
fao_flour_Multan,2024-10-01,3332.8
fao_flour_Multan,2024-11-01,3346.8
fao_flour_Multan,2024-12-01,3318.4
fao_flour_Multan,2025-01-01,3335.2
fao_flour_Multan,2025-02-01,3354.4
fao_flour_Multan,2025-03-01,3312.0
fao_flour_Multan,2025-04-01,3075.6
fao_flour_Multan,2025-05-01,2966.0
fao_flour_Multan,2025-06-01,2994.8
fao_flour_Multan,2025-07-01,2955.6
fao_flour_Multan,2025-08-01,3176.0
fao_flour_Multan,2025-09-01,3805.2
fao_flour_Multan,2025-10-01,3867.2
fao_flour_Multan,2025-11-01,3842.0
fao_flour_Multan,2025-12-01,3826.4
fao_flour_Multan,2026-01-01,4032.4
fao_flour_Multan,2026-02-01,3952.8
fao_flour_Multan,2026-03-01,3739.2
fao_flour_Multan,2026-04-01,4045.2
fao_flour_Multan,2026-05-01,4379.6
fao_flour_Multan,2026-06-01,4548.0
fao_flour_Multan,2026-07-01,4793.2
fao_flour_Multan,2026-08-01,4716.8
fao_flour_Multan,2026-09-01,4716.8
fao_flour_Multan,2026-10-01,4826.0
fao_flour_Lahore,2006-01-01,528.0
fao_flour_Lahore,2006-02-01,528.0
fao_flour_Lahore,2006-03-01,524.0
fao_flour_Lahore,2006-04-01,528.0
fao_flour_Lahore,2006-05-01,516.0
fao_flour_Lahore,2006-06-01,516.0
fao_flour_Lahore,2006-07-01,524.0
fao_flour_Lahore,2006-08-01,532.0
fao_flour_Lahore,2006-09-01,532.0
fao_flour_Lahore,2006-10-01,520.0
fao_flour_Lahore,2006-11-01,536.0
fao_flour_Lahore,2006-12-01,544.0
fao_flour_Lahore,2007-01-01,640.0
fao_flour_Lahore,2007-02-01,640.0
fao_flour_Lahore,2007-03-01,640.0
fao_flour_Lahore,2007-04-01,640.0
fao_flour_Lahore,2007-05-01,540.0
fao_flour_Lahore,2007-06-01,640.0
fao_flour_Lahore,2007-07-01,680.0
fao_flour_Lahore,2007-08-01,680.0
fao_flour_Lahore,2007-09-01,624.0
fao_flour_Lahore,2007-10-01,780.0
fao_flour_Lahore,2007-11-01,820.0
fao_flour_Lahore,2007-12-01,680.0
fao_flour_Lahore,2008-01-01,728.0
fao_flour_Lahore,2008-02-01,648.0
fao_flour_Lahore,2008-03-01,652.0
fao_flour_Lahore,2008-04-01,828.0
fao_flour_Lahore,2008-05-01,828.0
fao_flour_Lahore,2008-06-01,828.0
fao_flour_Lahore,2008-07-01,884.0
fao_flour_Lahore,2008-08-01,892.0
fao_flour_Lahore,2008-09-01,828.0
fao_flour_Lahore,2008-10-01,924.0
fao_flour_Lahore,2008-11-01,932.0
fao_flour_Lahore,2008-12-01,930.0
fao_flour_Lahore,2009-01-01,892.0
fao_flour_Lahore,2009-02-01,880.0
fao_flour_Lahore,2009-03-01,880.0
fao_flour_Lahore,2009-04-01,984.0
fao_flour_Lahore,2009-05-01,1035.2
fao_flour_Lahore,2009-06-01,1030.0
fao_flour_Lahore,2009-07-01,1092.0
fao_flour_Lahore,2009-08-01,1135.2
fao_flour_Lahore,2009-09-01,1135.2
fao_flour_Lahore,2009-10-01,1135.2
fao_flour_Lahore,2009-11-01,1135.2
fao_flour_Lahore,2009-12-01,1135.2
fao_flour_Lahore,2010-01-01,1168.0
fao_flour_Lahore,2010-02-01,1172.0
fao_flour_Lahore,2010-03-01,1172.0
fao_flour_Lahore,2010-04-01,1152.4
fao_flour_Lahore,2010-05-01,1100.0
fao_flour_Lahore,2010-06-01,1100.0
fao_flour_Lahore,2010-07-01,1076.0
fao_flour_Lahore,2010-08-01,1031.2
fao_flour_Lahore,2010-09-01,1134.4
fao_flour_Lahore,2010-10-01,1184.8
fao_flour_Lahore,2010-11-01,1184.8
fao_flour_Lahore,2010-12-01,1184.8
fao_flour_Lahore,2011-01-01,1184.8
fao_flour_Lahore,2011-02-01,1184.8
fao_flour_Lahore,2011-03-01,1184.8
fao_flour_Lahore,2011-04-01,1180.4
fao_flour_Lahore,2011-05-01,1131.2
fao_flour_Lahore,2011-06-01,1094.8
fao_flour_Lahore,2011-07-01,1155.2
fao_flour_Lahore,2011-08-01,1120.0
fao_flour_Lahore,2011-09-01,1172.0
fao_flour_Lahore,2011-10-01,1172.0
fao_flour_Lahore,2011-11-01,1188.8
fao_flour_Lahore,2011-12-01,1200.0
fao_flour_Lahore,2012-01-01,1200.0
fao_flour_Lahore,2012-02-01,1200.0
fao_flour_Lahore,2012-03-01,1200.0
fao_flour_Lahore,2012-04-01,1200.0
fao_flour_Lahore,2012-05-01,1200.0
fao_flour_Lahore,2012-06-01,1200.0
fao_flour_Lahore,2012-07-01,1210.0
fao_flour_Lahore,2012-08-01,1258.4
fao_flour_Lahore,2012-09-01,1300.0
fao_flour_Lahore,2012-10-01,1300.0
fao_flour_Lahore,2012-11-01,1320.0
fao_flour_Lahore,2012-12-01,1324.8
fao_flour_Lahore,2013-01-01,1372.0
fao_flour_Lahore,2013-02-01,1380.0
fao_flour_Lahore,2013-03-01,1380.0
fao_flour_Lahore,2013-04-01,1380.0
fao_flour_Lahore,2013-05-01,1392.0
fao_flour_Lahore,2013-06-01,1459.2
fao_flour_Lahore,2013-07-01,1460.0
fao_flour_Lahore,2013-08-01,1541.6
fao_flour_Lahore,2013-09-01,1588.0
fao_flour_Lahore,2013-10-01,1588.0
fao_flour_Lahore,2013-11-01,1588.0
fao_flour_Lahore,2013-12-01,1588.0
fao_flour_Lahore,2014-01-01,1588.0
fao_flour_Lahore,2014-02-01,1588.0
fao_flour_Lahore,2014-03-01,1588.0
fao_flour_Lahore,2014-04-01,1578.4
fao_flour_Lahore,2014-05-01,1504.0
fao_flour_Lahore,2014-06-01,1504.0
fao_flour_Lahore,2014-07-01,1535.2
fao_flour_Lahore,2014-08-01,1595.2
fao_flour_Lahore,2014-09-01,1596.0
fao_flour_Lahore,2014-10-01,1596.0
fao_flour_Lahore,2014-11-01,1596.0
fao_flour_Lahore,2014-12-01,1596.0
fao_flour_Lahore,2015-01-01,1548.0
fao_flour_Lahore,2015-02-01,1540.0
fao_flour_Lahore,2015-03-01,1555.2
fao_flour_Lahore,2015-04-01,1548.0
fao_flour_Lahore,2015-05-01,1479.2
fao_flour_Lahore,2015-06-01,1462.0
fao_flour_Lahore,2015-07-01,1524.0
fao_flour_Lahore,2015-08-01,1479.2
fao_flour_Lahore,2015-09-01,1502.0
fao_flour_Lahore,2015-10-01,1552.8
fao_flour_Lahore,2015-11-01,1560.0
fao_flour_Lahore,2015-12-01,1564.0
fao_flour_Lahore,2016-01-01,1560.0
fao_flour_Lahore,2016-02-01,1560.0
fao_flour_Lahore,2016-03-01,1560.0
fao_flour_Lahore,2016-04-01,1544.0
fao_flour_Lahore,2016-05-01,1500.0
fao_flour_Lahore,2016-06-01,1492.8
fao_flour_Lahore,2016-07-01,1512.0
fao_flour_Lahore,2016-08-01,1518.0
fao_flour_Lahore,2016-09-01,1520.0
fao_flour_Lahore,2016-10-01,1520.0
fao_flour_Lahore,2016-11-01,1520.0
fao_flour_Lahore,2016-12-01,1520.0
fao_flour_Lahore,2017-01-01,1520.0
fao_flour_Lahore,2017-02-01,1520.0
fao_flour_Lahore,2017-03-01,1520.0
fao_flour_Lahore,2017-04-01,1506.0
fao_flour_Lahore,2017-05-01,1465.2
fao_flour_Lahore,2017-06-01,1460.0
fao_flour_Lahore,2017-07-01,1460.0
fao_flour_Lahore,2017-08-01,1428.0
fao_flour_Lahore,2017-09-01,1485.2
fao_flour_Lahore,2017-10-01,1520.0
fao_flour_Lahore,2017-11-01,1588.0
fao_flour_Lahore,2017-12-01,1600.0
fao_flour_Lahore,2018-01-01,1580.0
fao_flour_Lahore,2018-02-01,1560.0
fao_flour_Lahore,2018-03-01,1560.0
fao_flour_Lahore,2018-04-01,1520.0
fao_flour_Lahore,2018-05-01,1455.2
fao_flour_Lahore,2018-06-01,1400.0
fao_flour_Lahore,2018-07-01,1560.0
fao_flour_Lahore,2018-08-01,1520.0
fao_flour_Lahore,2018-09-01,1542.4
fao_flour_Lahore,2018-10-01,1570.0
fao_flour_Lahore,2018-11-01,1570.0
fao_flour_Lahore,2018-12-01,1570.0
fao_flour_Lahore,2019-01-01,1570.0
fao_flour_Lahore,2019-02-01,1570.0
fao_flour_Lahore,2019-03-01,1570.0
fao_flour_Lahore,2019-04-01,1570.0
fao_flour_Lahore,2019-05-01,1594.0
fao_flour_Lahore,2019-06-01,1613.2
fao_flour_Lahore,2019-07-01,1640.0
fao_flour_Lahore,2019-08-01,1670.0
fao_flour_Lahore,2019-09-01,1620.0
fao_flour_Lahore,2019-10-01,1620.0
fao_flour_Lahore,2019-11-01,1620.0
fao_flour_Lahore,2019-12-01,1620.0
fao_flour_Lahore,2020-01-01,1616.0
fao_flour_Lahore,2020-02-01,1610.0
fao_flour_Lahore,2020-03-01,1610.0
fao_flour_Lahore,2020-04-01,1610.0
fao_flour_Lahore,2020-05-01,1620.0
fao_flour_Lahore,2020-06-01,1990.0
fao_flour_Lahore,2020-07-01,1902.8
fao_flour_Lahore,2020-08-01,1720.0
fao_flour_Lahore,2020-09-01,1720.0
fao_flour_Lahore,2020-10-01,1720.0
fao_flour_Lahore,2020-11-01,1720.0
fao_flour_Lahore,2020-12-01,1720.0
fao_flour_Lahore,2021-01-01,1720.0
fao_flour_Lahore,2021-02-01,1720.0
fao_flour_Lahore,2021-03-01,1720.0
fao_flour_Lahore,2021-04-01,1800.4
fao_flour_Lahore,2021-05-01,2122.8
fao_flour_Lahore,2021-06-01,2133.2
fao_flour_Lahore,2021-07-01,2177.6
fao_flour_Lahore,2021-08-01,2200.4
fao_flour_Lahore,2021-09-01,2321.2
fao_flour_Lahore,2021-10-01,2200.0
fao_flour_Lahore,2021-11-01,2200.0
fao_flour_Lahore,2021-12-01,2200.0
fao_flour_Lahore,2022-01-01,2200.0
fao_flour_Lahore,2022-02-01,2200.0
fao_flour_Lahore,2022-03-01,2200.0
fao_flour_Lahore,2022-04-01,2050.0
fao_flour_Lahore,2022-05-01,2304.0
fao_flour_Lahore,2022-06-01,1960.0
fao_flour_Lahore,2022-07-01,1960.0
fao_flour_Lahore,2022-08-01,1960.0
fao_flour_Lahore,2022-09-01,2275.2
fao_flour_Lahore,2022-10-01,2590.0
fao_flour_Lahore,2022-11-01,2590.0
fao_flour_Lahore,2022-12-01,2590.0
fao_flour_Lahore,2023-01-01,2590.0
fao_flour_Lahore,2023-02-01,2590.0
fao_flour_Lahore,2023-03-01,3611.2
fao_flour_Lahore,2023-04-01,4790.0
fao_flour_Lahore,2023-05-01,5227.2
fao_flour_Lahore,2023-06-01,4965.6
fao_flour_Lahore,2023-07-01,5604.0
fao_flour_Lahore,2023-08-01,5538.4
fao_flour_Lahore,2023-09-01,5524.8
fao_flour_Lahore,2023-10-01,5430.4
fao_flour_Lahore,2023-11-01,5612.0
fao_flour_Lahore,2023-12-01,5596.0
fao_flour_Lahore,2024-01-01,5596.0
fao_flour_Lahore,2024-02-01,5596.0
fao_flour_Lahore,2024-03-01,5547.2
fao_flour_Lahore,2024-04-01,4446.0
fao_flour_Lahore,2024-05-01,3541.6
fao_flour_Lahore,2024-06-01,3382.0
fao_flour_Lahore,2024-07-01,3646.8
fao_flour_Lahore,2024-08-01,3490.0
fao_flour_Lahore,2024-09-01,3440.0
fao_flour_Lahore,2024-10-01,3432.0
fao_flour_Lahore,2024-11-01,3460.0
fao_flour_Lahore,2024-12-01,3460.0
fao_flour_Lahore,2025-01-01,3431.2
fao_flour_Lahore,2025-02-01,3404.8
fao_flour_Lahore,2025-03-01,3420.0
fao_flour_Lahore,2025-04-01,3038.8
fao_flour_Lahore,2025-05-01,2808.8
fao_flour_Lahore,2025-06-01,2844.0
fao_flour_Lahore,2025-07-01,2760.4
fao_flour_Lahore,2025-08-01,3011.2
fao_flour_Lahore,2025-09-01,3770.4
fao_flour_Lahore,2025-10-01,3620.0
fao_flour_Lahore,2025-11-01,3620.0
fao_flour_Lahore,2025-12-01,3620.0
fao_flour_Lahore,2026-01-01,3660.0
fao_flour_Lahore,2026-02-01,3620.0
fao_flour_Lahore,2026-03-01,3620.0
fao_flour_Lahore,2026-04-01,3620.0
fao_flour_Lahore,2026-05-01,4248.0
fao_flour_Lahore,2026-06-01,4414.0
fao_flour_Lahore,2026-07-01,4516.4
fao_flour_Lahore,2026-08-01,4535.6
fao_flour_Lahore,2026-09-01,4437.6
fao_flour_Lahore,2026-10-01,4696.8
fao_flour_Karachi,2006-01-01,552.0
fao_flour_Karachi,2006-02-01,560.0
fao_flour_Karachi,2006-03-01,556.0
fao_flour_Karachi,2006-04-01,552.0
fao_flour_Karachi,2006-05-01,548.0
fao_flour_Karachi,2006-06-01,548.0
fao_flour_Karachi,2006-07-01,556.0
fao_flour_Karachi,2006-08-01,576.0
fao_flour_Karachi,2006-09-01,588.0
fao_flour_Karachi,2006-10-01,552.0
fao_flour_Karachi,2006-11-01,592.0
fao_flour_Karachi,2006-12-01,600.0
fao_flour_Karachi,2007-01-01,710.8
fao_flour_Karachi,2007-02-01,710.8
fao_flour_Karachi,2007-03-01,710.8
fao_flour_Karachi,2007-04-01,710.4
fao_flour_Karachi,2007-05-01,614.0
fao_flour_Karachi,2007-06-01,720.0
fao_flour_Karachi,2007-07-01,780.0
fao_flour_Karachi,2007-08-01,780.0
fao_flour_Karachi,2007-09-01,714.8
fao_flour_Karachi,2007-10-01,840.0
fao_flour_Karachi,2007-11-01,860.0
fao_flour_Karachi,2007-12-01,832.8
fao_flour_Karachi,2008-01-01,808.0
fao_flour_Karachi,2008-02-01,664.0
fao_flour_Karachi,2008-03-01,700.0
fao_flour_Karachi,2008-04-01,943.2
fao_flour_Karachi,2008-05-01,1160.0
fao_flour_Karachi,2008-06-01,1109.2
fao_flour_Karachi,2008-07-01,1168.0
fao_flour_Karachi,2008-08-01,1180.0
fao_flour_Karachi,2008-09-01,1208.0
fao_flour_Karachi,2008-10-01,1372.0
fao_flour_Karachi,2008-11-01,1296.0
fao_flour_Karachi,2008-12-01,1210.8
fao_flour_Karachi,2009-01-01,1036.0
fao_flour_Karachi,2009-02-01,1018.0
fao_flour_Karachi,2009-03-01,1052.0
fao_flour_Karachi,2009-04-01,1167.6
fao_flour_Karachi,2009-05-01,1168.4
fao_flour_Karachi,2009-06-01,1194.0
fao_flour_Karachi,2009-07-01,1213.6
fao_flour_Karachi,2009-08-01,1240.0
fao_flour_Karachi,2009-09-01,1240.0
fao_flour_Karachi,2009-10-01,1240.0
fao_flour_Karachi,2009-11-01,1250.0
fao_flour_Karachi,2009-12-01,1260.0
fao_flour_Karachi,2010-01-01,1280.0
fao_flour_Karachi,2010-02-01,1280.0
fao_flour_Karachi,2010-03-01,1280.0
fao_flour_Karachi,2010-04-01,1230.8
fao_flour_Karachi,2010-05-01,1177.2
fao_flour_Karachi,2010-06-01,1195.2
fao_flour_Karachi,2010-07-01,1195.2
fao_flour_Karachi,2010-08-01,1196.4
fao_flour_Karachi,2010-09-01,1257.2
fao_flour_Karachi,2010-10-01,1280.0
fao_flour_Karachi,2010-11-01,1280.0
fao_flour_Karachi,2010-12-01,1280.0
fao_flour_Karachi,2011-01-01,1287.2
fao_flour_Karachi,2011-02-01,1294.0
fao_flour_Karachi,2011-03-01,1324.8
fao_flour_Karachi,2011-04-01,1238.4
fao_flour_Karachi,2011-05-01,1201.6
fao_flour_Karachi,2011-06-01,1188.8
fao_flour_Karachi,2011-07-01,1304.8
fao_flour_Karachi,2011-08-01,1267.2
fao_flour_Karachi,2011-09-01,1258.4
fao_flour_Karachi,2011-10-01,1284.8
fao_flour_Karachi,2011-11-01,1320.0
fao_flour_Karachi,2011-12-01,1336.0
fao_flour_Karachi,2012-01-01,1360.0
fao_flour_Karachi,2012-02-01,1360.0
fao_flour_Karachi,2012-03-01,1384.0
fao_flour_Karachi,2012-04-01,1400.0
fao_flour_Karachi,2012-05-01,1400.0
fao_flour_Karachi,2012-06-01,1400.0
fao_flour_Karachi,2012-07-01,1410.0
fao_flour_Karachi,2012-08-01,1448.0
fao_flour_Karachi,2012-09-01,1480.0
fao_flour_Karachi,2012-10-01,1480.0
fao_flour_Karachi,2012-11-01,1517.2
fao_flour_Karachi,2012-12-01,1544.8
fao_flour_Karachi,2013-01-01,1628.0
fao_flour_Karachi,2013-02-01,1650.0
fao_flour_Karachi,2013-03-01,1680.0
fao_flour_Karachi,2013-04-01,1535.6
fao_flour_Karachi,2013-05-01,1639.2
fao_flour_Karachi,2013-06-01,1680.0
fao_flour_Karachi,2013-07-01,1750.0
fao_flour_Karachi,2013-08-01,1820.0
fao_flour_Karachi,2013-09-01,1810.0
fao_flour_Karachi,2013-10-01,1840.0
fao_flour_Karachi,2013-11-01,1840.0
fao_flour_Karachi,2013-12-01,1870.0
fao_flour_Karachi,2014-01-01,1920.0
fao_flour_Karachi,2014-02-01,1920.0
fao_flour_Karachi,2014-03-01,1899.2
fao_flour_Karachi,2014-04-01,1772.4
fao_flour_Karachi,2014-05-01,1772.4
fao_flour_Karachi,2014-06-01,1712.4
fao_flour_Karachi,2014-07-01,1755.6
fao_flour_Karachi,2014-08-01,1778.4
fao_flour_Karachi,2014-09-01,1772.4
fao_flour_Karachi,2014-10-01,1724.4
fao_flour_Karachi,2014-11-01,1722.4
fao_flour_Karachi,2014-12-01,1723.2
fao_flour_Karachi,2015-01-01,1684.8
fao_flour_Karachi,2015-02-01,1662.4
fao_flour_Karachi,2015-03-01,1652.4
fao_flour_Karachi,2015-04-01,1628.4
fao_flour_Karachi,2015-05-01,1612.4
fao_flour_Karachi,2015-06-01,1612.4
fao_flour_Karachi,2015-07-01,1620.4
fao_flour_Karachi,2015-08-01,1652.4
fao_flour_Karachi,2015-09-01,1652.4
fao_flour_Karachi,2015-10-01,1716.4
fao_flour_Karachi,2015-11-01,1748.4
fao_flour_Karachi,2015-12-01,1772.8
fao_flour_Karachi,2016-01-01,1774.0
fao_flour_Karachi,2016-02-01,1737.2
fao_flour_Karachi,2016-03-01,1654.4
fao_flour_Karachi,2016-04-01,1624.4
fao_flour_Karachi,2016-05-01,1646.0
fao_flour_Karachi,2016-06-01,1646.0
fao_flour_Karachi,2016-07-01,1674.0
fao_flour_Karachi,2016-08-01,1688.8
fao_flour_Karachi,2016-09-01,1673.6
fao_flour_Karachi,2016-10-01,1750.0
fao_flour_Karachi,2016-11-01,1770.8
fao_flour_Karachi,2016-12-01,1778.4
fao_flour_Karachi,2017-01-01,1778.4
fao_flour_Karachi,2017-02-01,1786.0
fao_flour_Karachi,2017-03-01,1793.2
fao_flour_Karachi,2017-04-01,1736.8
fao_flour_Karachi,2017-05-01,1689.6
fao_flour_Karachi,2017-06-01,1655.2
fao_flour_Karachi,2017-07-01,1639.6
fao_flour_Karachi,2017-08-01,1628.8
fao_flour_Karachi,2017-09-01,1630.4
fao_flour_Karachi,2017-10-01,1700.4
fao_flour_Karachi,2017-11-01,1742.4
fao_flour_Karachi,2017-12-01,1757.6
fao_flour_Karachi,2018-01-01,1771.2
fao_flour_Karachi,2018-02-01,1768.4
fao_flour_Karachi,2018-03-01,1750.0
fao_flour_Karachi,2018-04-01,1693.2
fao_flour_Karachi,2018-05-01,1663.6
fao_flour_Karachi,2018-06-01,1673.6
fao_flour_Karachi,2018-07-01,1722.8
fao_flour_Karachi,2018-08-01,1742.8
fao_flour_Karachi,2018-09-01,1755.6
fao_flour_Karachi,2018-10-01,1765.2
fao_flour_Karachi,2018-11-01,1773.2
fao_flour_Karachi,2018-12-01,1789.2
fao_flour_Karachi,2019-01-01,1803.2
fao_flour_Karachi,2019-02-01,1804.8
fao_flour_Karachi,2019-03-01,1800.0
fao_flour_Karachi,2019-04-01,1703.2
fao_flour_Karachi,2019-05-01,1756.8
fao_flour_Karachi,2019-06-01,1846.8
fao_flour_Karachi,2019-07-01,1900.4
fao_flour_Karachi,2019-08-01,1926.4
fao_flour_Karachi,2019-09-01,2035.6
fao_flour_Karachi,2019-10-01,2101.6
fao_flour_Karachi,2019-11-01,2206.0
fao_flour_Karachi,2019-12-01,2176.8
fao_flour_Karachi,2020-01-01,2395.2
fao_flour_Karachi,2020-02-01,2230.4
fao_flour_Karachi,2020-03-01,2166.0
fao_flour_Karachi,2020-04-01,2153.2
fao_flour_Karachi,2020-05-01,2152.4
fao_flour_Karachi,2020-06-01,2235.2
fao_flour_Karachi,2020-07-01,2407.2
fao_flour_Karachi,2020-08-01,2705.6
fao_flour_Karachi,2020-09-01,2682.4
fao_flour_Karachi,2020-10-01,2804.0
fao_flour_Karachi,2020-11-01,2423.6
fao_flour_Karachi,2020-12-01,2419.6
fao_flour_Karachi,2021-01-01,2476.4
fao_flour_Karachi,2021-02-01,2560.0
fao_flour_Karachi,2021-03-01,2560.8
fao_flour_Karachi,2021-04-01,2441.2
fao_flour_Karachi,2021-05-01,2578.4
fao_flour_Karachi,2021-06-01,2496.0
fao_flour_Karachi,2021-07-01,2473.6
fao_flour_Karachi,2021-08-01,2511.2
fao_flour_Karachi,2021-09-01,2725.6
fao_flour_Karachi,2021-10-01,2936.4
fao_flour_Karachi,2021-11-01,2905.2
fao_flour_Karachi,2021-12-01,2667.2
fao_flour_Karachi,2022-01-01,2306.0
fao_flour_Karachi,2022-02-01,2223.2
fao_flour_Karachi,2022-03-01,2434.0
fao_flour_Karachi,2022-04-01,2868.8
fao_flour_Karachi,2022-05-01,3166.8
fao_flour_Karachi,2022-06-01,3511.2
fao_flour_Karachi,2022-07-01,3770.8
fao_flour_Karachi,2022-08-01,3854.8
fao_flour_Karachi,2022-09-01,4270.4
fao_flour_Karachi,2022-10-01,3806.4
fao_flour_Karachi,2022-11-01,3832.4
fao_flour_Karachi,2022-12-01,4205.2
fao_flour_Karachi,2023-01-01,4936.0
fao_flour_Karachi,2023-02-01,4841.6
fao_flour_Karachi,2023-03-01,5706.8
fao_flour_Karachi,2023-04-01,5560.0
fao_flour_Karachi,2023-05-01,5692.8
fao_flour_Karachi,2023-06-01,5490.8
fao_flour_Karachi,2023-07-01,6162.8
fao_flour_Karachi,2023-08-01,6194.0
fao_flour_Karachi,2023-09-01,6037.6
fao_flour_Karachi,2023-10-01,5727.2
fao_flour_Karachi,2023-11-01,5341.6
fao_flour_Karachi,2023-12-01,5466.8
fao_flour_Karachi,2024-01-01,5556.8
fao_flour_Karachi,2024-02-01,5372.8
fao_flour_Karachi,2024-03-01,5193.6
fao_flour_Karachi,2024-04-01,4597.2
fao_flour_Karachi,2024-05-01,4071.6
fao_flour_Karachi,2024-06-01,3886.4
fao_flour_Karachi,2024-07-01,4022.8
fao_flour_Karachi,2024-08-01,3974.4
fao_flour_Karachi,2024-09-01,3758.0
fao_flour_Karachi,2024-10-01,3950.4
fao_flour_Karachi,2024-11-01,4036.0
fao_flour_Karachi,2024-12-01,3916.4
fao_flour_Karachi,2025-01-01,3975.2
fao_flour_Karachi,2025-02-01,3810.8
fao_flour_Karachi,2025-03-01,3732.8
fao_flour_Karachi,2025-04-01,3276.0
fao_flour_Karachi,2025-05-01,3140.4
fao_flour_Karachi,2025-06-01,3168.8
fao_flour_Karachi,2025-07-01,3282.4
fao_flour_Karachi,2025-08-01,3389.6
fao_flour_Karachi,2025-09-01,4442.4
fao_flour_Karachi,2025-10-01,4786.0
fao_flour_Karachi,2025-11-01,4712.8
fao_flour_Karachi,2025-12-01,4795.6
fao_flour_Karachi,2026-01-01,5034.8
fao_flour_Karachi,2026-02-01,5022.0
fao_flour_Karachi,2026-03-01,4664.8
fao_flour_Karachi,2026-04-01,4470.0
fao_flour_Karachi,2026-05-01,5029.2
fao_flour_Karachi,2026-06-01,5290.0
fao_flour_Karachi,2026-07-01,5510.4
fao_flour_Karachi,2026-08-01,5835.2
fao_flour_Karachi,2026-09-01,6058.4
fao_flour_Karachi,2026-10-01,6726.4
pbs_whsl_Lahore,2006-07-01,450.0
pbs_whsl_Lahore,2006-08-01,450.0
pbs_whsl_Lahore,2006-09-01,450.0
pbs_whsl_Lahore,2006-10-01,460.0
pbs_whsl_Lahore,2006-11-01,460.0
pbs_whsl_Lahore,2006-12-01,460.0
pbs_whsl_Lahore,2007-01-01,460.0
pbs_whsl_Lahore,2007-02-01,460.0
pbs_whsl_Lahore,2007-03-01,460.0
pbs_whsl_Lahore,2007-04-01,460.0
pbs_whsl_Lahore,2007-05-01,470.0
pbs_whsl_Lahore,2007-06-01,470.0
pbs_whsl_Lahore,2007-07-01,486.0
pbs_whsl_Lahore,2007-08-01,490.0
pbs_whsl_Lahore,2007-09-01,550.0
pbs_whsl_Lahore,2007-10-01,568.0
pbs_whsl_Lahore,2007-11-01,606.0
pbs_whsl_Lahore,2007-12-01,682.0
pbs_whsl_Lahore,2008-01-01,736.0
pbs_whsl_Lahore,2008-02-01,608.0
pbs_whsl_Lahore,2008-03-01,642.0
pbs_whsl_Lahore,2008-04-01,703.0
pbs_whsl_Lahore,2008-05-01,721.0
pbs_whsl_Lahore,2008-06-01,736.0
pbs_whsl_Lahore,2008-07-01,766.0
pbs_whsl_Lahore,2008-08-01,801.0
pbs_whsl_Lahore,2008-09-01,800.0
pbs_whsl_Lahore,2008-10-01,871.0
pbs_whsl_Lahore,2008-11-01,867.25
pbs_whsl_Lahore,2008-12-01,889.0
pbs_whsl_Lahore,2009-01-01,951.67
pbs_whsl_Lahore,2009-02-01,940.5
pbs_whsl_Lahore,2009-03-01,917.75
pbs_whsl_Lahore,2009-04-01,959.0
pbs_whsl_Lahore,2009-05-01,950.0
pbs_whsl_Lahore,2009-06-01,933.0
pbs_whsl_Lahore,2009-07-01,986.0
pbs_whsl_Lahore,2009-08-01,997.0
pbs_whsl_Lahore,2009-09-01,1003.0
pbs_whsl_Lahore,2009-10-01,1026.0
pbs_whsl_Lahore,2009-11-01,1034.0
pbs_whsl_Lahore,2009-12-01,1059.0
pbs_whsl_Lahore,2010-01-01,1068.0
pbs_whsl_Lahore,2010-02-01,1077.0
pbs_whsl_Lahore,2010-03-01,1055.0
pbs_whsl_Lahore,2010-04-01,971.0
pbs_whsl_Lahore,2010-05-01,967.0
pbs_whsl_Lahore,2010-06-01,956.0
pbs_whsl_Lahore,2010-07-01,935.0
pbs_whsl_Lahore,2010-08-01,947.0
pbs_whsl_Lahore,2010-09-01,945.0
pbs_whsl_Lahore,2010-10-01,954.0
pbs_whsl_Lahore,2010-11-01,1000.0
pbs_whsl_Lahore,2010-12-01,1000.0
pbs_whsl_Lahore,2011-01-01,1000.0
pbs_whsl_Lahore,2011-02-01,1007.0
pbs_whsl_Lahore,2011-03-01,1000.0
pbs_whsl_Lahore,2011-04-01,992.0
pbs_whsl_Lahore,2011-05-01,861.0
pbs_whsl_Lahore,2011-06-01,909.0
pbs_whsl_Lahore,2011-07-01,949.0
pbs_whsl_Lahore,2011-08-01,950.0
pbs_whsl_Lahore,2011-09-01,950.0
pbs_whsl_Lahore,2011-10-01,950.0
pbs_whsl_Lahore,2011-11-01,950.0
pbs_whsl_Lahore,2011-12-01,954.0
pbs_whsl_Lahore,2012-01-01,1006.0
pbs_whsl_Lahore,2012-02-01,1050.0
pbs_whsl_Lahore,2012-03-01,1049.0
pbs_whsl_Lahore,2012-04-01,1050.0
pbs_whsl_Lahore,2012-05-01,1039.0
pbs_whsl_Lahore,2012-06-01,1035.0
pbs_whsl_Lahore,2012-07-01,1035.0
pbs_whsl_Lahore,2012-08-01,1055.0
pbs_whsl_Lahore,2012-09-01,1056.67
pbs_whsl_Lahore,2012-10-01,1102.5
pbs_whsl_Lahore,2012-11-01,1120.0
pbs_whsl_Lahore,2012-12-01,1120.0
pbs_whsl_Lahore,2013-01-01,1225.0
pbs_whsl_Lahore,2013-02-01,1235.0
pbs_whsl_Lahore,2013-03-01,1242.5
pbs_whsl_Lahore,2013-04-01,1267.5
pbs_whsl_Lahore,2013-05-01,1200.0
pbs_whsl_Lahore,2013-06-01,1255.0
pbs_whsl_Lahore,2013-07-01,1305.2
pbs_whsl_Lahore,2013-08-01,1325.2
pbs_whsl_Lahore,2013-09-01,1445.2
pbs_whsl_Lahore,2013-10-01,1435.2
pbs_whsl_Lahore,2013-11-01,1505.2
pbs_whsl_Lahore,2013-12-01,1595.2
pbs_whsl_Lahore,2014-01-01,1595.2
pbs_whsl_Lahore,2014-02-01,1625.2
pbs_whsl_Lahore,2014-03-01,1652.4
pbs_whsl_Lahore,2014-04-01,1652.4
pbs_whsl_Lahore,2014-05-01,1295.2
pbs_whsl_Lahore,2014-06-01,1295.2
pbs_whsl_Lahore,2014-07-01,1295.0
pbs_whsl_Lahore,2014-08-01,1375.0
pbs_whsl_Lahore,2014-09-01,1375.0
pbs_whsl_Lahore,2014-10-01,1375.0
pbs_whsl_Lahore,2014-11-01,1390.0
pbs_whsl_Lahore,2014-12-01,1390.0
pbs_whsl_Lahore,2015-01-01,1350.0
pbs_whsl_Lahore,2015-02-01,1390.0
pbs_whsl_Lahore,2015-03-01,1390.0
pbs_whsl_Lahore,2015-04-01,1390.0
pbs_whsl_Lahore,2015-05-01,1290.0
pbs_whsl_Lahore,2015-06-01,1270.0
pbs_whsl_Lahore,2015-07-01,1335.0
pbs_whsl_Lahore,2015-08-01,1345.0
pbs_whsl_Lahore,2015-09-01,1345.0
pbs_whsl_Lahore,2015-10-01,1375.0
pbs_whsl_Lahore,2015-11-01,1385.0
pbs_whsl_Lahore,2015-12-01,1405.0
pbs_whsl_Lahore,2016-01-01,1405.0
pbs_whsl_Lahore,2016-02-01,1425.0
pbs_whsl_Lahore,2016-03-01,1425.0
pbs_whsl_Lahore,2016-04-01,1425.0
pbs_whsl_Lahore,2016-05-01,1425.0
pbs_whsl_Lahore,2016-06-01,1295.0
pbs_whsl_Lahore,2016-07-01,1220.0
pbs_whsl_Lahore,2016-08-01,1258.0
pbs_whsl_Lahore,2016-09-01,1262.0
pbs_whsl_Lahore,2016-10-01,1290.0
pbs_whsl_Lahore,2016-11-01,1296.0
pbs_whsl_Lahore,2016-12-01,1302.0
pbs_whsl_Lahore,2017-01-01,1316.0
pbs_whsl_Lahore,2017-02-01,1320.0
pbs_whsl_Lahore,2017-03-01,1308.0
pbs_whsl_Lahore,2017-04-01,1311.0
pbs_whsl_Lahore,2017-05-01,1222.0
pbs_whsl_Lahore,2017-06-01,1234.0
pbs_whsl_Lahore,2017-07-01,1257.2
pbs_whsl_Lahore,2017-08-01,1210.0
pbs_whsl_Lahore,2017-09-01,1210.0
pbs_whsl_Lahore,2017-10-01,1210.0
pbs_whsl_Lahore,2017-11-01,1312.4
pbs_whsl_Lahore,2017-12-01,1362.6
pbs_whsl_Lahore,2018-01-01,1362.6
pbs_whsl_Lahore,2018-02-01,1362.6
pbs_whsl_Lahore,2018-03-01,1362.6
pbs_whsl_Lahore,2018-04-01,1362.6
pbs_whsl_Lahore,2018-05-01,1220.0
pbs_whsl_Lahore,2018-06-01,1220.0
pbs_whsl_Lahore,2018-07-01,1260.0
pbs_whsl_Lahore,2018-08-01,1290.0
pbs_whsl_Lahore,2018-09-01,1290.0
pbs_whsl_Lahore,2018-10-01,1330.0
pbs_whsl_Lahore,2018-11-01,1350.0
pbs_whsl_Lahore,2018-12-01,1350.0
pbs_whsl_Lahore,2019-01-01,1350.0
pbs_whsl_Lahore,2019-02-01,1350.0
pbs_whsl_Lahore,2019-03-01,1350.0
pbs_whsl_Lahore,2019-04-01,1355.0
pbs_whsl_Lahore,2019-05-01,1355.0
pbs_whsl_Lahore,2019-06-01,1355.0
pbs_whsl_Lahore,2019-07-01,1353.32
pbs_whsl_Lahore,2019-08-01,1373.2
pbs_whsl_Lahore,2019-09-01,1373.2
pbs_whsl_Lahore,2019-10-01,1373.2
pbs_whsl_Lahore,2019-11-01,1493.22
pbs_whsl_Lahore,2019-12-01,1693.23
pbs_whsl_Lahore,2020-01-01,1719.77
pbs_whsl_Lahore,2020-02-01,1626.24
pbs_whsl_Lahore,2020-03-01,1626.24
pbs_whsl_Lahore,2020-04-01,1680.0
pbs_whsl_Lahore,2020-05-01,1446.66
pbs_whsl_Lahore,2020-06-01,1800.0
pbs_whsl_Lahore,2020-07-01,2000.0
pbs_whsl_Lahore,2020-08-01,2140.0
pbs_whsl_Lahore,2020-09-01,2166.65
pbs_whsl_Lahore,2020-10-01,2400.0
pbs_whsl_Lahore,2020-11-01,2350.0
pbs_whsl_Lahore,2020-12-01,2150.0
pbs_whsl_Lahore,2021-01-01,2350.0
pbs_whsl_Lahore,2021-02-01,2400.0
pbs_whsl_Lahore,2021-03-01,2400.0
pbs_whsl_Lahore,2021-04-01,2100.0
pbs_whsl_Lahore,2021-05-01,2050.0
pbs_whsl_Lahore,2021-06-01,2080.0
pbs_whsl_Lahore,2021-07-01,2040.0
pbs_whsl_Lahore,2021-08-01,2080.0
pbs_whsl_Lahore,2021-09-01,2200.0
pbs_whsl_Lahore,2021-10-01,2380.0
pbs_whsl_Lahore,2021-11-01,2360.0
pbs_whsl_Lahore,2021-12-01,2360.0
pbs_whsl_Lahore,2022-01-01,2500.0
pbs_whsl_Lahore,2022-02-01,2460.0
pbs_whsl_Lahore,2022-03-01,2440.0
pbs_whsl_Lahore,2022-04-01,2480.0
pbs_whsl_Lahore,2022-05-01,2480.0
pbs_whsl_Lahore,2022-06-01,2480.0
pbs_whsl_Lahore,2022-07-01,3060.0
pbs_whsl_Lahore,2022-08-01,2980.0
pbs_whsl_Lahore,2022-09-01,3480.0
pbs_whsl_Lahore,2022-10-01,3560.0
pbs_whsl_Lahore,2022-11-01,3560.0
pbs_whsl_Lahore,2022-12-01,3950.0
pbs_whsl_Lahore,2023-01-01,5170.0
pbs_whsl_Lahore,2023-02-01,4520.0
pbs_whsl_Lahore,2023-03-01,4620.0
pbs_whsl_Lahore,2023-04-01,4470.0
pbs_whsl_Lahore,2023-05-01,4770.0
pbs_whsl_Lahore,2023-06-01,4420.0
pbs_whsl_Lahore,2023-07-01,4900.0
pbs_whsl_Lahore,2023-08-01,4860.0
pbs_whsl_Lahore,2023-09-01,4860.0
pbs_whsl_Lahore,2023-10-01,4860.0
pbs_whsl_Lahore,2023-11-01,4950.0
pbs_whsl_Lahore,2023-12-01,5170.0
pbs_whsl_Lahore,2024-01-01,5170.0
pbs_whsl_Lahore,2024-02-01,5170.0
pbs_whsl_Lahore,2024-03-01,5070.0
pbs_whsl_Lahore,2024-04-01,5020.0
pbs_whsl_Lahore,2024-05-01,3300.0
pbs_whsl_Lahore,2024-06-01,3050.0
pbs_whsl_Lahore,2024-07-01,3200.0
pbs_whsl_Lahore,2024-08-01,3000.0
pbs_whsl_Lahore,2024-09-01,2900.0
pbs_whsl_Lahore,2024-10-01,3150.0
pbs_whsl_Lahore,2024-11-01,3300.0
pbs_whsl_Lahore,2024-12-01,3300.0
pbs_whsl_Lahore,2025-01-01,3150.0
pbs_whsl_Lahore,2025-02-01,3150.0
pbs_whsl_Lahore,2025-03-01,3150.0
pbs_whsl_Lahore,2025-04-01,2550.0
pbs_whsl_Lahore,2025-05-01,2600.0
pbs_whsl_Rawalpindi,2006-07-01,490.0
pbs_whsl_Rawalpindi,2006-08-01,490.0
pbs_whsl_Rawalpindi,2006-09-01,490.0
pbs_whsl_Rawalpindi,2006-10-01,490.0
pbs_whsl_Rawalpindi,2006-11-01,490.0
pbs_whsl_Rawalpindi,2006-12-01,490.0
pbs_whsl_Rawalpindi,2007-01-01,490.0
pbs_whsl_Rawalpindi,2007-02-01,490.0
pbs_whsl_Rawalpindi,2007-03-01,480.0
pbs_whsl_Rawalpindi,2007-04-01,480.0
pbs_whsl_Rawalpindi,2007-05-01,480.0
pbs_whsl_Rawalpindi,2007-06-01,480.0
pbs_whsl_Rawalpindi,2007-07-01,483.0
pbs_whsl_Rawalpindi,2007-08-01,493.0
pbs_whsl_Rawalpindi,2007-09-01,531.0
pbs_whsl_Rawalpindi,2007-10-01,544.0
pbs_whsl_Rawalpindi,2007-11-01,605.0
pbs_whsl_Rawalpindi,2007-12-01,700.0
pbs_whsl_Rawalpindi,2008-01-01,754.0
pbs_whsl_Rawalpindi,2008-02-01,659.0
pbs_whsl_Rawalpindi,2008-03-01,680.0
pbs_whsl_Rawalpindi,2008-04-01,764.0
pbs_whsl_Rawalpindi,2008-05-01,821.0
pbs_whsl_Rawalpindi,2008-06-01,834.0
pbs_whsl_Rawalpindi,2008-07-01,809.0
pbs_whsl_Rawalpindi,2008-08-01,823.0
pbs_whsl_Rawalpindi,2008-09-01,815.0
pbs_whsl_Rawalpindi,2008-10-01,863.0
pbs_whsl_Rawalpindi,2008-11-01,922.5
pbs_whsl_Rawalpindi,2008-12-01,937.0
pbs_whsl_Rawalpindi,2009-01-01,946.0
pbs_whsl_Rawalpindi,2009-02-01,966.25
pbs_whsl_Rawalpindi,2009-03-01,970.0
pbs_whsl_Rawalpindi,2009-04-01,990.0
pbs_whsl_Rawalpindi,2009-05-01,1000.0
pbs_whsl_Rawalpindi,2009-06-01,1000.0
pbs_whsl_Rawalpindi,2009-07-01,996.0
pbs_whsl_Rawalpindi,2009-08-01,1017.0
pbs_whsl_Rawalpindi,2009-09-01,1034.0
pbs_whsl_Rawalpindi,2009-10-01,1050.0
pbs_whsl_Rawalpindi,2009-11-01,1050.0
pbs_whsl_Rawalpindi,2009-12-01,1088.0
pbs_whsl_Rawalpindi,2010-01-01,1143.0
pbs_whsl_Rawalpindi,2010-02-01,1160.0
pbs_whsl_Rawalpindi,2010-03-01,1140.0
pbs_whsl_Rawalpindi,2010-04-01,1082.0
pbs_whsl_Rawalpindi,2010-05-01,1060.0
pbs_whsl_Rawalpindi,2010-06-01,1020.0
pbs_whsl_Rawalpindi,2010-07-01,968.0
pbs_whsl_Rawalpindi,2010-08-01,976.0
pbs_whsl_Rawalpindi,2010-09-01,1002.0
pbs_whsl_Rawalpindi,2010-10-01,965.0
pbs_whsl_Rawalpindi,2010-11-01,989.0
pbs_whsl_Rawalpindi,2010-12-01,1013.0
pbs_whsl_Rawalpindi,2011-01-01,1020.0
pbs_whsl_Rawalpindi,2011-02-01,1055.0
pbs_whsl_Rawalpindi,2011-03-01,1070.0
pbs_whsl_Rawalpindi,2011-04-01,943.0
pbs_whsl_Rawalpindi,2011-05-01,891.0
pbs_whsl_Rawalpindi,2011-06-01,932.0
pbs_whsl_Rawalpindi,2011-07-01,960.0
pbs_whsl_Rawalpindi,2011-08-01,955.0
pbs_whsl_Rawalpindi,2011-09-01,951.0
pbs_whsl_Rawalpindi,2011-10-01,973.0
pbs_whsl_Rawalpindi,2011-11-01,991.0
pbs_whsl_Rawalpindi,2011-12-01,1043.0
pbs_whsl_Rawalpindi,2012-01-01,1054.0
pbs_whsl_Rawalpindi,2012-02-01,1063.0
pbs_whsl_Rawalpindi,2012-03-01,1080.0
pbs_whsl_Rawalpindi,2012-04-01,1052.0
pbs_whsl_Rawalpindi,2012-05-01,1024.0
pbs_whsl_Rawalpindi,2012-06-01,1069.0
pbs_whsl_Rawalpindi,2012-07-01,1090.0
pbs_whsl_Rawalpindi,2012-08-01,1090.0
pbs_whsl_Rawalpindi,2012-09-01,1140.0
pbs_whsl_Rawalpindi,2012-10-01,1150.0
pbs_whsl_Rawalpindi,2012-11-01,1150.0
pbs_whsl_Rawalpindi,2012-12-01,1200.0
pbs_whsl_Rawalpindi,2013-01-01,1270.0
pbs_whsl_Rawalpindi,2013-02-01,1350.0
pbs_whsl_Rawalpindi,2013-03-01,1330.0
pbs_whsl_Rawalpindi,2013-04-01,1276.0
pbs_whsl_Rawalpindi,2013-05-01,1210.0
pbs_whsl_Rawalpindi,2013-06-01,1250.0
pbs_whsl_Rawalpindi,2013-07-01,1250.0
pbs_whsl_Rawalpindi,2013-08-01,1385.2
pbs_whsl_Rawalpindi,2013-09-01,1410.0
pbs_whsl_Rawalpindi,2013-10-01,1430.0
pbs_whsl_Rawalpindi,2013-11-01,1430.0
pbs_whsl_Rawalpindi,2013-12-01,1430.0
pbs_whsl_Rawalpindi,2014-01-01,1540.0
pbs_whsl_Rawalpindi,2014-02-01,1580.0
pbs_whsl_Rawalpindi,2014-03-01,1610.0
pbs_whsl_Rawalpindi,2014-04-01,1610.0
pbs_whsl_Rawalpindi,2014-05-01,1380.0
pbs_whsl_Rawalpindi,2014-06-01,1380.0
pbs_whsl_Rawalpindi,2014-07-01,1420.0
pbs_whsl_Rawalpindi,2014-08-01,1420.0
pbs_whsl_Rawalpindi,2014-09-01,1460.0
pbs_whsl_Rawalpindi,2014-10-01,1460.0
pbs_whsl_Rawalpindi,2014-11-01,1460.0
pbs_whsl_Rawalpindi,2014-12-01,1460.0
pbs_whsl_Rawalpindi,2015-01-01,1410.0
pbs_whsl_Rawalpindi,2015-02-01,1424.8
pbs_whsl_Rawalpindi,2015-03-01,1426.0
pbs_whsl_Rawalpindi,2015-04-01,1410.0
pbs_whsl_Rawalpindi,2015-05-01,1330.0
pbs_whsl_Rawalpindi,2015-06-01,1330.0
pbs_whsl_Rawalpindi,2015-07-01,1370.0
pbs_whsl_Rawalpindi,2015-08-01,1370.0
pbs_whsl_Rawalpindi,2015-09-01,1370.0
pbs_whsl_Rawalpindi,2015-10-01,1410.0
pbs_whsl_Rawalpindi,2015-11-01,1410.0
pbs_whsl_Rawalpindi,2015-12-01,1420.0
pbs_whsl_Rawalpindi,2016-01-01,1420.0
pbs_whsl_Rawalpindi,2016-02-01,1460.0
pbs_whsl_Rawalpindi,2016-03-01,1480.0
pbs_whsl_Rawalpindi,2016-04-01,1480.0
pbs_whsl_Rawalpindi,2016-05-01,1350.0
pbs_whsl_Rawalpindi,2016-06-01,1310.0
pbs_whsl_Rawalpindi,2016-07-01,1254.0
pbs_whsl_Rawalpindi,2016-08-01,1259.0
pbs_whsl_Rawalpindi,2016-09-01,1319.0
pbs_whsl_Rawalpindi,2016-10-01,1302.0
pbs_whsl_Rawalpindi,2016-11-01,1370.0
pbs_whsl_Rawalpindi,2016-12-01,1381.0
pbs_whsl_Rawalpindi,2017-01-01,1400.0
pbs_whsl_Rawalpindi,2017-02-01,1326.0
pbs_whsl_Rawalpindi,2017-03-01,1354.0
pbs_whsl_Rawalpindi,2017-04-01,1316.0
pbs_whsl_Rawalpindi,2017-05-01,1242.0
pbs_whsl_Rawalpindi,2017-06-01,1235.0
pbs_whsl_Rawalpindi,2017-07-01,1330.0
pbs_whsl_Rawalpindi,2017-08-01,1330.0
pbs_whsl_Rawalpindi,2017-09-01,1330.0
pbs_whsl_Rawalpindi,2017-10-01,1330.0
pbs_whsl_Rawalpindi,2017-11-01,1360.0
pbs_whsl_Rawalpindi,2017-12-01,1380.0
pbs_whsl_Rawalpindi,2018-01-01,1380.0
pbs_whsl_Rawalpindi,2018-02-01,1370.0
pbs_whsl_Rawalpindi,2018-03-01,1370.0
pbs_whsl_Rawalpindi,2018-04-01,1370.0
pbs_whsl_Rawalpindi,2018-05-01,1330.0
pbs_whsl_Rawalpindi,2018-06-01,1300.0
pbs_whsl_Rawalpindi,2018-07-01,1310.0
pbs_whsl_Rawalpindi,2018-08-01,1310.0
pbs_whsl_Rawalpindi,2018-09-01,1330.0
pbs_whsl_Rawalpindi,2018-10-01,1330.0
pbs_whsl_Rawalpindi,2018-11-01,1380.0
pbs_whsl_Rawalpindi,2018-12-01,1430.0
pbs_whsl_Rawalpindi,2019-01-01,1430.0
pbs_whsl_Rawalpindi,2019-02-01,1450.0
pbs_whsl_Rawalpindi,2019-03-01,1450.0
pbs_whsl_Rawalpindi,2019-04-01,1450.0
pbs_whsl_Rawalpindi,2019-05-01,1450.0
pbs_whsl_Rawalpindi,2019-06-01,1490.0
pbs_whsl_Rawalpindi,2019-07-01,1489.97
pbs_whsl_Rawalpindi,2019-08-01,1513.3
pbs_whsl_Rawalpindi,2019-09-01,1513.3
pbs_whsl_Rawalpindi,2019-10-01,1513.3
pbs_whsl_Rawalpindi,2019-11-01,1586.55
pbs_whsl_Rawalpindi,2019-12-01,1693.31
pbs_whsl_Rawalpindi,2020-01-01,1779.92
pbs_whsl_Rawalpindi,2020-02-01,1779.92
pbs_whsl_Rawalpindi,2020-03-01,1733.31
pbs_whsl_Rawalpindi,2020-04-01,1972.97
pbs_whsl_Rawalpindi,2020-05-01,1593.3
pbs_whsl_Rawalpindi,2020-06-01,1746.56
pbs_whsl_Rawalpindi,2020-07-01,1986.58
pbs_whsl_Rawalpindi,2020-08-01,2113.18
pbs_whsl_Rawalpindi,2020-09-01,2166.65
pbs_whsl_Rawalpindi,2020-10-01,2339.94
pbs_whsl_Rawalpindi,2020-11-01,2400.0
pbs_whsl_Rawalpindi,2020-12-01,2279.77
pbs_whsl_Rawalpindi,2021-01-01,2333.26
pbs_whsl_Rawalpindi,2021-02-01,2426.59
pbs_whsl_Rawalpindi,2021-03-01,2400.0
pbs_whsl_Rawalpindi,2021-04-01,2306.59
pbs_whsl_Rawalpindi,2021-05-01,2013.24
pbs_whsl_Rawalpindi,2021-06-01,2019.93
pbs_whsl_Rawalpindi,2021-07-01,2033.31
pbs_whsl_Rawalpindi,2021-08-01,2033.31
pbs_whsl_Rawalpindi,2021-09-01,2099.9
pbs_whsl_Rawalpindi,2021-10-01,2239.76
pbs_whsl_Rawalpindi,2021-11-01,2359.9
pbs_whsl_Rawalpindi,2021-12-01,2386.6
pbs_whsl_Rawalpindi,2022-01-01,2399.9
pbs_whsl_Rawalpindi,2022-02-01,2609.988
pbs_whsl_Rawalpindi,2022-03-01,2499.99
pbs_whsl_Rawalpindi,2022-04-01,2511.99
pbs_whsl_Rawalpindi,2022-05-01,2511.99
pbs_whsl_Rawalpindi,2022-06-01,2511.99
pbs_whsl_Rawalpindi,2022-07-01,2839.952
pbs_whsl_Rawalpindi,2022-08-01,2899.956
pbs_whsl_Rawalpindi,2022-09-01,3766.664
pbs_whsl_Rawalpindi,2022-10-01,3559.852
pbs_whsl_Rawalpindi,2022-11-01,3559.852
pbs_whsl_Rawalpindi,2022-12-01,3726.656
pbs_whsl_Rawalpindi,2023-01-01,4959.892
pbs_whsl_Rawalpindi,2023-02-01,4623.996
pbs_whsl_Rawalpindi,2023-03-01,4499.972
pbs_whsl_Rawalpindi,2023-04-01,4586.628
pbs_whsl_Rawalpindi,2023-05-01,5453.104
pbs_whsl_Rawalpindi,2023-06-01,4639.884
pbs_whsl_Rawalpindi,2023-07-01,4786.628
pbs_whsl_Rawalpindi,2023-08-01,4786.628
pbs_whsl_Rawalpindi,2023-09-01,4799.556
pbs_whsl_Rawalpindi,2023-10-01,4799.556
pbs_whsl_Rawalpindi,2023-11-01,5099.972
pbs_whsl_Rawalpindi,2023-12-01,5099.972
pbs_whsl_Rawalpindi,2024-01-01,5099.972
pbs_whsl_Rawalpindi,2024-02-01,5099.972
pbs_whsl_Rawalpindi,2024-03-01,5099.972
pbs_whsl_Rawalpindi,2024-04-01,4113.324
pbs_whsl_Rawalpindi,2024-05-01,3419.96
pbs_whsl_Rawalpindi,2024-06-01,3379.96
pbs_whsl_Rawalpindi,2024-07-01,3580.0
pbs_whsl_Rawalpindi,2024-08-01,3340.0
pbs_whsl_Rawalpindi,2024-09-01,3060.0
pbs_whsl_Rawalpindi,2024-10-01,3180.0
pbs_whsl_Rawalpindi,2024-11-01,3420.0
pbs_whsl_Rawalpindi,2024-12-01,3420.0
pbs_whsl_Rawalpindi,2025-01-01,3360.0
pbs_whsl_Rawalpindi,2025-02-01,3360.0
pbs_whsl_Rawalpindi,2025-03-01,3360.0
pbs_whsl_Rawalpindi,2025-04-01,2980.0
pbs_whsl_Rawalpindi,2025-05-01,2700.0
pbs_whsl_Multan,2009-04-01,976.0
pbs_whsl_Multan,2009-05-01,950.0
pbs_whsl_Multan,2009-06-01,910.0
pbs_whsl_Multan,2009-07-01,933.0
pbs_whsl_Multan,2009-08-01,1044.0
pbs_whsl_Multan,2009-09-01,1035.0
pbs_whsl_Multan,2009-10-01,1054.0
pbs_whsl_Multan,2009-11-01,1020.0
pbs_whsl_Multan,2009-12-01,1020.0
pbs_whsl_Multan,2010-01-01,1031.0
pbs_whsl_Multan,2010-02-01,1050.0
pbs_whsl_Multan,2010-03-01,1030.0
pbs_whsl_Multan,2010-04-01,950.0
pbs_whsl_Multan,2010-05-01,920.0
pbs_whsl_Multan,2010-06-01,868.0
pbs_whsl_Multan,2010-07-01,896.0
pbs_whsl_Multan,2010-08-01,907.0
pbs_whsl_Multan,2010-09-01,974.0
pbs_whsl_Multan,2010-10-01,993.0
pbs_whsl_Multan,2010-11-01,1034.0
pbs_whsl_Multan,2010-12-01,1034.0
pbs_whsl_Multan,2011-01-01,1032.0
pbs_whsl_Multan,2011-02-01,1038.0
pbs_whsl_Multan,2011-03-01,1044.0
pbs_whsl_Multan,2011-04-01,967.0
pbs_whsl_Multan,2011-05-01,883.0
pbs_whsl_Multan,2011-06-01,899.0
pbs_whsl_Multan,2011-07-01,938.0
pbs_whsl_Multan,2011-08-01,932.0
pbs_whsl_Multan,2011-09-01,947.0
pbs_whsl_Multan,2011-10-01,967.0
pbs_whsl_Multan,2011-11-01,1007.0
pbs_whsl_Multan,2011-12-01,1050.0
pbs_whsl_Multan,2012-01-01,1035.0
pbs_whsl_Multan,2012-02-01,1027.0
pbs_whsl_Multan,2012-03-01,1035.0
pbs_whsl_Multan,2012-04-01,1021.0
pbs_whsl_Multan,2012-05-01,979.0
pbs_whsl_Multan,2012-06-01,996.0
pbs_whsl_Multan,2012-07-01,980.0
pbs_whsl_Multan,2012-08-01,1050.0
pbs_whsl_Multan,2012-09-01,1100.0
pbs_whsl_Multan,2012-10-01,1120.0
pbs_whsl_Multan,2012-11-01,1160.0
pbs_whsl_Multan,2012-12-01,1160.0
pbs_whsl_Multan,2013-01-01,1300.0
pbs_whsl_Multan,2013-02-01,1240.0
pbs_whsl_Multan,2013-03-01,1260.0
pbs_whsl_Multan,2013-04-01,1140.0
pbs_whsl_Multan,2013-05-01,1160.0
pbs_whsl_Multan,2013-06-01,1250.0
pbs_whsl_Multan,2013-07-01,1320.0
pbs_whsl_Multan,2013-08-01,1420.0
pbs_whsl_Multan,2013-09-01,1400.0
pbs_whsl_Multan,2013-10-01,1420.0
pbs_whsl_Multan,2013-11-01,1450.0
pbs_whsl_Multan,2013-12-01,1560.0
pbs_whsl_Multan,2014-01-01,1480.0
pbs_whsl_Multan,2014-02-01,1520.0
pbs_whsl_Multan,2014-03-01,1588.0
pbs_whsl_Multan,2014-04-01,1560.0
pbs_whsl_Multan,2014-05-01,1230.0
pbs_whsl_Multan,2014-06-01,1260.0
pbs_whsl_Multan,2014-07-01,1320.0
pbs_whsl_Multan,2014-08-01,1400.0
pbs_whsl_Multan,2014-09-01,1400.0
pbs_whsl_Multan,2014-10-01,1360.0
pbs_whsl_Multan,2014-11-01,1360.0
pbs_whsl_Multan,2014-12-01,1300.0
pbs_whsl_Multan,2015-01-01,1280.0
pbs_whsl_Multan,2015-02-01,1320.0
pbs_whsl_Multan,2015-03-01,1300.0
pbs_whsl_Multan,2015-04-01,1290.0
pbs_whsl_Multan,2015-05-01,1260.0
pbs_whsl_Multan,2015-06-01,1180.0
pbs_whsl_Multan,2015-07-01,1220.0
pbs_whsl_Multan,2015-08-01,1230.0
pbs_whsl_Multan,2015-09-01,1240.0
pbs_whsl_Multan,2015-10-01,1280.0
pbs_whsl_Multan,2015-11-01,1340.0
pbs_whsl_Multan,2015-12-01,1340.0
pbs_whsl_Multan,2016-01-01,1340.0
pbs_whsl_Multan,2016-02-01,1350.0
pbs_whsl_Multan,2016-03-01,1370.0
pbs_whsl_Multan,2016-04-01,1300.0
pbs_whsl_Multan,2016-05-01,1200.0
pbs_whsl_Multan,2016-06-01,1250.0
pbs_whsl_Multan,2016-07-01,1269.0
pbs_whsl_Multan,2016-08-01,1255.0
pbs_whsl_Multan,2016-09-01,1290.0
pbs_whsl_Multan,2016-10-01,1290.0
pbs_whsl_Multan,2016-11-01,1279.0
pbs_whsl_Multan,2016-12-01,1317.0
pbs_whsl_Multan,2017-01-01,1334.0
pbs_whsl_Multan,2017-02-01,1322.0
pbs_whsl_Multan,2017-03-01,1345.0
pbs_whsl_Multan,2017-04-01,1202.0
pbs_whsl_Multan,2017-05-01,1187.0
pbs_whsl_Multan,2017-06-01,1204.0
pbs_whsl_Multan,2017-07-01,1210.0
pbs_whsl_Multan,2017-08-01,1170.0
pbs_whsl_Multan,2017-09-01,1190.0
pbs_whsl_Multan,2017-10-01,1190.0
pbs_whsl_Multan,2017-11-01,1270.0
pbs_whsl_Multan,2017-12-01,1290.0
pbs_whsl_Multan,2018-01-01,1310.0
pbs_whsl_Multan,2018-02-01,1310.0
pbs_whsl_Multan,2018-03-01,1220.0
pbs_whsl_Multan,2018-04-01,1220.0
pbs_whsl_Multan,2018-05-01,1170.0
pbs_whsl_Multan,2018-06-01,1170.0
pbs_whsl_Multan,2018-07-01,1230.0
pbs_whsl_Multan,2018-08-01,1300.0
pbs_whsl_Multan,2018-09-01,1300.0
pbs_whsl_Multan,2018-10-01,1300.0
pbs_whsl_Multan,2018-11-01,1300.0
pbs_whsl_Multan,2018-12-01,1300.0
pbs_whsl_Multan,2019-01-01,1320.0
pbs_whsl_Multan,2019-02-01,1342.0
pbs_whsl_Multan,2019-03-01,1360.0
pbs_whsl_Multan,2019-04-01,1325.4
pbs_whsl_Multan,2019-05-01,1310.0
pbs_whsl_Multan,2019-06-01,1400.0
pbs_whsl_Multan,2019-07-01,1406.64
pbs_whsl_Multan,2019-08-01,1406.64
pbs_whsl_Multan,2019-09-01,1443.32
pbs_whsl_Multan,2019-10-01,1443.32
pbs_whsl_Multan,2019-11-01,1566.64
pbs_whsl_Multan,2019-12-01,1626.64
pbs_whsl_Multan,2020-01-01,1886.64
pbs_whsl_Multan,2020-02-01,1743.33
pbs_whsl_Multan,2020-03-01,1629.86
pbs_whsl_Multan,2020-04-01,1446.64
pbs_whsl_Multan,2020-05-01,1406.64
pbs_whsl_Multan,2020-06-01,1506.64
pbs_whsl_Multan,2020-07-01,1773.23
pbs_whsl_Multan,2020-08-01,2073.31
pbs_whsl_Multan,2020-09-01,2086.64
pbs_whsl_Multan,2020-10-01,2266.65
pbs_whsl_Multan,2020-11-01,2266.65
pbs_whsl_Multan,2020-12-01,2266.65
pbs_whsl_Multan,2021-01-01,2266.65
pbs_whsl_Multan,2021-02-01,2286.65
pbs_whsl_Multan,2021-03-01,2286.65
pbs_whsl_Multan,2021-04-01,1806.64
pbs_whsl_Multan,2021-05-01,1866.64
pbs_whsl_Multan,2021-06-01,1966.64
pbs_whsl_Multan,2021-07-01,1986.644
pbs_whsl_Multan,2021-08-01,1986.644
pbs_whsl_Multan,2021-09-01,2073.312
pbs_whsl_Multan,2021-10-01,2186.648
pbs_whsl_Multan,2021-11-01,2286.648
pbs_whsl_Multan,2021-12-01,2286.648
pbs_whsl_Multan,2022-01-01,2286.648
pbs_whsl_Multan,2022-02-01,2373.26
pbs_whsl_Multan,2022-03-01,2373.26
pbs_whsl_Multan,2022-04-01,2200.0
pbs_whsl_Multan,2022-05-01,2200.0
pbs_whsl_Multan,2022-06-01,2200.0
pbs_whsl_Multan,2022-07-01,2600.0
pbs_whsl_Multan,2022-08-01,2626.6
pbs_whsl_Multan,2022-09-01,2973.272
pbs_whsl_Multan,2022-10-01,3173.276
pbs_whsl_Multan,2022-11-01,3173.276
pbs_whsl_Multan,2022-12-01,3693.284
pbs_whsl_Multan,2023-01-01,4800.0
pbs_whsl_Multan,2023-02-01,4529.484
pbs_whsl_Multan,2023-03-01,4000.0
pbs_whsl_Multan,2023-04-01,4000.0
pbs_whsl_Multan,2023-05-01,4100.0
pbs_whsl_Multan,2023-06-01,4100.0
pbs_whsl_Multan,2023-07-01,5000.0
pbs_whsl_Multan,2023-08-01,4966.44
pbs_whsl_Multan,2023-09-01,4682.604
pbs_whsl_Multan,2023-10-01,4615.944
pbs_whsl_Multan,2023-11-01,4700.0
pbs_whsl_Multan,2023-12-01,4800.0
pbs_whsl_Multan,2024-01-01,4800.0
pbs_whsl_Multan,2024-02-01,4800.0
pbs_whsl_Multan,2024-03-01,4800.0
pbs_whsl_Multan,2024-04-01,3733.036
pbs_whsl_Multan,2024-05-01,3232.992
pbs_whsl_Multan,2024-06-01,3030.792
pbs_whsl_Multan,2024-07-01,3065.2
pbs_whsl_Multan,2024-08-01,3012.4
pbs_whsl_Multan,2024-09-01,2559.2
pbs_whsl_Multan,2024-10-01,2766.4
pbs_whsl_Multan,2024-11-01,2932.8
pbs_whsl_Multan,2024-12-01,2932.8
pbs_whsl_Multan,2025-01-01,2932.8
pbs_whsl_Multan,2025-02-01,2932.8
pbs_whsl_Multan,2025-03-01,2932.8
pbs_whsl_Multan,2025-04-01,2366.0
pbs_whsl_Multan,2025-05-01,2286.8
pbs_whsl_Multan,2025-06-01,2380.0
pbs_whsl_Hyderabad,2012-07-01,1020.0
pbs_whsl_Hyderabad,2012-08-01,1112.0
pbs_whsl_Hyderabad,2012-09-01,1160.0
pbs_whsl_Hyderabad,2012-10-01,1168.0
pbs_whsl_Hyderabad,2012-11-01,1176.0
pbs_whsl_Hyderabad,2012-12-01,1176.0
pbs_whsl_Hyderabad,2013-01-01,1280.0
pbs_whsl_Hyderabad,2013-02-01,1340.0
pbs_whsl_Hyderabad,2013-03-01,1240.0
pbs_whsl_Hyderabad,2013-04-01,1128.0
pbs_whsl_Hyderabad,2013-05-01,1240.0
pbs_whsl_Hyderabad,2013-06-01,1248.0
pbs_whsl_Hyderabad,2013-07-01,1328.0
pbs_whsl_Hyderabad,2013-08-01,1420.0
pbs_whsl_Hyderabad,2013-09-01,1420.0
pbs_whsl_Hyderabad,2013-10-01,1420.0
pbs_whsl_Hyderabad,2013-11-01,1460.0
pbs_whsl_Hyderabad,2013-12-01,1460.0
pbs_whsl_Hyderabad,2014-01-01,1500.0
pbs_whsl_Hyderabad,2014-02-01,1520.0
pbs_whsl_Hyderabad,2014-03-01,1480.0
pbs_whsl_Hyderabad,2014-04-01,1280.0
pbs_whsl_Hyderabad,2014-05-01,1310.0
pbs_whsl_Hyderabad,2014-06-01,1312.0
pbs_whsl_Hyderabad,2014-07-01,1320.0
pbs_whsl_Hyderabad,2014-08-01,1312.0
pbs_whsl_Hyderabad,2014-09-01,1312.0
pbs_whsl_Hyderabad,2014-10-01,1360.0
pbs_whsl_Hyderabad,2014-11-01,1364.0
pbs_whsl_Hyderabad,2014-12-01,1356.0
pbs_whsl_Hyderabad,2015-01-01,1326.0
pbs_whsl_Hyderabad,2015-02-01,1328.0
pbs_whsl_Hyderabad,2015-03-01,1240.0
pbs_whsl_Hyderabad,2015-04-01,1200.0
pbs_whsl_Hyderabad,2015-05-01,1252.0
pbs_whsl_Hyderabad,2015-06-01,1192.0
pbs_whsl_Hyderabad,2015-07-01,1192.0
pbs_whsl_Hyderabad,2015-08-01,1248.0
pbs_whsl_Hyderabad,2015-09-01,1260.0
pbs_whsl_Hyderabad,2015-10-01,1360.0
pbs_whsl_Hyderabad,2015-11-01,1372.0
pbs_whsl_Hyderabad,2015-12-01,1378.0
pbs_whsl_Hyderabad,2016-01-01,1381.0
pbs_whsl_Hyderabad,2016-02-01,1381.0
pbs_whsl_Hyderabad,2016-03-01,1221.0
pbs_whsl_Hyderabad,2016-04-01,1160.0
pbs_whsl_Hyderabad,2016-05-01,1240.0
pbs_whsl_Hyderabad,2016-06-01,1300.0
pbs_whsl_Hyderabad,2016-07-01,1304.0
pbs_whsl_Hyderabad,2016-08-01,1304.0
pbs_whsl_Hyderabad,2016-09-01,1304.0
pbs_whsl_Hyderabad,2016-10-01,1336.0
pbs_whsl_Hyderabad,2016-11-01,1352.0
pbs_whsl_Hyderabad,2016-12-01,1352.0
pbs_whsl_Hyderabad,2017-01-01,1356.0
pbs_whsl_Hyderabad,2017-02-01,1360.0
pbs_whsl_Hyderabad,2017-03-01,1280.0
pbs_whsl_Hyderabad,2017-04-01,1212.0
pbs_whsl_Hyderabad,2017-05-01,1210.0
pbs_whsl_Hyderabad,2017-06-01,1240.0
pbs_whsl_Hyderabad,2017-07-01,1240.0
pbs_whsl_Hyderabad,2017-08-01,1224.0
pbs_whsl_Hyderabad,2017-09-01,1248.0
pbs_whsl_Hyderabad,2017-10-01,1300.0
pbs_whsl_Hyderabad,2017-11-01,1342.0
pbs_whsl_Hyderabad,2017-12-01,1349.0
pbs_whsl_Hyderabad,2018-01-01,1352.0
pbs_whsl_Hyderabad,2018-02-01,1340.0
pbs_whsl_Hyderabad,2018-03-01,1260.0
pbs_whsl_Hyderabad,2018-04-01,1190.0
pbs_whsl_Hyderabad,2018-05-01,1230.0
pbs_whsl_Hyderabad,2018-06-01,1296.0
pbs_whsl_Hyderabad,2018-07-01,1340.0
pbs_whsl_Hyderabad,2018-08-01,1344.0
pbs_whsl_Hyderabad,2018-09-01,1360.0
pbs_whsl_Hyderabad,2018-10-01,1336.0
pbs_whsl_Hyderabad,2018-11-01,1380.0
pbs_whsl_Hyderabad,2018-12-01,1360.0
pbs_whsl_Hyderabad,2019-01-01,1380.0
pbs_whsl_Hyderabad,2019-02-01,1380.0
pbs_whsl_Hyderabad,2019-03-01,1216.0
pbs_whsl_Hyderabad,2019-04-01,1188.0
pbs_whsl_Hyderabad,2019-05-01,1294.0
pbs_whsl_Hyderabad,2019-06-01,1408.0
pbs_whsl_Hyderabad,2019-07-01,1374.66
pbs_whsl_Hyderabad,2019-08-01,1408.0
pbs_whsl_Hyderabad,2019-09-01,1560.0
pbs_whsl_Hyderabad,2019-10-01,1580.0
pbs_whsl_Hyderabad,2019-11-01,1806.644
pbs_whsl_Hyderabad,2019-12-01,1733.232
pbs_whsl_Hyderabad,2020-01-01,1933.24
pbs_whsl_Hyderabad,2020-02-01,1733.232
pbs_whsl_Hyderabad,2020-03-01,1533.22
pbs_whsl_Hyderabad,2020-04-01,1394.66
pbs_whsl_Hyderabad,2020-05-01,1573.22
pbs_whsl_Hyderabad,2020-06-01,1660.0
pbs_whsl_Hyderabad,2020-07-01,1760.0
pbs_whsl_Hyderabad,2020-08-01,2000.0
pbs_whsl_Hyderabad,2020-09-01,1940.0
pbs_whsl_Hyderabad,2020-10-01,2213.252
pbs_whsl_Hyderabad,2020-11-01,2000.0
pbs_whsl_Hyderabad,2020-12-01,1913.312
pbs_whsl_Hyderabad,2021-01-01,2306.648
pbs_whsl_Hyderabad,2021-02-01,2266.648
pbs_whsl_Hyderabad,2021-03-01,2120.0
pbs_whsl_Hyderabad,2021-04-01,1980.0
pbs_whsl_Hyderabad,2021-05-01,2100.0
pbs_whsl_Hyderabad,2021-06-01,2080.0
pbs_whsl_Hyderabad,2021-07-01,2080.0
pbs_whsl_Hyderabad,2021-08-01,2046.644
pbs_whsl_Hyderabad,2021-09-01,2253.256
pbs_whsl_Hyderabad,2021-10-01,2386.648
pbs_whsl_Hyderabad,2021-11-01,2433.316
pbs_whsl_Hyderabad,2021-12-01,2380.0
pbs_whsl_Hyderabad,2022-01-01,2460.0
pbs_whsl_Hyderabad,2022-02-01,2333.316
pbs_whsl_Hyderabad,2022-03-01,2273.312
pbs_whsl_Hyderabad,2022-04-01,2293.256
pbs_whsl_Hyderabad,2022-05-01,2293.256
pbs_whsl_Hyderabad,2022-06-01,2293.256
pbs_whsl_Hyderabad,2022-07-01,3086.652
pbs_whsl_Hyderabad,2022-08-01,3042.664
pbs_whsl_Hyderabad,2022-09-01,3673.32
pbs_whsl_Hyderabad,2022-10-01,3466.464
pbs_whsl_Hyderabad,2022-11-01,3346.652
pbs_whsl_Hyderabad,2022-12-01,3813.144
pbs_whsl_Hyderabad,2023-01-01,3920.0
pbs_whsl_Hyderabad,2023-02-01,4200.0
pbs_whsl_Hyderabad,2023-03-01,4400.0
pbs_whsl_Hyderabad,2023-04-01,4520.0
pbs_whsl_Hyderabad,2023-05-01,4600.0
pbs_whsl_Hyderabad,2023-06-01,4480.0
pbs_whsl_Hyderabad,2023-07-01,5000.0
pbs_whsl_Hyderabad,2023-08-01,4946.524
pbs_whsl_Hyderabad,2023-09-01,4653.296
pbs_whsl_Hyderabad,2023-10-01,4560.0
pbs_whsl_Hyderabad,2023-11-01,4840.0
pbs_whsl_Hyderabad,2023-12-01,4740.0
pbs_whsl_Hyderabad,2024-01-01,4666.656
pbs_whsl_Hyderabad,2024-02-01,4606.656
pbs_whsl_Hyderabad,2024-03-01,3893.148
pbs_whsl_Hyderabad,2024-04-01,3573.284
pbs_whsl_Hyderabad,2024-05-01,3093.104
pbs_whsl_Hyderabad,2024-06-01,2973.32
pbs_whsl_Hyderabad,2024-07-01,3226.8
pbs_whsl_Hyderabad,2024-08-01,3000.0
pbs_whsl_Hyderabad,2024-09-01,2826.8
pbs_whsl_Hyderabad,2024-10-01,3093.2
pbs_whsl_Hyderabad,2024-11-01,3066.8
pbs_whsl_Hyderabad,2024-12-01,2986.8
pbs_whsl_Hyderabad,2025-01-01,3126.8
pbs_whsl_Hyderabad,2025-02-01,2846.8
pbs_whsl_Hyderabad,2025-03-01,3146.8
pbs_whsl_Hyderabad,2025-04-01,2546.4
pbs_whsl_Hyderabad,2025-05-01,2486.8
pbs_whsl_Hyderabad,2025-06-01,2533.2
pbs_whsl_Sukkur,2012-07-01,995.0
pbs_whsl_Sukkur,2012-08-01,1055.0
pbs_whsl_Sukkur,2012-09-01,1100.0
pbs_whsl_Sukkur,2012-10-01,1200.0
pbs_whsl_Sukkur,2012-11-01,1200.0
pbs_whsl_Sukkur,2012-12-01,1200.0
pbs_whsl_Sukkur,2013-01-01,1300.0
pbs_whsl_Sukkur,2013-02-01,1400.0
pbs_whsl_Sukkur,2013-03-01,1400.0
pbs_whsl_Sukkur,2013-04-01,1125.0
pbs_whsl_Sukkur,2013-05-01,1200.0
pbs_whsl_Sukkur,2013-06-01,1200.0
pbs_whsl_Sukkur,2013-07-01,1300.0
pbs_whsl_Sukkur,2013-08-01,1350.0
pbs_whsl_Sukkur,2013-09-01,1360.0
pbs_whsl_Sukkur,2013-10-01,1400.0
pbs_whsl_Sukkur,2013-11-01,1450.0
pbs_whsl_Sukkur,2013-12-01,1563.2
pbs_whsl_Sukkur,2014-01-01,1563.2
pbs_whsl_Sukkur,2014-02-01,1600.0
pbs_whsl_Sukkur,2014-03-01,1600.0
pbs_whsl_Sukkur,2014-04-01,1300.0
pbs_whsl_Sukkur,2014-05-01,1280.0
pbs_whsl_Sukkur,2014-06-01,1280.0
pbs_whsl_Sukkur,2014-07-01,1280.0
pbs_whsl_Sukkur,2014-08-01,1360.0
pbs_whsl_Sukkur,2014-09-01,1360.0
pbs_whsl_Sukkur,2014-10-01,1360.0
pbs_whsl_Sukkur,2014-11-01,1360.0
pbs_whsl_Sukkur,2014-12-01,1360.0
pbs_whsl_Sukkur,2015-01-01,1305.0
pbs_whsl_Sukkur,2015-02-01,1315.0
pbs_whsl_Sukkur,2015-03-01,1315.0
pbs_whsl_Sukkur,2015-04-01,1360.0
pbs_whsl_Sukkur,2015-05-01,1200.0
pbs_whsl_Sukkur,2015-06-01,1200.0
pbs_whsl_Sukkur,2015-07-01,1200.0
pbs_whsl_Sukkur,2015-08-01,1200.0
pbs_whsl_Sukkur,2015-09-01,1200.0
pbs_whsl_Sukkur,2015-10-01,1360.0
pbs_whsl_Sukkur,2015-11-01,1360.0
pbs_whsl_Sukkur,2015-12-01,1360.0
pbs_whsl_Sukkur,2016-01-01,1360.0
pbs_whsl_Sukkur,2016-02-01,1360.0
pbs_whsl_Sukkur,2016-03-01,1360.0
pbs_whsl_Sukkur,2016-04-01,1250.0
pbs_whsl_Sukkur,2016-05-01,1250.0
pbs_whsl_Sukkur,2016-06-01,1360.0
pbs_whsl_Sukkur,2016-07-01,1360.0
pbs_whsl_Sukkur,2016-08-01,1360.0
pbs_whsl_Sukkur,2016-09-01,1360.0
pbs_whsl_Sukkur,2016-10-01,1360.0
pbs_whsl_Sukkur,2016-11-01,1360.0
pbs_whsl_Sukkur,2016-12-01,1360.0
pbs_whsl_Sukkur,2017-01-01,1360.0
pbs_whsl_Sukkur,2017-02-01,1360.0
pbs_whsl_Sukkur,2017-03-01,1360.0
pbs_whsl_Sukkur,2017-04-01,1250.0
pbs_whsl_Sukkur,2017-05-01,1250.0
pbs_whsl_Sukkur,2017-06-01,1250.0
pbs_whsl_Sukkur,2017-07-01,1250.0
pbs_whsl_Sukkur,2017-08-01,1250.0
pbs_whsl_Sukkur,2017-09-01,1250.0
pbs_whsl_Sukkur,2017-10-01,1300.0
pbs_whsl_Sukkur,2017-11-01,1300.0
pbs_whsl_Sukkur,2017-12-01,1300.0
pbs_whsl_Sukkur,2018-01-01,1300.0
pbs_whsl_Sukkur,2018-02-01,1300.0
pbs_whsl_Sukkur,2018-03-01,1300.0
pbs_whsl_Sukkur,2018-04-01,1300.0
pbs_whsl_Sukkur,2018-05-01,1200.0
pbs_whsl_Sukkur,2018-06-01,1200.0
pbs_whsl_Sukkur,2018-07-01,1200.0
pbs_whsl_Sukkur,2018-08-01,1300.0
pbs_whsl_Sukkur,2018-09-01,1300.0
pbs_whsl_Sukkur,2018-10-01,1300.0
pbs_whsl_Sukkur,2018-11-01,1300.0
pbs_whsl_Sukkur,2018-12-01,1300.0
pbs_whsl_Sukkur,2019-01-01,1300.0
pbs_whsl_Sukkur,2019-02-01,1300.0
pbs_whsl_Sukkur,2019-03-01,1300.0
pbs_whsl_Sukkur,2019-04-01,1280.0
pbs_whsl_Sukkur,2019-05-01,1280.0
pbs_whsl_Sukkur,2019-06-01,1300.0
pbs_whsl_Sukkur,2019-07-01,1300.0
pbs_whsl_Sukkur,2019-08-01,1300.0
pbs_whsl_Sukkur,2019-09-01,1400.0
pbs_whsl_Sukkur,2019-10-01,1400.0
pbs_whsl_Sukkur,2019-11-01,1400.0
pbs_whsl_Sukkur,2019-12-01,1760.0
pbs_whsl_Sukkur,2020-01-01,1900.0
pbs_whsl_Sukkur,2020-02-01,1900.0
pbs_whsl_Sukkur,2020-03-01,1700.0
pbs_whsl_Sukkur,2020-04-01,1400.0
pbs_whsl_Sukkur,2020-05-01,1400.0
pbs_whsl_Sukkur,2020-06-01,1600.0
pbs_whsl_Sukkur,2020-07-01,1800.0
pbs_whsl_Sukkur,2020-08-01,2000.0
pbs_whsl_Sukkur,2020-09-01,2000.0
pbs_whsl_Sukkur,2020-10-01,2200.0
pbs_whsl_Sukkur,2020-11-01,2200.0
pbs_whsl_Sukkur,2020-12-01,2100.0
pbs_whsl_Sukkur,2021-01-01,2200.0
pbs_whsl_Sukkur,2021-02-01,2200.0
pbs_whsl_Sukkur,2021-03-01,2200.0
pbs_whsl_Sukkur,2021-04-01,1900.0
pbs_whsl_Sukkur,2021-05-01,1900.0
pbs_whsl_Sukkur,2021-06-01,2026.4
pbs_whsl_Sukkur,2021-07-01,2040.0
pbs_whsl_Sukkur,2021-08-01,2040.0
pbs_whsl_Sukkur,2021-09-01,2200.0
pbs_whsl_Sukkur,2021-10-01,2400.0
pbs_whsl_Sukkur,2021-11-01,2296.0
pbs_whsl_Sukkur,2021-12-01,2296.0
pbs_whsl_Sukkur,2022-01-01,2500.0
pbs_whsl_Sukkur,2022-02-01,2500.0
pbs_whsl_Sukkur,2022-03-01,2362.08
pbs_whsl_Sukkur,2022-04-01,2200.0
pbs_whsl_Sukkur,2022-05-01,2200.0
pbs_whsl_Sukkur,2022-06-01,2200.0
pbs_whsl_Sukkur,2022-07-01,2800.0
pbs_whsl_Sukkur,2022-08-01,2800.0
pbs_whsl_Sukkur,2022-09-01,3600.0
pbs_whsl_Sukkur,2022-10-01,3600.0
pbs_whsl_Sukkur,2022-11-01,3600.0
pbs_whsl_Sukkur,2022-12-01,3600.0
pbs_whsl_Sukkur,2023-01-01,4200.0
pbs_whsl_Sukkur,2023-02-01,4700.0
pbs_whsl_Sukkur,2023-03-01,4700.0
pbs_whsl_Sukkur,2023-04-01,4700.0
pbs_whsl_Sukkur,2023-05-01,4200.0
pbs_whsl_Sukkur,2023-06-01,4265.636
pbs_whsl_Sukkur,2023-07-01,5000.0
pbs_whsl_Sukkur,2023-08-01,4800.0
pbs_whsl_Sukkur,2023-09-01,4700.0
pbs_whsl_Sukkur,2023-10-01,4493.324
pbs_whsl_Sukkur,2023-11-01,4800.0
pbs_whsl_Sukkur,2023-12-01,5000.0
pbs_whsl_Sukkur,2024-01-01,4600.0
pbs_whsl_Sukkur,2024-02-01,4800.0
pbs_whsl_Sukkur,2024-03-01,4200.0
pbs_whsl_Sukkur,2024-04-01,4000.0
pbs_whsl_Sukkur,2024-05-01,3000.0
pbs_whsl_Sukkur,2024-06-01,2986.608
pbs_whsl_Sukkur,2024-07-01,3300.0
pbs_whsl_Sukkur,2024-08-01,2986.8
pbs_whsl_Sukkur,2024-09-01,2700.0
pbs_whsl_Sukkur,2024-10-01,3050.0
pbs_whsl_Sukkur,2024-11-01,3000.0
pbs_whsl_Sukkur,2024-12-01,2950.0
pbs_whsl_Sukkur,2025-01-01,3200.0
pbs_whsl_Sukkur,2025-02-01,2800.0
pbs_whsl_Sukkur,2025-03-01,3100.0
pbs_whsl_Sukkur,2025-04-01,2500.0
pbs_whsl_Sukkur,2025-05-01,2500.0
pbs_whsl_Sukkur,2025-06-01,2600.0
pbs_whsl_Peshawar,2012-07-01,1028.0
pbs_whsl_Peshawar,2012-08-01,1072.0
pbs_whsl_Peshawar,2012-09-01,1120.0
pbs_whsl_Peshawar,2012-10-01,1136.0
pbs_whsl_Peshawar,2012-11-01,1150.0
pbs_whsl_Peshawar,2012-12-01,1200.0
pbs_whsl_Peshawar,2013-01-01,1320.0
pbs_whsl_Peshawar,2013-02-01,1320.0
pbs_whsl_Peshawar,2013-03-01,1320.0
pbs_whsl_Peshawar,2013-04-01,1240.0
pbs_whsl_Peshawar,2013-05-01,1180.0
pbs_whsl_Peshawar,2013-06-01,1208.0
pbs_whsl_Peshawar,2013-07-01,1280.0
pbs_whsl_Peshawar,2013-08-01,1304.0
pbs_whsl_Peshawar,2013-09-01,1350.0
pbs_whsl_Peshawar,2013-10-01,1350.0
pbs_whsl_Peshawar,2013-11-01,1430.0
pbs_whsl_Peshawar,2013-12-01,1540.0
pbs_whsl_Peshawar,2014-01-01,1560.0
pbs_whsl_Peshawar,2014-02-01,1600.0
pbs_whsl_Peshawar,2014-03-01,1600.0
pbs_whsl_Peshawar,2014-04-01,1540.0
pbs_whsl_Peshawar,2014-05-01,1288.0
pbs_whsl_Peshawar,2014-06-01,1288.0
pbs_whsl_Peshawar,2014-07-01,1360.0
pbs_whsl_Peshawar,2014-08-01,1360.0
pbs_whsl_Peshawar,2014-09-01,1360.0
pbs_whsl_Peshawar,2014-10-01,1360.0
pbs_whsl_Peshawar,2014-11-01,1360.0
pbs_whsl_Peshawar,2014-12-01,1400.0
pbs_whsl_Peshawar,2015-01-01,1500.0
pbs_whsl_Peshawar,2015-02-01,1600.0
pbs_whsl_Peshawar,2015-03-01,1700.0
pbs_whsl_Peshawar,2015-04-01,1600.0
pbs_whsl_Peshawar,2015-05-01,1600.0
pbs_whsl_Peshawar,2015-06-01,1500.0
pbs_whsl_Peshawar,2015-07-01,1140.0
pbs_whsl_Peshawar,2015-08-01,1100.0
pbs_whsl_Peshawar,2015-09-01,1090.0
pbs_whsl_Peshawar,2015-10-01,1200.0
pbs_whsl_Peshawar,2015-11-01,1240.0
pbs_whsl_Peshawar,2015-12-01,1240.0
pbs_whsl_Peshawar,2016-01-01,1272.0
pbs_whsl_Peshawar,2016-02-01,1280.0
pbs_whsl_Peshawar,2016-03-01,1260.0
pbs_whsl_Peshawar,2016-04-01,1260.0
pbs_whsl_Peshawar,2016-05-01,1200.0
pbs_whsl_Peshawar,2016-06-01,1200.0
pbs_whsl_Peshawar,2016-07-01,1280.0
pbs_whsl_Peshawar,2016-08-01,1304.0
pbs_whsl_Peshawar,2016-09-01,1280.0
pbs_whsl_Peshawar,2016-10-01,1306.0
pbs_whsl_Peshawar,2016-11-01,1360.0
pbs_whsl_Peshawar,2016-12-01,1360.0
pbs_whsl_Peshawar,2017-01-01,1400.0
pbs_whsl_Peshawar,2017-02-01,1400.0
pbs_whsl_Peshawar,2017-03-01,1400.0
pbs_whsl_Peshawar,2017-04-01,1400.0
pbs_whsl_Peshawar,2017-05-01,1200.0
pbs_whsl_Peshawar,2017-06-01,1200.0
pbs_whsl_Peshawar,2017-07-01,1192.0
pbs_whsl_Peshawar,2017-08-01,1160.0
pbs_whsl_Peshawar,2017-09-01,1160.0
pbs_whsl_Peshawar,2017-10-01,1200.0
pbs_whsl_Peshawar,2017-11-01,1280.0
pbs_whsl_Peshawar,2017-12-01,1280.0
pbs_whsl_Peshawar,2018-01-01,1290.0
pbs_whsl_Peshawar,2018-02-01,1320.0
pbs_whsl_Peshawar,2018-03-01,1304.0
pbs_whsl_Peshawar,2018-04-01,1280.0
pbs_whsl_Peshawar,2018-05-01,1170.0
pbs_whsl_Peshawar,2018-06-01,1170.0
pbs_whsl_Peshawar,2018-07-01,1280.0
pbs_whsl_Peshawar,2018-08-01,1280.0
pbs_whsl_Peshawar,2018-09-01,1200.0
pbs_whsl_Peshawar,2018-10-01,1240.0
pbs_whsl_Peshawar,2018-11-01,1240.0
pbs_whsl_Peshawar,2018-12-01,1240.0
pbs_whsl_Peshawar,2019-01-01,1290.0
pbs_whsl_Peshawar,2019-02-01,1290.0
pbs_whsl_Peshawar,2019-03-01,1290.0
pbs_whsl_Peshawar,2019-04-01,1290.0
pbs_whsl_Peshawar,2019-05-01,1320.0
pbs_whsl_Peshawar,2019-06-01,1330.0
pbs_whsl_Peshawar,2019-07-01,1440.0
pbs_whsl_Peshawar,2019-08-01,1400.0
pbs_whsl_Peshawar,2019-09-01,1440.0
pbs_whsl_Peshawar,2019-10-01,1480.0
pbs_whsl_Peshawar,2019-11-01,1586.552
pbs_whsl_Peshawar,2019-12-01,1573.22
pbs_whsl_Peshawar,2020-01-01,1679.684
pbs_whsl_Peshawar,2020-02-01,1640.0
pbs_whsl_Peshawar,2020-03-01,1706.256
pbs_whsl_Peshawar,2020-04-01,1626.236
pbs_whsl_Peshawar,2020-05-01,1546.212
pbs_whsl_Peshawar,2020-06-01,1865.112
pbs_whsl_Peshawar,2020-07-01,1930.544
pbs_whsl_Peshawar,2020-08-01,2184.416
pbs_whsl_Peshawar,2020-09-01,2319.08
pbs_whsl_Peshawar,2020-10-01,2280.0
pbs_whsl_Peshawar,2020-11-01,2306.36
pbs_whsl_Peshawar,2020-12-01,1971.916
pbs_whsl_Peshawar,2021-01-01,2066.324
pbs_whsl_Peshawar,2021-02-01,2106.332
pbs_whsl_Peshawar,2021-03-01,2186.344
pbs_whsl_Peshawar,2021-04-01,1973.244
pbs_whsl_Peshawar,2021-05-01,1880.0
pbs_whsl_Peshawar,2021-06-01,1933.24
pbs_whsl_Peshawar,2021-07-01,1906.644
pbs_whsl_Peshawar,2021-08-01,1920.0
pbs_whsl_Peshawar,2021-09-01,2066.58
pbs_whsl_Peshawar,2021-10-01,2266.588
pbs_whsl_Peshawar,2021-11-01,2266.588
pbs_whsl_Peshawar,2021-12-01,2213.252
pbs_whsl_Peshawar,2022-01-01,2360.0
pbs_whsl_Peshawar,2022-02-01,2333.256
pbs_whsl_Peshawar,2022-03-01,2200.0
pbs_whsl_Peshawar,2022-04-01,2360.0
pbs_whsl_Peshawar,2022-05-01,2520.0
pbs_whsl_Peshawar,2022-06-01,2960.0
pbs_whsl_Peshawar,2022-07-01,3240.0
pbs_whsl_Peshawar,2022-08-01,3160.0
pbs_whsl_Peshawar,2022-09-01,3640.0
pbs_whsl_Peshawar,2022-10-01,3560.0
pbs_whsl_Peshawar,2022-11-01,3600.0
pbs_whsl_Peshawar,2022-12-01,4105.968
pbs_whsl_Peshawar,2023-01-01,4600.0
pbs_whsl_Peshawar,2023-02-01,4640.0
pbs_whsl_Peshawar,2023-03-01,5000.0
pbs_whsl_Peshawar,2023-04-01,4960.0
pbs_whsl_Peshawar,2023-05-01,5400.0
pbs_whsl_Peshawar,2023-06-01,4920.0
pbs_whsl_Peshawar,2023-07-01,4920.0
pbs_whsl_Peshawar,2023-08-01,4920.0
pbs_whsl_Peshawar,2023-09-01,4813.296
pbs_whsl_Peshawar,2023-10-01,4880.0
pbs_whsl_Peshawar,2023-11-01,4960.0
pbs_whsl_Peshawar,2023-12-01,5040.0
pbs_whsl_Peshawar,2024-01-01,5186.66
pbs_whsl_Peshawar,2024-02-01,5120.0
pbs_whsl_Peshawar,2024-03-01,5093.324
pbs_whsl_Peshawar,2024-04-01,3872.3
pbs_whsl_Peshawar,2024-05-01,3573.132
pbs_whsl_Peshawar,2024-06-01,3106.608
pbs_whsl_Peshawar,2024-07-01,3353.2
pbs_whsl_Peshawar,2024-08-01,3186.8
pbs_whsl_Peshawar,2024-09-01,2906.8
pbs_whsl_Peshawar,2024-10-01,3200.0
pbs_whsl_Peshawar,2024-11-01,3146.8
pbs_whsl_Peshawar,2024-12-01,3193.2
pbs_whsl_Peshawar,2025-01-01,3200.0
pbs_whsl_Peshawar,2025-02-01,3086.8
pbs_whsl_Peshawar,2025-03-01,3226.8
pbs_whsl_Peshawar,2025-04-01,2773.2
pbs_whsl_Peshawar,2025-05-01,2720.0
pbs_whsl_Peshawar,2025-06-01,2640.0


In [ ]:
%%writefile /content/world.csv
ds,world_rs40_lag1
2006-02-01,400.28
2006-03-01,430.78
2006-04-01,418.78
2006-05-01,432.79
2006-06-01,464.35
2006-07-01,469.92
2006-08-01,488.12
2006-09-01,458.32
2006-10-01,474.52
2006-11-01,514.23
2006-12-01,509.55
2007-01-01,497.72
2007-02-01,477.41
2007-03-01,485.96
2007-04-01,483.47
2007-05-01,481.49
2007-06-01,474.94
2007-07-01,540.81
2007-08-01,575.97
2007-09-01,628.72
2007-10-01,791.92
2007-11-01,813.31
2007-12-01,785.57
2008-01-01,902.08
2008-02-01,925.16
2008-03-01,1065.44
2008-04-01,1103.71
2008-05-01,922.09
2008-06-01,891.31
2008-07-01,939.97
2008-08-01,927.87
2008-09-01,982.76
2008-10-01,914.08
2008-11-01,764.23
2008-12-01,726.4
2009-01-01,696.22
2009-02-01,757.91
2009-03-01,715.26
2009-04-01,743.01
2009-05-01,752.38
2009-06-01,843.79
2009-07-01,832.5
2009-08-01,739.13
2009-09-01,697.36
2009-10-01,634.04
2009-11-01,662.4
2009-12-01,705.13
2010-01-01,694.03
2010-02-01,680.81
2010-03-01,659.3
2010-04-01,644.66
2010-05-01,648.33
2010-06-01,613.52
2010-07-01,538.54
2010-08-01,670.48
2010-09-01,843.78
2010-10-01,933.05
2010-11-01,929.56
2010-12-01,937.45
2011-01-01,1051.69
2011-02-01,1120.2
2011-03-01,1188.39
2011-04-01,1081.59
2011-05-01,1138.25
2011-06-01,1212.11
2011-07-01,1120.67
2011-08-01,1046.68
2011-09-01,1134.4
2011-10-01,1106.35
2011-11-01,1004.94
2011-12-01,977.15
2012-01-01,962.11
2012-02-01,992.53
2012-03-01,1008.1
2012-04-01,1031.07
2012-05-01,966.21
2012-06-01,966.26
2012-07-01,1041.23
2012-08-01,1305.81
2012-09-01,1320.77
2012-10-01,1337.68
2012-11-01,1368.43
2012-12-01,1385.73
2013-01-01,1353.55
2013-02-01,1309.0
2013-03-01,1250.47
2013-04-01,1215.81
2013-05-01,1213.19
2013-06-01,1258.74
2013-07-01,1238.12
2013-08-01,1225.6
2013-09-01,1258.01
2013-10-01,1295.37
2013-11-01,1385.12
2013-12-01,1320.62
2014-01-01,1247.8
2014-02-01,1162.7
2014-03-01,1228.59
2014-04-01,1297.78
2014-05-01,1268.46
2014-06-01,1321.44
2014-07-01,1207.94
2014-08-01,1108.15
2014-09-01,1057.48
2014-10-01,999.25
2014-11-01,1009.56
2014-12-01,1055.94
2015-01-01,1088.31
2015-02-01,1001.87
2015-03-01,962.27
2015-04-01,940.63
2015-05-01,909.3
2015-06-01,877.41
2015-07-01,854.41
2015-08-01,803.28
2015-09-01,737.03
2015-10-01,720.99
2015-11-01,722.8
2015-12-01,747.3
2016-01-01,793.02
2016-02-01,811.34
2016-03-01,783.19
2016-04-01,800.56
2016-05-01,785.2
2016-06-01,720.77
2016-07-01,724.8
2016-08-01,636.68
2016-09-01,625.04
2016-10-01,630.31
2016-11-01,635.88
2016-12-01,631.34
2017-01-01,594.56
2017-02-01,642.49
2017-03-01,650.33
2017-04-01,647.13
2017-05-01,696.61
2017-06-01,756.57
2017-07-01,795.37
2017-08-01,854.84
2017-09-01,721.71
2017-10-01,753.06
2017-11-01,740.45
2017-12-01,758.02
2018-01-01,803.69
2018-02-01,849.87
2018-03-01,850.01
2018-04-01,864.16
2018-05-01,988.68
2018-06-01,988.76
2018-07-01,1048.09
2018-08-01,1090.51
2018-09-01,1174.29
2018-10-01,1055.57
2018-11-01,1117.32
2018-12-01,1090.03
2019-01-01,1172.97
2019-02-01,1165.02
2019-03-01,1214.96
2019-04-01,1147.28
2019-05-01,1128.0
2019-06-01,1165.85
2019-07-01,1285.69
2019-08-01,1249.27
2019-09-01,1145.99
2019-10-01,1185.19
2019-11-01,1245.38
2019-12-01,1263.29
2020-01-01,1307.6
2020-02-01,1389.38
2020-03-01,1328.95
2020-04-01,1330.15
2020-05-01,1442.17
2020-06-01,1321.96
2020-07-01,1314.11
2020-08-01,1483.46
2020-09-01,1498.64
2020-10-01,1645.2
2020-11-01,1773.85
2020-12-01,1740.57
2021-01-01,1722.5
2021-02-01,1856.01
2021-03-01,1843.65
2021-04-01,1705.32
2021-05-01,1723.61
2021-06-01,1825.84
2021-07-01,1788.25
2021-08-01,1884.6
2021-09-01,2134.86
2021-10-01,2274.7
2021-11-01,2440.56
2021-12-01,2636.37
2022-01-01,2675.97
2022-02-01,2642.39
2022-03-01,2744.09
2022-04-01,3503.6
2022-05-01,3660.76
2022-06-01,4096.13
2022-07-01,3765.1
2022-08-01,3378.32
2022-09-01,3382.95
2022-10-01,3872.2
2022-11-01,3867.85
2022-12-01,3763.86
2023-01-01,3476.51
2023-02-01,3578.94
2023-03-01,4216.04
2023-04-01,4162.72
2023-05-01,4313.99
2023-06-01,4207.08
2023-07-01,3963.72
2023-08-01,3893.11
2023-09-01,3720.21
2023-10-01,3747.57
2023-11-01,3346.65
2023-12-01,3242.59
2024-01-01,3299.25
2024-02-01,3184.6
2024-03-01,3111.84
2024-04-01,3063.07
2024-05-01,3030.08
2024-06-01,3221.3
2024-07-01,2958.14
2024-08-01,2898.87
2024-09-01,2795.63
2024-10-01,3001.38
2024-11-01,3031.52
2024-12-01,2820.89
2025-01-01,2806.76
2025-02-01,2833.21
2025-03-01,2956.71
2025-04-01,2861.52
2025-05-01,2802.91
2025-06-01,2671.05
2025-07-01,2718.84
2025-08-01,2671.52
2025-09-01,2608.52
2025-10-01,2632.87
2025-11-01,2595.22
2025-12-01,2761.13
2026-01-01,2722.38
2026-02-01,2798.41
2026-03-01,2881.21
2026-04-01,3083.05
2026-05-01,3146.6
2026-06-01,3376.98
2026-07-01,3183.98
2026-08-01,3446.84
2026-09-01,3664.81
2026-10-01,3848.76
2026-11-01,3511.36


In [ ]:
%%writefile /content/fc_helpers.py
import os
import numpy as np
import pandas as pd

H = 12                                              # forecast horizon in months
QL = [round(0.05 * i, 2) for i in range(1, 20)]     # quantile levels 0.05 ... 0.95
ORIGIN_START = pd.Timestamp('2010-01-01')           # first forecast origin
MIN_CONTEXT = 36                                    # at least 3 years of history before forecasting
OUTDIR = '/content/drive/MyDrive/wheat_benchmark_outputs'   # Google Drive: survives disconnects
os.makedirs(OUTDIR, exist_ok=True)


def done(name):
    """True if this result is already saved in Drive (so a re-run skips it)."""
    ok = os.path.exists(f'{OUTDIR}/{name}.csv.gz')
    if ok:
        print(f'{name}: already saved in Drive, skipping')
    return ok


def load_panel():
    df = pd.read_csv('/content/panel.csv', parse_dates=['ds'])
    return df.sort_values(['unique_id', 'ds']).reset_index(drop=True)


def load_world():
    return pd.read_csv('/content/world.csv', parse_dates=['ds']).set_index('ds')['world_rs40_lag1']


def make_origins(df):
    """Every (series, month) with enough history. The context stops at the origin month: no look-ahead."""
    out = []
    for uid, g in df.groupby('unique_id', sort=True):
        y = g['y'].to_numpy(dtype='float32'); ds = list(g['ds'])
        for i in range(MIN_CONTEXT - 1, len(g)):
            if ds[i] >= ORIGIN_START:
                out.append((uid, ds[i], y[: i + 1], ds[: i + 1]))
    return out


def to_wide(model, O, Q):
    """Q has shape (number of origins, H, len(QL)); one output row per origin and horizon."""
    n = len(O)
    assert Q.shape == (n, H, len(QL)), Q.shape
    rec = {'model': np.repeat(model, n * H),
           'unique_id': np.repeat([o[0] for o in O], H),
           'origin': np.repeat([o[1].strftime('%Y-%m') for o in O], H),
           'h': np.tile(np.arange(1, H + 1), n)}
    for j, q in enumerate(QL):
        rec[f'q{q:.2f}'] = Q[:, :, j].reshape(-1)
    return pd.DataFrame(rec)


def save(df, name):
    path = f'{OUTDIR}/{name}.csv.gz'
    df.to_csv(path, index=False, compression='gzip', float_format='%.3f')
    print(f'saved {path}: {len(df):,} rows')


## 2. Foundation models (zero-shot)

In [ ]:
!pip -q install "chronos-forecasting>=2.3.2,<3" "timesfm[torch]>=3.0.2"

In [ ]:
import sys, time, traceback
sys.path.insert(0, '/content')
import numpy as np, pandas as pd, torch
from fc_helpers import *
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
df = load_panel(); world = load_world(); O = make_origins(df)
print(f'{len(O):,} forecast origins across {df.unique_id.nunique()} series | device: {dev}')
if dev == 'cpu':
    print('WARNING: no GPU found. Use Runtime > Change runtime type > T4 GPU, otherwise this will be slow.')

In [ ]:
# 2a. Chronos-Bolt base. It was trained on quantiles 0.1-0.9; levels outside that range are set to the 0.1 / 0.9 values.
from chronos import BaseChronosPipeline
try:
    if done('chronos_bolt_base'): raise StopIteration
    t0 = time.time()
    pipe = BaseChronosPipeline.from_pretrained('amazon/chronos-bolt-base', device_map=dev, torch_dtype=torch.float32)
    res = []
    for i in range(0, len(O), 256):
        q, _ = pipe.predict_quantiles([torch.tensor(o[2]) for o in O[i:i + 256]], prediction_length=H, quantile_levels=QL)
        res.append(q.float().cpu().numpy())
    save(to_wide('chronos_bolt_base', O, np.concatenate(res)), 'chronos_bolt_base')
    print(f'Chronos-Bolt done in {time.time() - t0:.0f}s'); del pipe
except StopIteration:
    pass
except Exception:
    traceback.print_exc()

In [ ]:
# 2b. Chronos-2, univariate (own price history only). cross_learning=False keeps every series and origin separate.
try:
    if done('chronos2'): raise StopIteration
    t0 = time.time()
    pipe2 = BaseChronosPipeline.from_pretrained('amazon/chronos-2', device_map=dev)
    res = []
    for i in range(0, len(O), 128):
        qs, _ = pipe2.predict_quantiles([o[2] for o in O[i:i + 128]], prediction_length=H, quantile_levels=QL, cross_learning=False)
        res.append(np.stack([x[0].float().cpu().numpy() for x in qs]))
    save(to_wide('chronos2', O, np.concatenate(res)), 'chronos2')
    print(f'Chronos-2 done in {time.time() - t0:.0f}s')
except StopIteration:
    pass
except Exception:
    traceback.print_exc()

In [ ]:
# 2c. Chronos-2 with a past covariate: last month's US wheat price in rupees (information set B).
try:
    if done('chronos2_worldcov'): raise StopIteration
    t0 = time.time()
    if 'pipe2' not in globals():
        pipe2 = BaseChronosPipeline.from_pretrained('amazon/chronos-2', device_map=dev)
    def with_cov(o):
        cov = world.reindex(pd.DatetimeIndex(o[3])).bfill().ffill().to_numpy(dtype='float32')
        return {'target': o[2], 'past_covariates': {'world_rs40': cov}}
    res = []
    for i in range(0, len(O), 128):
        qs, _ = pipe2.predict_quantiles([with_cov(o) for o in O[i:i + 128]], prediction_length=H, quantile_levels=QL, cross_learning=False)
        res.append(np.stack([x[0].float().cpu().numpy() for x in qs]))
    save(to_wide('chronos2_worldcov', O, np.concatenate(res)), 'chronos2_worldcov')
    print(f'Chronos-2 + covariate done in {time.time() - t0:.0f}s'); del pipe2
except StopIteration:
    pass
except Exception:
    traceback.print_exc()

In [ ]:
# 2d. TimesFM 2.5 (Apache-2.0 weights). Output index 0 is the mean, 1-9 are the 0.1-0.9 quantiles.
try:
    if done('timesfm25'): raise StopIteration
    import timesfm
    t0 = time.time()
    tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch', torch_compile=False)
    tfm.compile(timesfm.ForecastConfig(max_context=512, max_horizon=128, normalize_inputs=True, per_core_batch_size=64,
                                       use_continuous_quantile_head=True, force_flip_invariance=True,
                                       infer_is_positive=True, fix_quantile_crossing=True))
    _, qf = tfm.forecast(horizon=H, inputs=[o[2].astype('float64') for o in O])
    dec = np.asarray(qf)[:, :H, 1:10]
    levels = np.round(np.arange(0.1, 0.91, 0.1), 2)
    Q = np.stack([[np.interp(QL, levels, dec[i, h]) for h in range(H)] for i in range(len(O))])
    save(to_wide('timesfm25', O, Q), 'timesfm25')
    print(f'TimesFM done in {time.time() - t0:.0f}s'); del tfm
except StopIteration:
    pass
except Exception:
    traceback.print_exc()

## 3. Deep-learning models trained on the panel (refitted every April)

In [ ]:
!pip -q install neuralforecast

In [ ]:
import sys, re, time, logging, warnings, traceback
sys.path.insert(0, '/content')
import numpy as np, pandas as pd
from fc_helpers import *
from neuralforecast import NeuralForecast
from neuralforecast.models import DeepAR, TFT
from neuralforecast.losses.pytorch import MQLoss, DistributionLoss
warnings.filterwarnings('ignore')
for name in ['pytorch_lightning', 'lightning', 'lightning.pytorch', 'lightning_fabric']:
    logging.getLogger(name).setLevel(logging.ERROR)

df = load_panel()
INPUT, STEPS = 24, 300
NAMES = ['DeepAR', 'TFT']

def build_models():
    common = dict(h=H, input_size=INPUT, max_steps=STEPS, scaler_type='robust', random_seed=1,
                  enable_progress_bar=False, enable_model_summary=False, logger=False)
    return [DeepAR(loss=DistributionLoss(distribution='StudentT', quantiles=QL), **common),
            TFT(loss=MQLoss(quantiles=QL), hidden_size=32, **common)]

def quantile_columns(fc, name):
    cols = {}
    for c in fc.columns:
        if not c.startswith(name + '-'):
            continue
        if c.endswith('-median'):
            cols[0.5] = c
        m = re.search(r'-(lo|hi)-([0-9.]+)$', c)
        if m:
            lvl = float(m.group(2)); q = (100 - lvl) / 200 if m.group(1) == 'lo' else 1 - (100 - lvl) / 200
            cols[round(q, 4)] = c
    return [cols[min(cols, key=lambda k: abs(k - q))] for q in QL]

cutoffs = pd.date_range('2010-04-01', df.ds.max(), freq='12MS')
t_all = time.time()
for cut in cutoffs:
    t0 = time.time()
    tag = f'deep_cutoff_{cut:%Y}'
    if done(tag):
        continue
    results = {n: ([], []) for n in NAMES}
    try:
        train = df[df.ds <= cut]
        n_obs = train.groupby('unique_id').size()
        train = train[train.unique_id.isin(n_obs[n_obs >= INPUT + H + 12].index)]
        nf = NeuralForecast(models=build_models(), freq='MS')
        nf.fit(df=train)
        last_origin = min(cut + pd.DateOffset(months=11), df.ds.max())
        for origin in pd.date_range(cut, last_origin, freq='MS'):
            hist = df[df.ds <= origin]
            last = hist.groupby('unique_id').ds.max(); n_h = hist.groupby('unique_id').size()
            ok = [u for u in last.index if last[u] == origin and n_h[u] >= MIN_CONTEXT]
            if not ok:
                continue
            fc = nf.predict(df=hist[hist.unique_id.isin(ok)])
            if 'unique_id' not in fc.columns:
                fc = fc.reset_index()
            fc = fc.sort_values(['unique_id', 'ds'])
            for name in NAMES:
                qc = quantile_columns(fc, name)
                for uid, g in fc.groupby('unique_id', sort=True):
                    q = np.sort(g[qc].to_numpy(dtype='float64')[:H], axis=1)
                    results[name][0].append((uid, origin)); results[name][1].append(q)
        parts = [to_wide(name.lower(), results[name][0], np.stack(results[name][1])) for name in NAMES if results[name][1]]
        if parts:
            save(pd.concat(parts), tag)                      # one Drive file per year: a disconnect loses at most one year
        print(f'cutoff {cut:%Y-%m}: trained on {train.unique_id.nunique()} series, done in {time.time() - t0:.0f}s')
    except Exception:
        print(f'cutoff {cut:%Y-%m} failed:'); traceback.print_exc()
print(f'all deep models done in {(time.time() - t_all) / 60:.0f} min')

## 4. Check what is saved in Drive

In [ ]:
import glob, os, pandas as pd
OUTDIR = '/content/drive/MyDrive/wheat_benchmark_outputs'
files_ = sorted(glob.glob(OUTDIR + '/*.csv.gz'))
for p in files_:
    print(f'{os.path.basename(p):32s} {os.path.getsize(p) / 1e6:6.2f} MB')
need = ['chronos_bolt_base', 'chronos2', 'chronos2_worldcov', 'timesfm25'] + [f'deep_cutoff_{y}' for y in range(2010, pd.Timestamp.today().year + 1) if pd.Timestamp(f'{y}-04-01') <= pd.Timestamp('2026-10-01')]
missing = [n for n in need if not os.path.exists(f'{OUTDIR}/{n}.csv.gz')]
print('ALL DONE: the forecast files are in the Drive folder wheat_benchmark_outputs.' if not missing else f'Still missing: {missing}. Reconnect and Run all again.')